In [19]:
import os
import hashlib
import zipfile
import requests
import shutil
from pathlib import Path

def calculate_md5(filepath):
    hash_md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

def verify_zip(filepath, expected_md5):
    if not os.path.exists(filepath):
        return False, "Файл не существует"

    # Вычисление MD5
    actual_md5 = calculate_md5(filepath)
    if actual_md5 != expected_md5:
        return False, f"Неверная контрольная сумма MD5: {actual_md5} (ожидалась {expected_md5})"

    # Проверка целостности ZIP
    try:
        with zipfile.ZipFile(filepath) as z:
            bad_file = z.testzip()
            if bad_file is not None:
                return False, f"Архив поврежден, первый битый файл: {bad_file}"
            return True, f"Архив цел, MD5 совпадает: {actual_md5}"
    except zipfile.BadZipFile:
        return False, "Файл не является корректным ZIP-архивом"

EXPECTED_MD5 = "11e95fae502b9fffcd9771aa1d9cb01a"
local_path = "/content/AtCoder.mongo.zip"
drive_dir = "/content/drive/MyDrive/Research_Code_Retrieval/data"
drive_path = os.path.join(drive_dir, "AtCoder.mongo.zip")

# Проверяем пути
found_valid = False
for path in [drive_path, local_path]:
    if os.path.exists(path):
        print(f"Проверяю существующий файл {path}...")
        ok, msg = verify_zip(path, EXPECTED_MD5)
        print(msg)
        if ok:
            found_valid = True
            if path == local_path and not os.path.exists(drive_path):
                print("Копирую локальный корректный архив на Google Drive...")
                os.makedirs(drive_dir, exist_ok=True)
                shutil.copy2(local_path, drive_path)
            break

if found_valid:
    print("\n[УСПЕХ] Корректный архив AtCoder уже присутствует в системе!")
else:
    print("\nВалидный архив не найден. Начинаю загрузку с Zenodo...")
    url = "https://zenodo.org/records/4002869/files/AtCoder.mongo.zip?download=1"
    temp_path = "/content/AtCoder.mongo.zip.part"

    os.makedirs(drive_dir, exist_ok=True)

    try:
        response = requests.get(url, stream=True, timeout=120)
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))

        print(f"Размер файла для скачивания: {total_size / (1024*1024):.2f} MB")

        downloaded = 0
        with open(temp_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total_size:
                        print(f"Прогресс: {downloaded / total_size * 100:.1f}% ({downloaded / (1024*1024):.2f} MB)", end='\r')

        print("\nЗагрузка завершена. Проверяю скачанный файл...")
        ok, msg = verify_zip(temp_path, EXPECTED_MD5)
        print(msg)

        if ok:
            shutil.move(temp_path, drive_path)
            print(f"Файл успешно верифицирован и перемещен в: {drive_path}")
            # Создадим символическую ссылку в /content для удобства доступа
            if os.path.exists(local_path):
                os.remove(local_path)
            os.symlink(drive_path, local_path)
            print(f"Создана символическая ссылка {local_path} -> {drive_path}")
        else:
            print("[ОШИБКА] Скачанный файл не прошел валидацию!")
            if os.path.exists(temp_path):
                os.remove(temp_path)
    except Exception as e:
        print(f"Ошибка при загрузке: {e}")
        if os.path.exists(temp_path):
            os.remove(temp_path)


Валидный архив не найден. Начинаю загрузку с Zenodo...
Размер файла для скачивания: 333.85 MB
Прогресс: 100.0% (333.85 MB)
Загрузка завершена. Проверяю скачанный файл...
Архив цел, MD5 совпадает: 11e95fae502b9fffcd9771aa1d9cb01a
Файл успешно верифицирован и перемещен в: /content/drive/MyDrive/Research_Code_Retrieval/data/AtCoder.mongo.zip
Создана символическая ссылка /content/AtCoder.mongo.zip -> /content/drive/MyDrive/Research_Code_Retrieval/data/AtCoder.mongo.zip


In [ ]:
%pip -q install gdown==5.2.0
# OPTIONAL LEGACY XLCoST EXPLORATION — not required for the final AtCoder/CoREB analyses.
# This Google Drive share link is an external legacy resource; public availability has not been independently verified.

import gdown
import zipfile
import json
import re
import ast
import hashlib
from pathlib import Path, PurePosixPath
from collections import Counter

url = "https://drive.google.com/file/d/1Cp3vFITRaUEJwPoeI_uv0cC6KVyvDc4F/view?usp=sharing"
archive = "/content/XLCoST_data.zip"

print("Скачиваю официальный архив XLCoST...")
downloaded = gdown.download(url, archive, quiet=False, fuzzy=True)

if not downloaded or not zipfile.is_zipfile(archive):
    raise RuntimeError(
        "Не удалось скачать или открыть архив. "
        "Пришли мне текст ошибки."
    )

def restore(tokens):
    if not isinstance(tokens, list):
        return ""
    out, indent, start = [], 0, True
    for token in tokens:
        token = str(token)
        if token == "NEW_LINE":
            out.append("\n")
            start = True
        elif token == "INDENT":
            indent += 1
        elif token == "DEDENT":
            indent = max(0, indent - 1)
        else:
            if start:
                out.append("    " * indent)
                start = False
            elif out and out[-1] != "\n":
                out.append(" ")
            out.append(token)
    return "".join(out).strip()

report = {
    "source": "Official XLCoST archive",
    "published_reference_counts": {
        "train": 8991, "valid": 471, "test": 882
    },
    "files": {},
    "python_syntax_sample": {},
    "cross_split_overlap": {}
}
id_sets = {}
python_samples = []
found = 0

with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        p = PurePosixPath(name)
        parts = [x.lower() for x in p.parts]

        if p.suffix.lower() != ".jsonl":
            continue
        if "code2code_search" not in parts or "program_level" not in parts:
            continue

        j = parts.index("program_level")
        if j + 2 >= len(parts):
            continue

        lang = p.parts[j + 1]
        split = p.stem.lower()
        if lang.lower() not in ("java", "python"):
            continue
        if split not in ("train", "valid", "validation", "val", "test"):
            continue

        split = "valid" if split in ("validation", "val") else split
        key = f"{lang}/{split}"
        found += 1

        stats = {
            "rows": 0,
            "bad_json": 0,
            "missing_idx": 0,
            "missing_query_tokens": 0,
            "missing_target_tokens": 0,
            "empty_query_tokens": 0,
            "empty_target_tokens": 0
        }
        ids = Counter()
        targets = Counter()
        pair_counts = Counter()
        target_hashes = Counter()
        file_ids = set()

        with z.open(name) as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    row = json.loads(line)
                except Exception:
                    stats["bad_json"] += 1
                    continue

                stats["rows"] += 1
                idx = row.get("idx")
                query = row.get("docstring_tokens")
                target = row.get("code_tokens")

                if not idx:
                    stats["missing_idx"] += 1
                else:
                    ids[str(idx)] += 1
                    file_ids.add(str(idx))
                    pair = str(idx).split("/", 1)
                    if len(pair) == 2:
                        qlang = pair[0].rsplit("-", 1)[-1]
                        tlang = pair[1].rsplit("-", 1)[-1]
                        pair_counts[f"{qlang}->{tlang}"] += 1
                        if qlang.lower() == "python" and isinstance(query, list):
                            if len(python_samples) < 150:
                                python_samples.append(restore(query))
                        if tlang.lower() == "python" and isinstance(target, list):
                            if len(python_samples) < 150:
                                python_samples.append(restore(target))

                if not isinstance(query, list):
                    stats["missing_query_tokens"] += 1
                elif not query:
                    stats["empty_query_tokens"] += 1

                if not isinstance(target, list):
                    stats["missing_target_tokens"] += 1
                elif not target:
                    stats["empty_target_tokens"] += 1
                else:
                    fingerprint = hashlib.sha256(
                        "\x1f".join(map(str, target)).encode()
                    ).hexdigest()
                    target_hashes[fingerprint] += 1

        stats["duplicate_idx_groups"] = sum(n > 1 for n in ids.values())
        stats["duplicate_target_code_groups"] = sum(
            n > 1 for n in target_hashes.values()
        )
        stats["java_python_pair_rows"] = sum(
            n for pair, n in pair_counts.items()
            if {x.lower() for x in pair.split("->")} == {"java", "python"}
        )
        stats["target_language_counts"] = dict(pair_counts)
        report["files"][key] = stats
        id_sets[key] = file_ids

if not found:
    raise RuntimeError(
        "В архиве не найдены ожидаемые JSONL-файлы "
        "code2code_search/program_level. "
        "Нужно проверить структуру архива."
    )

for lang in ("Java", "Python"):
    for a, b in (("train", "valid"), ("train", "test"), ("valid", "test")):
        ka, kb = f"{lang}/{a}", f"{lang}/{b}"
        if ka in id_sets and kb in id_sets:
            report["cross_split_overlap"][f"{ka} vs {kb}"] = len(
                id_sets[ka] & id_sets[kb]
            )

parsed = failed = 0
for source in python_samples:
    if not source:
        continue
    try:
        ast.parse(source)
        parsed += 1
    except (SyntaxError, ValueError, TypeError):
        failed += 1

report["python_syntax_sample"] = {
    "sampled": len(python_samples),
    "parsed": parsed,
    "failed": failed
}

output = "/content/XLCoST_data_audit_report.json"
with open(output, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\n=== ОТЧЁТ АУДИТА ===")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("\nОтчёт сохранён в:", output)

In [ ]:
%pip -q install scikit-learn==1.5.2

import json, zipfile, re
import numpy as np
from pathlib import PurePosixPath
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

ARCHIVE = "/content/XLCoST_data.zip"
z = zipfile.ZipFile(ARCHIVE)

def find_file(lang, split):
    for name in z.namelist():
        p = PurePosixPath(name)
        parts = [x.lower() for x in p.parts]
        if "code2code_search" not in parts:
            continue
        if "program_level" not in parts:
            continue
        i = parts.index("program_level")
        if i + 1 < len(parts):
            if parts[i+1] == lang.lower() and p.stem.lower() == split:
                return name
    raise FileNotFoundError(f"Не найден файл {lang}/{split}")

def load_pairs(source, target, split="valid"):
    rows = []
    name = find_file(source, split)

    with z.open(name) as f:
        for line in f:
            r = json.loads(line)
            left, right = r["idx"].split("/", 1)
            qid, qlang = left.rsplit("-", 1)
            tid, tlang = right.rsplit("-", 1)

            if qlang.lower() != source.lower():
                continue
            if tlang.lower() != target.lower():
                continue

            rows.append({
                "qid": qid,
                "tid": tid,
                "query": r["docstring_tokens"],
                "target": r["code_tokens"]
            })

    return rows

# Общие категории для Python и Java
ALIASES = {
    "if":"IF", "else":"ELSE", "elif":"ELSE",
    "for":"FOR", "while":"WHILE", "do":"DO",
    "return":"RETURN", "break":"BREAK", "continue":"CONTINUE",
    "try":"TRY", "catch":"CATCH", "except":"CATCH",
    "finally":"FINALLY", "throw":"THROW", "raise":"THROW",
    "switch":"SWITCH", "match":"SWITCH", "case":"CASE",
    "and":"LOG_AND", "or":"LOG_OR", "not":"LOG_NOT",
    "def":"FUNC_DECL", "function":"FUNC_DECL",
    "class":"CLASS_DECL", "new":"NEW",
    "import":"IMPORT", "from":"IMPORT",
    "public":"MODIFIER", "private":"MODIFIER",
    "protected":"MODIFIER", "static":"MODIFIER",
    "int":"TYPE", "long":"TYPE", "short":"TYPE",
    "float":"TYPE", "double":"TYPE", "boolean":"TYPE",
    "char":"TYPE", "void":"TYPE", "string":"TYPE",
    "str":"TYPE", "list":"TYPE",
    "true":"BOOL", "false":"BOOL", "none":"NULL",
    "null":"NULL", "self":"THIS", "this":"THIS"
}

SYMBOLS = {
    "&&":"LOG_AND", "||":"LOG_OR", "!":"LOG_NOT",
    "==":"EQ", "===":"EQ", "!=":"NEQ", "!==":"NEQ",
    ">":"GT", "<":"LT", ">=":"GTE", "<=":"LTE",
    "=":"ASSIGN", "+=":"ADD_ASSIGN", "-=":"SUB_ASSIGN",
    "+":"ADD", "-":"SUB", "*":"MUL", "/":"DIV",
    "%":"MOD", "++":"INC", "--":"DEC",
    "NEW_LINE":"STMT_END", ";":"STMT_END",
    "INDENT":"BLOCK_OPEN", "DEDENT":"BLOCK_CLOSE",
    "{":"BLOCK_OPEN", "}":"BLOCK_CLOSE"
}

IDENT = re.compile(r"^[A-Za-z_$][A-Za-z0-9_$]*$")
NUMBER = re.compile(r"^\d+(\.\d+)?$")

def fingerprint(tokens):
    out, variables = [], {}
    for i, raw in enumerate(tokens):
        t = str(raw)
        low = t.lower()

        if t in SYMBOLS:
            out.append(SYMBOLS[t])
        elif low in ALIASES:
            out.append(ALIASES[low])
        elif NUMBER.match(t):
            out.append("NUMBER")
        elif len(t) >= 2 and t[0] in "'\"" and t[-1] == t[0]:
            out.append("STRING")
        elif IDENT.match(t):
            next_token = str(tokens[i+1]) if i+1 < len(tokens) else ""
            if next_token == "(" and low not in ("if", "while", "for"):
                out.append("CALL")
            else:
                if t not in variables:
                    variables[t] = f"VAR_{len(variables)+1}"
                out.append(variables[t])
        else:
            out.append(t)

    return out

def metrics(scores, relevant_ids, candidate_ids):
    positions = {x:i for i,x in enumerate(candidate_ids)}
    results = {"Recall@1": [], "Recall@5": [],
               "Recall@10": [], "MRR": []}

    for i, relevant in enumerate(relevant_ids):
        row = scores[i]
        score = row[positions[relevant]]
        higher = int(np.sum(row > score + 1e-12))
        tied = max(1, int(np.sum(np.abs(row-score) <= 1e-12)))

        # Усредняем результат по случайному порядку равных оценок.
        for k in (1, 5, 10):
            hit = min(max(k-higher, 0), tied) / tied
            results[f"Recall@{k}"].append(hit)

        reciprocal = sum(
            1 / (higher+j) for j in range(1, tied+1)
        ) / tied
        results["MRR"].append(reciprocal)

    return {k: round(float(np.mean(v)), 4)
            for k, v in results.items()}

def run_direction(source, target):
    rows = load_pairs(source, target, "valid")
    if not rows:
        raise ValueError(f"Нет пар для {source} -> {target}")

    # Один кандидат на каждый идентификатор целевой программы.
    candidates = {}
    for r in rows:
        if r["tid"] in candidates:
            if candidates[r["tid"]] != r["target"]:
                raise ValueError("Один ID связан с разным кодом")
        candidates[r["tid"]] = r["target"]

    qids = [r["qid"] for r in rows]
    relevant = [r["tid"] for r in rows]
    if len(set(qids)) != len(qids):
        raise ValueError("Есть повторяющиеся ID запросов")
    if not set(relevant).issubset(candidates):
        raise ValueError("Не удалось найти все правильные кандидаты")

    candidate_ids = list(candidates.keys())
    queries = [" ".join(map(str, r["query"])) for r in rows]
    target_docs = [
        " ".join(map(str, candidates[c])) for c in candidate_ids
    ]

    def retrieve(qdocs, cdocs):
        vec = TfidfVectorizer(
            tokenizer=str.split, token_pattern=None,
            lowercase=False, ngram_range=(1, 2),
            sublinear_tf=True
        )
        C = vec.fit_transform(cdocs)
        Q = vec.transform(qdocs)
        return (Q @ C.T).toarray()

    raw_scores = retrieve(queries, target_docs)

    q_struct = [" ".join(fingerprint(r["query"])) for r in rows]
    c_struct = [
        " ".join(fingerprint(candidates[c])) for c in candidate_ids
    ]
    struct_scores = retrieve(q_struct, c_struct)

    duplicates = sum(
        n-1 for n in Counter(
            tuple(candidates[c]) for c in candidate_ids
        ).values() if n > 1
    )

    return {
        "direction": f"{source} -> {target}",
        "queries": len(rows),
        "candidates": len(candidate_ids),
        "extra_exact_duplicate_candidates": duplicates,
        "raw_token_TFIDF": metrics(raw_scores, relevant, candidate_ids),
        "normalized_fingerprint_TFIDF": metrics(
            struct_scores, relevant, candidate_ids
        )
    }

for source, target in [("Python", "Java"), ("Java", "Python")]:
    result = run_direction(source, target)
    print("\n", json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
def find_file(lang, split):
    aliases = {
        "valid": {"valid", "validation", "val"},
        "validation": {"valid", "validation", "val"},
        "val": {"valid", "validation", "val"},
        "train": {"train"},
        "test": {"test"}
    }

    allowed = aliases.get(split.lower(), {split.lower()})

    for name in z.namelist():
        p = PurePosixPath(name)
        parts = [x.lower() for x in p.parts]

        if "code2code_search" not in parts:
            continue
        if "program_level" not in parts:
            continue

        i = parts.index("program_level")

        if i + 1 >= len(parts):
            continue

        if (
            parts[i + 1] == lang.lower()
            and p.stem.lower() in allowed
        ):
            print("Найден файл:", name)
            return name

    raise FileNotFoundError(
        f"Не найден файл {lang}/{split}. "
        "Проверь структуру архива."
    )

In [ ]:
for source, target in [
    ("Python", "Java"),
    ("Java", "Python")
]:
    result = run_direction(source, target)
    print("\n", json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    ))

Найден файл: XLCoST_data/retrieval/code2code_search/program_level/Python/val.jsonl

 {
  "direction": "Python -> Java",
  "queries": 471,
  "candidates": 471,
  "extra_exact_duplicate_candidates": 5,
  "raw_token_TFIDF": {
    "Recall@1": 0.9682,
    "Recall@5": 0.9979,
    "Recall@10": 1.0,
    "MRR": 0.983
  },
  "normalized_fingerprint_TFIDF": {
    "Recall@1": 0.3482,
    "Recall@5": 0.5839,
    "Recall@10": 0.6709,
    "MRR": 0.4629
  }
}
Найден файл: XLCoST_data/retrieval/code2code_search/program_level/Java/val.jsonl

 {
  "direction": "Java -> Python",
  "queries": 471,
  "candidates": 471,
  "extra_exact_duplicate_candidates": 2,
  "raw_token_TFIDF": {
    "Recall@1": 0.9682,
    "Recall@5": 0.9936,
    "Recall@10": 0.9979,
    "MRR": 0.9812
  },
  "normalized_fingerprint_TFIDF": {
    "Recall@1": 0.3355,
    "Recall@5": 0.5541,
    "Recall@10": 0.6667,
    "MRR": 0.4403
  }
}


In [ ]:
import json
import zipfile
from pathlib import PurePosixPath
from collections import Counter

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

z = zipfile.ZipFile("/content/XLCoST_data.zip")

def parse_ids(idx):
    left, right = idx.split("/", 1)
    query_id, query_lang = left.rsplit("-", 1)
    target_id, target_lang = right.rsplit("-", 1)
    return query_id, query_lang, target_id, target_lang

def split_name(path):
    name = path.stem.lower()
    return "valid" if name in ("val", "validation", "valid") else name

# 1. Берём 471 проверочный запрос Python -> Java.
query_file = None

for name in z.namelist():
    p = PurePosixPath(name)
    parts = [x.lower() for x in p.parts]

    if (
        "code2code_search" in parts
        and "program_level" in parts
        and "python" in parts
        and split_name(p) == "valid"
    ):
        query_file = name
        break

if query_file is None:
    raise FileNotFoundError("Не найден файл Python/validation")

queries = []

with z.open(query_file) as f:
    for line in f:
        r = json.loads(line)
        qid, qlang, tid, tlang = parse_ids(r["idx"])

        if tlang.lower() == "java":
            queries.append({
                "query_id": qid,
                "target_id": tid,
                "tokens": r["docstring_tokens"]
            })

# 2. Собираем Java-кандидатов из train и validation.
# Один и тот же Java-код встречается в нескольких языковых парах,
# поэтому дедуплицируем по ID целевой программы.
train_pool = {}
valid_pool = {}
conflicts = {"train": set(), "valid": set()}

for name in z.namelist():
    p = PurePosixPath(name)
    parts = [x.lower() for x in p.parts]

    if "code2code_search" not in parts or "program_level" not in parts:
        continue

    split = split_name(p)
    if split not in ("train", "valid"):
        continue

    if "python" not in parts and "java" not in parts:
        continue

    pool = train_pool if split == "train" else valid_pool

    with z.open(name) as f:
        for line in f:
            r = json.loads(line)
            _, _, tid, tlang = parse_ids(r["idx"])

            if tlang.lower() != "java":
                continue

            tokens = r["code_tokens"]

            if tid in pool:
                if pool[tid] != tokens:
                    conflicts[split].add(tid)
            else:
                pool[tid] = tokens

# Validation-версия имеет приоритет, если ID встречается в обоих пулах.
overlap = set(train_pool) & set(valid_pool)
candidates = dict(train_pool)
candidates.update(valid_pool)

if not queries:
    raise ValueError("Нет Python -> Java запросов")

candidate_ids = list(candidates)
positions = {tid: i for i, tid in enumerate(candidate_ids)}

missing = [
    q["target_id"] for q in queries
    if q["target_id"] not in positions
]

print("Число запросов:", len(queries))
print("Java-кандидатов из train:", len(train_pool))
print("Java-кандидатов из validation:", len(valid_pool))
print("Всего уникальных кандидатов:", len(candidates))
print("ID, встречающиеся в обоих разделах:", len(overlap))
print("ID с разными версиями кода внутри train:", len(conflicts["train"]))
print("ID с разными версиями кода внутри validation:", len(conflicts["valid"]))
print("Запросов без правильного кандидата:", len(missing))

if missing:
    raise ValueError("Есть запросы без правильного кандидата; оценку остановили")

# 3. Оцениваем обычный TF-IDF-поиск.
query_docs = [" ".join(map(str, q["tokens"])) for q in queries]
candidate_docs = [
    " ".join(map(str, candidates[tid]))
    for tid in candidate_ids
]

vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(candidate_docs)
Q = vectorizer.transform(query_docs)
scores = (Q @ X.T).toarray()

relevant = [q["target_id"] for q in queries]
hits = {1: [], 5: [], 10: []}
reciprocal_ranks = []

for i, target_id in enumerate(relevant):
    row = scores[i]
    correct = row[positions[target_id]]

    higher = int(np.sum(row > correct + 1e-12))
    tied = max(1, int(np.sum(np.abs(row - correct) <= 1e-12)))

    for k in hits:
        hits[k].append(
            min(max(k - higher, 0), tied) / tied
        )

    reciprocal_ranks.append(
        sum(1 / (higher + j) for j in range(1, tied + 1)) / tied
    )

report = {
    "queries": len(queries),
    "candidates": len(candidate_ids),
    "Recall@1": round(float(np.mean(hits[1])), 4),
    "Recall@5": round(float(np.mean(hits[5])), 4),
    "Recall@10": round(float(np.mean(hits[10])), 4),
    "MRR": round(float(np.mean(reciprocal_ranks)), 4)
}

print("\n=== EXPANDED POOL RESULTS ===")
print(json.dumps(report, indent=2))


FileNotFoundError: [Errno 2] No such file or directory: '/content/XLCoST_data.zip'

In [ ]:
%pip -q install gdown==5.2.0 scikit-learn==1.5.2

import gdown
import zipfile
import os

archive = "/content/XLCoST_data.zip"
url = "https://drive.google.com/file/d/1Cp3vFITRaUEJwPoeI_uv0cC6KVyvDc4F/view?usp=sharing"

if not zipfile.is_zipfile(archive):
    gdown.download(url, archive, quiet=False, fuzzy=True)

if zipfile.is_zipfile(archive):
    print("Архив успешно восстановлен!")
else:
    print("Не удалось восстановить архив. Пришли мне эту ошибку.")

In [ ]:
%pip -q install gdown==5.2.0 scikit-learn==1.5.2
# OPTIONAL LEGACY XLCoST EXPLORATION — not required for the final AtCoder/CoREB analyses.
# This Google Drive share link is an external legacy resource; public availability has not been independently verified.

import os, json, zipfile, re
from pathlib import PurePosixPath
from collections import Counter
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

ARCHIVE = "/content/XLCoST_data.zip"
URL = "https://drive.google.com/uc?id=1Cp3vFITRaUEJwPoeI_uv0cC6KVyvDc4F"

if not zipfile.is_zipfile(ARCHIVE):
    import gdown
    gdown.download(URL, ARCHIVE, quiet=False, fuzzy=True)

if not zipfile.is_zipfile(ARCHIVE):
    raise RuntimeError("Архив не найден или повреждён. Пришли ошибку.")

z = zipfile.ZipFile(ARCHIVE)

def parse_ids(idx):
    left, right = idx.split("/", 1)
    qid, qlang = left.rsplit("-", 1)
    tid, tlang = right.rsplit("-", 1)
    return qid, qlang, tid, tlang

def get_split(path):
    s = path.stem.lower()
    return "valid" if s in ("val", "valid", "validation") else s

queries = []
baseline_pool = {}
all_valid_pool = {}
train_pool = {}
conflicts = {"train": set(), "valid": set()}

print("Читаю файлы XLCoST...")

for name in z.namelist():
    p = PurePosixPath(name)
    parts = [x.lower() for x in p.parts]

    if "code2code_search" not in parts or "program_level" not in parts:
        continue

    i = parts.index("program_level")
    if i + 1 >= len(parts):
        continue

    source_lang = parts[i + 1]
    split = get_split(p)

    if split not in ("train", "valid"):
        continue

    if source_lang not in {
        "python", "java", "c", "c++", "c#", "php", "javascript"
    }:
        continue

    with z.open(name) as f:
        for line in f:
            r = json.loads(line)
            qid, qlang, tid, tlang = parse_ids(r["idx"])

            if qlang.lower() != source_lang:
                continue

            if (
                source_lang == "python"
                and split == "valid"
                and tlang.lower() == "java"
            ):
                queries.append({
                    "qid": qid,
                    "tid": tid,
                    "tokens": r["docstring_tokens"]
                })
                baseline_pool[tid] = r["code_tokens"]

            if tlang.lower() != "java":
                continue

            pool = train_pool if split == "train" else all_valid_pool
            if tid in pool and pool[tid] != r["code_tokens"]:
                conflicts[split].add(tid)
            else:
                pool[tid] = r["code_tokens"]

if not queries:
    raise RuntimeError("Не найдены проверочные запросы Python -> Java.")

# Исключаем из train кандидатов ID, которые встречаются в validation.
# Это снижает риск пересечения задач между разделами.
train_only = {
    k: v for k, v in train_pool.items()
    if k not in all_valid_pool
}
expanded_pool = dict(train_only)
expanded_pool.update(all_valid_pool)

def evaluate(pool, label):
    candidate_ids = list(pool)
    positions = {tid: i for i, tid in enumerate(candidate_ids)}

    missing = sum(q["tid"] not in positions for q in queries)
    if missing:
        raise RuntimeError(
            f"{label}: для {missing} запросов нет правильного кандидата."
        )

    query_docs = [" ".join(map(str, q["tokens"])) for q in queries]
    candidate_docs = [
        " ".join(map(str, pool[tid])) for tid in candidate_ids
    ]

    vectorizer = TfidfVectorizer(
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        sublinear_tf=True
    )

    X = vectorizer.fit_transform(candidate_docs)
    Q = vectorizer.transform(query_docs)
    scores = (Q @ X.T).toarray()

    hits = {1: [], 5: [], 10: []}
    reciprocal = []

    for i, q in enumerate(queries):
        correct = scores[i, positions[q["tid"]]]
        higher = int(np.sum(scores[i] > correct + 1e-12))
        tied = max(
            1,
            int(np.sum(np.abs(scores[i] - correct) <= 1e-12))
        )

        for k in hits:
            hits[k].append(min(max(k - higher, 0), tied) / tied)

        reciprocal.append(
            sum(1 / (higher + j) for j in range(1, tied + 1)) / tied
        )

    return {
        "experiment": label,
        "queries": len(queries),
        "candidates": len(candidate_ids),
        "Recall@1": round(float(np.mean(hits[1])), 4),
        "Recall@5": round(float(np.mean(hits[5])), 4),
        "Recall@10": round(float(np.mean(hits[10])), 4),
        "MRR": round(float(np.mean(reciprocal)), 4)
    }

print("Запросов:", len(queries))
print("Кандидатов в исходном validation-пуле:", len(baseline_pool))
print("Кандидатов только из train после удаления пересечений:", len(train_only))
print("Конфликтующих ID в train:", len(conflicts["train"]))
print("Конфликтующих ID в validation:", len(conflicts["valid"]))

results = [
    evaluate(baseline_pool, "Validation-only"),
    evaluate(expanded_pool, "Expanded train + validation")
]

print("\n=== РЕЗУЛЬТАТЫ ===")
print(json.dumps(results, indent=2, ensure_ascii=False))

In [ ]:
%pip -q install pymongo==4.10.1 requests==2.32.3

import os
import zipfile
import requests
from bson import decode_file_iter

url = "https://zenodo.org/records/4002869/files/AtCoder.mongo.zip?download=1"
archive = "/content/AtCoder.mongo.zip"

# Скачиваем архив только при необходимости
if not zipfile.is_zipfile(archive):
    print("Скачиваю AtCoder (около 350 МБ)...")

    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(archive, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

if not zipfile.is_zipfile(archive):
    raise RuntimeError("Архив не удалось проверить как ZIP.")

with zipfile.ZipFile(archive) as z:
    files = [f for f in z.infolist() if not f.is_dir()]
    bson_files = [
        f for f in files if f.filename.lower().endswith(".bson")
    ]

    print("\nФайлов в архиве:", len(files))
    print("BSON-файлов:", len(bson_files))

    print("\nСписок BSON-файлов:")
    for f in bson_files[:20]:
        print(f"{f.filename} — {f.file_size / 1024**2:.2f} МБ")

    if not bson_files:
        raise RuntimeError("В архиве не найдено BSON-коллекций.")

    sample_file = bson_files[0]
    with z.open(sample_file.filename) as stream:
        sample = next(decode_file_iter(stream), None)

    print("\nПервая коллекция:", sample_file.filename)

    if sample is None:
        print("Коллекция пуста.")
    else:
        print("\nПоля документа:")
        for key, value in sample.items():
            if isinstance(value, str):
                preview = value[:120].replace("\n", " ")
            elif isinstance(value, list):
                preview = f"список, элементов: {len(value)}"
            elif isinstance(value, dict):
                preview = f"объект, полей: {len(value)}"
            else:
                preview = repr(value)[:120]

            print(f"{key}: {type(value).__name__} — {preview}")

In [ ]:
%pip -q install pymongo==4.10.1

import zipfile
from pathlib import Path
from bson import decode_file_iter

# Находим архив, внутри которого лежат BSON-файлы AtCoder.
archive = None

for path in Path("/content").rglob("*.zip"):
    try:
        with zipfile.ZipFile(path) as z:
            if any(n.endswith("code_store.bson") for n in z.namelist()):
                archive = path
                break
    except Exception:
        pass

if archive is None:
    raise FileNotFoundError(
        "Архив AtCoder не найден. Пришли мне список файлов /content."
    )

print("Архив:", archive)

def describe(value, depth=0):
    if isinstance(value, str):
        return {
            "type": "str",
            "length": len(value),
            "preview": value[:180].replace("\n", "\\n")
        }

    if isinstance(value, bytes):
        return {"type": "bytes", "length": len(value)}

    if isinstance(value, dict):
        result = {
            "type": "dict",
            "keys": list(value.keys())[:15]
        }
        if depth < 1:
            result["sample"] = {
                str(k): describe(v, depth + 1)
                for k, v in list(value.items())[:3]
            }
        return result

    if isinstance(value, (list, tuple)):
        result = {"type": type(value).__name__, "length": len(value)}
        if depth < 1 and value:
            result["first_items"] = [
                describe(v, depth + 1) for v in value[:3]
            ]
        return result

    return {"type": type(value).__name__, "preview": repr(value)[:180]}

with zipfile.ZipFile(archive) as z:
    for collection in [
        "meta_store.bson",
        "code_store.bson",
        "file_store.bson"
    ]:
        matches = [
            name for name in z.namelist()
            if name.endswith(collection)
        ]

        if not matches:
            print(f"\n{collection}: не найден")
            continue

        print(f"\n{'=' * 20} {collection} {'=' * 20}")

        with z.open(matches[0]) as stream:
            for i, doc in enumerate(decode_file_iter(stream)):
                print(f"\nЗапись {i + 1}:")
                for key, value in doc.items():
                    print(f"  {key}: {describe(value)}")
                if i == 2:
                    break

In [ ]:
%pip -q install pymongo==4.10.1

import zipfile
import re
import json
from pathlib import PurePosixPath
from collections import Counter, defaultdict
from bson import decode_file_iter

archive = "/content/AtCoder.mongo.zip"

with zipfile.ZipFile(archive) as z:
    names = z.namelist()

    code_file = next(
        n for n in names if n.endswith("code_store.bson")
    )
    meta_file = next(
        n for n in names if n.endswith("meta_store.bson")
    )

    # Читаем небольшую коллекцию метаданных.
    with z.open(meta_file) as f:
        meta_docs = list(decode_file_iter(f))

    metadata = {
        d["key"]: set(d["value"])
        for d in meta_docs
        if d.get("key") in {
            "projects_cached", "java_projects_cached"
        }
    }

    task_languages = defaultdict(Counter)
    task_submission_ids = defaultdict(set)
    submission_counts = Counter()
    language_records = Counter()
    parsed_paths = 0
    unparsed_paths = 0
    code_hashes = Counter()
    task_code_hashes = defaultdict(Counter)

    # Сканируем только записи с исходным кодом, а не AST-расстояния.
    with z.open(code_file) as f:
        for doc in decode_file_iter(f):
            language = str(doc.get("language", "unknown")).lower()
            language_records[language] += 1

            path = str(doc.get("sourceFile", ""))
            match = re.search(
                r"/C(\d+)/P(\d+)/S(\d+)/",
                path,
                flags=re.IGNORECASE
            )

            if not match:
                unparsed_paths += 1
                continue

            parsed_paths += 1
            contest, problem, submission = match.groups()
            task = f"C:{contest}-P:{problem}"

            task_languages[task][language] += 1
            task_submission_ids[task].add(submission)
            submission_counts[language] += 1

            code = str(doc.get("code", ""))
            if code:
                import hashlib
                digest = hashlib.sha256(
                    code.encode("utf-8", errors="replace")
                ).hexdigest()
                code_hashes[language, digest] += 1
                task_code_hashes[task, language][digest] += 1

# Сводка по языкам.
tasks_by_language = defaultdict(set)
for task, counts in task_languages.items():
    for language, count in counts.items():
        if count:
            tasks_by_language[language].add(task)

java_tasks = tasks_by_language["java"]
python_tasks = tasks_by_language["python"]
shared_tasks = java_tasks & python_tasks

def duplicate_extras(counter):
    return sum(n - 1 for n in counter.values() if n > 1)

report = {
    "records_by_language": dict(language_records),
    "parsed_source_paths": parsed_paths,
    "unparsed_source_paths": unparsed_paths,
    "unique_tasks": len(task_languages),
    "tasks_with_java": len(java_tasks),
    "tasks_with_python": len(python_tasks),
    "tasks_with_both_java_python": len(shared_tasks),
    "java_only_tasks": len(java_tasks - python_tasks),
    "python_only_tasks": len(python_tasks - java_tasks),
    "metadata": {},
    "duplicate_code_extra_records_by_language": {},
    "shared_task_submission_count_examples": {}
}

for key, values in metadata.items():
    report["metadata"][key] = {
        "listed_ids": len(values),
        "ids_matching_parsed_tasks": len(values & set(task_languages)),
        "ids_matching_java_tasks": len(values & java_tasks),
        "ids_matching_python_tasks": len(values & python_tasks),
        "ids_matching_shared_tasks": len(values & shared_tasks)
    }

for language in language_records:
    report["duplicate_code_extra_records_by_language"][language] = (
        duplicate_extras(Counter({
            digest: count
            for (lang, digest), count in code_hashes.items()
            if lang == language
        }))
    )

for task in sorted(shared_tasks)[:10]:
    report["shared_task_submission_count_examples"][task] = {
        "Java": task_languages[task]["java"],
        "Python": task_languages[task]["python"]
    }

print(json.dumps(report, indent=2, ensure_ascii=False))

In [ ]:
import zipfile, re
from pathlib import PurePosixPath
from bson import decode_file_iter

archive = "/content/AtCoder.mongo.zip"

with zipfile.ZipFile(archive) as z:
    name = next(n for n in z.namelist()
                if n.endswith("code_store.bson"))

    shown = {"java": 0, "py": 0}
    unmatched = {"java": 0, "py": 0}
    examples = {"java": [], "py": []}
    unmatched_examples = {"java": [], "py": []}

    with z.open(name) as f:
        for d in decode_file_iter(f):
            lang = str(d.get("language", "")).lower()
            if lang not in ("java", "py"):
                continue

            path = str(d.get("sourceFile", ""))
            match = re.search(r"/C(\d+)/P(\d+)/S(\d+)/", path)

            if match:
                if shown[lang] < 5:
                    examples[lang].append({
                        "path": path,
                        "task_id": f"C:{match.group(1)}-P:{match.group(2)}",
                        "submission_id": match.group(3)
                    })
                    shown[lang] += 1
            else:
                unmatched[lang] += 1
                if len(unmatched_examples[lang]) < 5:
                    unmatched_examples[lang].append(path)

    for lang in ("java", "py"):
        print(f"\n=== {lang.upper()} ===")
        print("Пути, соответствующие шаблону:")
        for x in examples[lang]:
            print(x)

        print("\nПути, не соответствующие шаблону:", unmatched[lang])
        for x in unmatched_examples[lang]:
            print(" ", repr(x))


=== JAVA ===
Пути, соответствующие шаблону:
{'path': 'CodeSeer/subjects/atcoder/src/main/java/C49/P0/S665463/Main.java', 'task_id': 'C:49-P:0', 'submission_id': '665463'}
{'path': 'CodeSeer/subjects/atcoder/src/main/java/C49/P0/S665487/Main.java', 'task_id': 'C:49-P:0', 'submission_id': '665487'}
{'path': 'CodeSeer/subjects/atcoder/src/main/java/C49/P0/S665492/Main.java', 'task_id': 'C:49-P:0', 'submission_id': '665492'}
{'path': 'CodeSeer/subjects/atcoder/src/main/java/C49/P0/S665497/Main.java', 'task_id': 'C:49-P:0', 'submission_id': '665497'}
{'path': 'CodeSeer/subjects/atcoder/src/main/java/C49/P0/S665504/Main.java', 'task_id': 'C:49-P:0', 'submission_id': '665504'}

Пути, не соответствующие шаблону: 1714
  '665506.java'
  '665724.java'
  '666090.java'
  '666967.java'
  '667069.java'

=== PY ===
Пути, соответствующие шаблону:

Пути, не соответствующие шаблону: 22317
  'CodeSeer/subjects/atcoder/src/main/python/C49/P0/665509.py'
  'CodeSeer/subjects/atcoder/src/main/python/C49/P0/6

In [ ]:
import zipfile
import re
import json
from collections import Counter, defaultdict
from bson import decode_file_iter

archive = "/content/AtCoder.mongo.zip"

with zipfile.ZipFile(archive) as z:
    code_file = next(
        n for n in z.namelist()
        if n.endswith("code_store.bson")
    )
    meta_file = next(
        n for n in z.namelist()
        if n.endswith("meta_store.bson")
    )

    with z.open(meta_file) as f:
        meta = {
            d["key"]: set(d["value"])
            for d in decode_file_iter(f)
            if d.get("key") in {
                "projects_cached", "java_projects_cached"
            }
        }

    task_languages = defaultdict(Counter)
    records = Counter()
    missing_path = Counter()
    task_ids = defaultdict(set)

    # C49/P0 соответствует задаче C:49-P:0
    pattern = re.compile(r"C(\d+)/P(\d+)(?:/|$)", re.I)

    with z.open(code_file) as f:
        for d in decode_file_iter(f):
            lang = str(d.get("language", "")).lower()
            if lang not in ("java", "py"):
                continue

            records[lang] += 1
            path = str(d.get("sourceFile", ""))
            m = pattern.search(path)

            if not m:
                missing_path[lang] += 1
                continue

            task = f"C:{m.group(1)}-P:{m.group(2)}"
            task_languages[task][lang] += 1
            task_ids[lang].add(task)

    shared = task_ids["java"] & task_ids["py"]

    report = {
        "records": dict(records),
        "records_without_task_path": dict(missing_path),
        "unique_java_tasks": len(task_ids["java"]),
        "unique_python_tasks": len(task_ids["py"]),
        "tasks_with_both_languages": len(shared),
        "java_only_tasks": len(task_ids["java"] - task_ids["py"]),
        "python_only_tasks": len(task_ids["py"] - task_ids["java"]),
        "metadata_comparison": {},
        "examples_of_shared_tasks": {}
    }

    for key, ids in meta.items():
        report["metadata_comparison"][key] = {
            "metadata_ids": len(ids),
            "matching_java_tasks": len(ids & task_ids["java"]),
            "matching_python_tasks": len(ids & task_ids["py"]),
            "matching_shared_tasks": len(ids & shared)
        }

    for task in sorted(shared)[:10]:
        report["examples_of_shared_tasks"][task] = {
            "java_records": task_languages[task]["java"],
            "python_records": task_languages[task]["py"]
        }

print(json.dumps(report, indent=2, ensure_ascii=False))

{
  "records": {
    "java": 20358,
    "py": 22317
  },
  "records_without_task_path": {
    "java": 1714
  },
  "unique_java_tasks": 360,
  "unique_python_tasks": 333,
  "tasks_with_both_languages": 332,
  "java_only_tasks": 28,
  "python_only_tasks": 1,
  "metadata_comparison": {
    "projects_cached": {
      "metadata_ids": 307,
      "matching_java_tasks": 307,
      "matching_python_tasks": 307,
      "matching_shared_tasks": 307
    },
    "java_projects_cached": {
      "metadata_ids": 361,
      "matching_java_tasks": 360,
      "matching_python_tasks": 332,
      "matching_shared_tasks": 332
    }
  },
  "examples_of_shared_tasks": {
    "C:1-P:0": {
      "java_records": 126,
      "python_records": 85
    },
    "C:1-P:1": {
      "java_records": 157,
      "python_records": 130
    },
    "C:1-P:2": {
      "java_records": 76,
      "python_records": 54
    },
    "C:1-P:3": {
      "java_records": 25,
      "python_records": 35
    },
    "C:10-P:0": {
      "java_record

In [ ]:
%pip -q install pymongo==4.10.1 scikit-learn==1.5.2

import zipfile, re, json, random
import numpy as np
from collections import defaultdict
from bson import decode_file_iter
from sklearn.feature_extraction.text import TfidfVectorizer

ARCHIVE = "/content/AtCoder.mongo.zip"
rng = random.Random(42)

# Читаем код и идентификаторы задач.
grouped = defaultdict(lambda: {"java": {}, "py": {}})
record_counts = defaultdict(int)
unmapped_counts = defaultdict(int)
pattern = re.compile(r"/C(\d+)/P(\d+)(?:/|$)", re.I)

with zipfile.ZipFile(ARCHIVE) as z:
    bson_path = next(
        n for n in z.namelist()
        if n.endswith("code_store.bson")
    )

    with z.open(bson_path) as f:
        for d in decode_file_iter(f):
            lang = str(d.get("language", "")).lower()
            if lang not in ("java", "py"):
                continue

            record_counts[lang] += 1
            m = pattern.search(str(d.get("sourceFile", "")))
            if not m:
                unmapped_counts[lang] += 1
                continue

            task = f"C:{int(m.group(1))}-P:{int(m.group(2))}"
            code = str(d.get("code") or "")
            code = code.replace("\r\n", "\n").strip()

            if code:
                # Убираем только точные повторы кода внутри одной задачи.
                grouped[task][lang].setdefault(code, code)

java_tasks = {
    task for task, langs in grouped.items() if langs["java"]
}
py_tasks = {
    task for task, langs in grouped.items() if langs["py"]
}
shared = sorted(java_tasks & py_tasks)

# До пяти Python-запросов на задачу, чтобы крупные задачи
# не доминировали в пилотной выборке.
queries = []
for task in shared:
    codes = list(grouped[task]["py"].values())
    rng.shuffle(codes)
    queries.extend((task, code) for code in codes[:5])

# Кандидатная база: Java-решения всех распознаваемых задач.
candidates = [
    (task, code)
    for task in sorted(java_tasks)
    for code in grouped[task]["java"].values()
]
rng.shuffle(candidates)  # Случайный порядок для разрешения равных оценок.

candidate_tasks = np.array([task for task, _ in candidates])
candidate_codes = [code for _, code in candidates]
query_tasks = [task for task, _ in queries]
query_codes = [code for _, code in queries]

# Токенизируем идентификаторы, числа, операторы и знаки пунктуации.
token_pattern = re.compile(
    r"[A-Za-z_]\w*|\d+(?:\.\d+)?|==|!=|<=|>=|&&|\|\||"
    r"\+\+|--|->|[^\s]"
)

def tokenize(code):
    return token_pattern.findall(code)

vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    sublinear_tf=True
)

print("Индексирую Java-кандидатов...")
X = vectorizer.fit_transform(candidate_codes)
Q = vectorizer.transform(query_codes)

metrics = defaultdict(list)
per_task = defaultdict(lambda: defaultdict(list))
positive_counts = []

print("Ранжирую результаты...")
batch_size = 64

for start in range(0, len(queries), batch_size):
    scores = (Q[start:start + batch_size] @ X.T).toarray()

    for offset, row in enumerate(scores):
        i = start + offset
        task = query_tasks[i]

        # Все решения одной задачи считаются релевантными.
        relevant_count = int(np.sum(candidate_tasks == task))
        positive_counts.append(relevant_count)

        order = np.argsort(-row, kind="stable")
        ranked_tasks = candidate_tasks[order]
        ranks = np.flatnonzero(ranked_tasks == task)

        if len(ranks) == 0:
            raise RuntimeError(f"Нет Java-кандидатов для {task}")

        first_rank = int(ranks[0]) + 1
        values = {
            "Success@1": float(first_rank <= 1),
            "Success@5": float(first_rank <= 5),
            "Success@10": float(first_rank <= 10),
            "MRR": 1.0 / first_rank
        }

        for name, value in values.items():
            metrics[name].append(value)
            per_task[task][name].append(value)

report = {
    "mapped_records": {
        "Java": record_counts["java"] - unmapped_counts["java"],
        "Python": record_counts["py"] - unmapped_counts["py"]
    },
    "unmapped_records_excluded": dict(unmapped_counts),
    "shared_tasks": len(shared),
    "queries_tested": len(queries),
    "java_candidates": len(candidates),
    "unique_positive_candidates_per_query": {
        "mean": round(float(np.mean(positive_counts)), 2),
        "median": float(np.median(positive_counts)),
        "min": int(np.min(positive_counts)),
        "max": int(np.max(positive_counts))
    },
    "query_weighted_metrics": {
        k: round(float(np.mean(v)), 4)
        for k, v in metrics.items()
    },
    "task_weighted_metrics": {
        k: round(float(np.mean([
            np.mean(per_task[t][k]) for t in per_task
        ])), 4)
        for k in metrics
    }
}

print("\n=== ATCODER PILOT RESULTS ===")
print(json.dumps(report, indent=2, ensure_ascii=False))

In [ ]:
%pip -q install pymongo==4.10.1 scikit-learn==1.5.2

import zipfile, re, random, json
import numpy as np
from collections import defaultdict
from bson import decode_file_iter
from sklearn.feature_extraction.text import TfidfVectorizer

ARCHIVE = "/content/AtCoder.mongo.zip"

# 1. Читаем код и группируем программы по задаче.
groups = defaultdict(lambda: {"java": {}, "py": {}})
pattern = re.compile(r"/C(\d+)/P(\d+)(?:/|$)", re.I)

with zipfile.ZipFile(ARCHIVE) as z:
    name = next(n for n in z.namelist()
                if n.endswith("code_store.bson"))

    with z.open(name) as f:
        for d in decode_file_iter(f):
            lang = str(d.get("language", "")).lower()
            if lang not in ("java", "py"):
                continue

            match = pattern.search(str(d.get("sourceFile", "")))
            if not match:
                continue

            task = f"C:{int(match.group(1))}-P:{int(match.group(2))}"
            code = str(d.get("code") or "").strip()

            if code:
                groups[task][lang].setdefault(code, code)

java_tasks = sorted(
    t for t, g in groups.items() if g["java"]
)
shared_tasks = sorted(
    t for t in java_tasks if groups[t]["py"]
)

# 2. Выбираем до пяти Python-запросов на каждую общую задачу.
rng = random.Random(42)
queries = []

for task in shared_tasks:
    codes = list(groups[task]["py"])
    rng.shuffle(codes)
    queries.extend((task, code) for code in codes[:5])

# Полный индекс: все уникальные Java-решения.
candidates = [
    (task, code)
    for task in java_tasks
    for code in groups[task]["java"]
]

candidate_tasks = np.array([t for t, _ in candidates])
candidate_codes = [c for _, c in candidates]
query_tasks = [t for t, _ in queries]
query_codes = [c for _, c in queries]

print("Задач с обоими языками:", len(shared_tasks))
print("Python-запросов:", len(queries))
print("Java-кандидатов:", len(candidates))
print("Строю TF-IDF индекс...")

# 3. Индексируем код тем же методом, что и в предыдущем эксперименте.
token_pattern = re.compile(
    r"[A-Za-z_]\w*|\d+(?:\.\d+)?|==|!=|<=|>=|&&|\|\||"
    r"\+\+|--|->|[^\s]"
)

def tokenize(code):
    return token_pattern.findall(code)

vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(candidate_codes)
Q = vectorizer.transform(query_codes)

# Индексы всех Java-решений каждой задачи.
task_indices = {
    task: np.flatnonzero(candidate_tasks == task)
    for task in java_tasks
}
task_position = {task: i for i, task in enumerate(java_tasks)}

# Для каждого случайного seed выбираем по одному Java-решению на задачу.
seeds = list(range(10))
random_choices = {}

for seed in seeds:
    r = random.Random(seed)
    random_choices[seed] = {
        task: r.choice(task_indices[task].tolist())
        for task in java_tasks
    }

def evaluate_task_scores(scores_by_task, correct_task):
    scores = np.asarray([scores_by_task[t] for t in java_tasks])
    gold = scores[task_position[correct_task]]

    higher = int(np.sum(scores > gold + 1e-12))
    tied = max(1, int(np.sum(np.abs(scores - gold) <= 1e-12)))

    out = {}
    for k in (1, 5, 10):
        out[f"Success@{k}"] = (
            min(max(k - higher, 0), tied) / tied
        )

    out["MRR"] = sum(
        1.0 / (higher + j) for j in range(1, tied + 1)
    ) / tied

    return out

# 4. Рассчитываем три варианта оценки задачи и 10 случайных индексов.
metric_names = ["Success@1", "Success@5", "Success@10", "MRR"]
methods = {
    "task_max": defaultdict(list),
    "task_top3_mean": defaultdict(list),
    "task_mean": defaultdict(list)
}
random_results = {
    seed: defaultdict(list) for seed in seeds
}

print("Сравниваю методы на уровне задач...")

for start in range(0, len(queries), 32):
    batch = (Q[start:start + 32] @ X.T).toarray()

    for offset, row in enumerate(batch):
        i = start + offset
        gold_task = query_tasks[i]

        task_max = {}
        task_top3 = {}
        task_mean = {}

        for task in java_tasks:
            values = row[task_indices[task]]

            task_max[task] = float(np.max(values))

            top = np.sort(values)[-min(3, len(values)):]
            task_top3[task] = float(np.mean(top))

            task_mean[task] = float(np.mean(values))

        for name, task_scores in [
            ("task_max", task_max),
            ("task_top3_mean", task_top3),
            ("task_mean", task_mean)
        ]:
            result = evaluate_task_scores(task_scores, gold_task)
            for metric, value in result.items():
                methods[name][metric].append(value)

        for seed in seeds:
            sampled_scores = {
                task: float(row[random_choices[seed][task]])
                for task in java_tasks
            }
            result = evaluate_task_scores(sampled_scores, gold_task)

            for metric, value in result.items():
                random_results[seed][metric].append(value)

def summarize(results):
    return {
        metric: round(float(np.mean(results[metric])), 4)
        for metric in metric_names
    }

report = {
    "queries": len(queries),
    "candidate_tasks": len(java_tasks),
    "all_submissions_baseline_previous_run": {
        "Success@1": 0.4236,
        "Success@5": 0.5901,
        "Success@10": 0.6588,
        "MRR": 0.5026
    },
    "task_level_aggregations": {
        name: summarize(result)
        for name, result in methods.items()
    },
    "one_random_solution_per_task": {
        "mean_across_10_seeds": {
            metric: round(float(np.mean([
                np.mean(random_results[s][metric])
                for s in seeds
            ])), 4)
            for metric in metric_names
        },
        "std_across_10_seeds": {
            metric: round(float(np.std([
                np.mean(random_results[s][metric])
                for s in seeds
            ])), 4)
            for metric in metric_names
        }
    }
}

print("\n=== TASK-LEVEL RESULTS ===")
print(json.dumps(report, indent=2, ensure_ascii=False))

In [ ]:
import random
import json
import numpy as np
from collections import defaultdict

# Используем уже созданные в предыдущих ячейках:
# queries, candidate_tasks, X, Q

java_tasks = sorted(set(candidate_tasks))
task_position = {task: i for i, task in enumerate(java_tasks)}

indices_by_task = {
    task: np.flatnonzero(candidate_tasks == task)
    for task in java_tasks
}

# Выбираем случайные подмножества отдельно для каждого запуска.
K_VALUES = [1, 5, 10, "all"]
SEEDS = range(5)

selections = {}

for seed in SEEDS:
    rng = random.Random(seed)

    for k in K_VALUES:
        selections[(seed, k)] = {}

        for task, indices in indices_by_task.items():
            indices = indices.tolist()

            if k == "all":
                chosen = indices
            else:
                chosen = rng.sample(indices, min(k, len(indices)))

            selections[(seed, k)][task] = np.asarray(chosen, dtype=int)

metric_names = ["Success@1", "Success@5", "Success@10", "MRR"]

query_results = defaultdict(lambda: defaultdict(list))
task_results = defaultdict(
    lambda: defaultdict(lambda: defaultdict(list))
)

def ranking_metrics(task_scores, gold_task):
    gold = task_scores[task_position[gold_task]]

    higher = int(np.sum(task_scores > gold + 1e-12))
    tied = max(
        1,
        int(np.sum(np.abs(task_scores - gold) <= 1e-12))
    )

    result = {}

    for k in (1, 5, 10):
        result[f"Success@{k}"] = (
            min(max(k - higher, 0), tied) / tied
        )

    result["MRR"] = np.mean([
        1 / (higher + j)
        for j in range(1, tied + 1)
    ])

    return result

query_tasks = [task for task, code in queries]

# Считаем сходства пакетами, чтобы не создавать огромную матрицу.
print("Запускаю контролируемый эксперимент...")

for start in range(0, len(queries), 32):
    scores_batch = (Q[start:start + 32] @ X.T).toarray()

    for offset, row in enumerate(scores_batch):
        i = start + offset
        gold_task = query_tasks[i]

        for seed in SEEDS:
            for k in K_VALUES:
                selected = selections[(seed, k)]

                max_scores = np.empty(len(java_tasks))
                mean_scores = np.empty(len(java_tasks))

                for j, task in enumerate(java_tasks):
                    values = row[selected[task]]
                    max_scores[j] = np.max(values)
                    mean_scores[j] = np.mean(values)

                for method, task_scores in [
                    ("max", max_scores),
                    ("mean", mean_scores)
                ]:
                    key = (seed, k, method)
                    result = ranking_metrics(task_scores, gold_task)

                    for metric, value in result.items():
                        query_results[key][metric].append(value)
                        task_results[key][gold_task][metric].append(value)

# Сводим результаты по нескольким случайным выборкам.
report = {
    "queries": len(queries),
    "candidate_tasks": len(java_tasks),
    "results": {}
}

for k in K_VALUES:
    report["results"][str(k)] = {}

    for method in ("max", "mean"):
        seed_summaries = []

        # Для "all" выборка одинакова при любом seed.
        used_seeds = [0] if k == "all" else list(SEEDS)

        for seed in used_seeds:
            key = (seed, k, method)

            query_means = {
                m: float(np.mean(query_results[key][m]))
                for m in metric_names
            }

            task_means = {
                m: float(np.mean([
                    np.mean(task_results[key][task][m])
                    for task in task_results[key]
                ]))
                for m in metric_names
            }

            seed_summaries.append({
                "query_weighted": query_means,
                "task_weighted": task_means
            })

        report["results"][str(k)][method] = {
            "query_weighted_mean": {
                m: round(float(np.mean([
                    s["query_weighted"][m] for s in seed_summaries
                ])), 4)
                for m in metric_names
            },
            "query_weighted_sd": {
                m: round(float(np.std([
                    s["query_weighted"][m] for s in seed_summaries
                ])), 4)
                for m in metric_names
            },
            "task_weighted_mean": {
                m: round(float(np.mean([
                    s["task_weighted"][m] for s in seed_summaries
                ])), 4)
                for m in metric_names
            }
        }

print("\n=== CONTROLLED K RESULTS ===")
print(json.dumps(report, indent=2, ensure_ascii=False))

Запускаю контролируемый эксперимент...

=== CONTROLLED K RESULTS ===
{
  "queries": 1603,
  "candidate_tasks": 360,
  "results": {
    "1": {
      "max": {
        "query_weighted_mean": {
          "Success@1": 0.202,
          "Success@5": 0.3495,
          "Success@10": 0.4277,
          "MRR": 0.2794
        },
        "query_weighted_sd": {
          "Success@1": 0.002,
          "Success@5": 0.0089,
          "Success@10": 0.0099,
          "MRR": 0.0049
        },
        "task_weighted_mean": {
          "Success@1": 0.1975,
          "Success@5": 0.3456,
          "Success@10": 0.4244,
          "MRR": 0.2753
        }
      },
      "mean": {
        "query_weighted_mean": {
          "Success@1": 0.202,
          "Success@5": 0.3495,
          "Success@10": 0.4277,
          "MRR": 0.2794
        },
        "query_weighted_sd": {
          "Success@1": 0.002,
          "Success@5": 0.0089,
          "Success@10": 0.0099,
          "MRR": 0.0049
        },
        "task_weig

In [ ]:
import numpy as np
import json

max_results = task_results[(0, "all", "max")]
mean_results = task_results[(0, "all", "mean")]

tasks = sorted(set(max_results) & set(mean_results))
metrics = ["Success@1", "Success@5", "Success@10", "MRR"]

rng = np.random.default_rng(20261028)
report = {
    "tasks": len(tasks),
    "bootstrap_repetitions": 10000,
    "results": {}
}

for metric in metrics:
    # Сначала усредняем запросы внутри каждой задачи.
    differences = np.array([
        np.mean(mean_results[t][metric])
        - np.mean(max_results[t][metric])
        for t in tasks
    ])

    # Затем выбираем задачи с возвращением.
    boot = np.empty(10000)
    for i in range(len(boot)):
        sample = rng.choice(
            differences, size=len(differences), replace=True
        )
        boot[i] = sample.mean()

    report["results"][metric] = {
        "mean_difference": round(float(differences.mean()), 4),
        "ci_95": [
            round(float(np.quantile(boot, 0.025)), 4),
            round(float(np.quantile(boot, 0.975)), 4)
        ],
        "tasks_improved_pct": round(
            100 * float(np.mean(differences > 1e-9)), 2
        ),
        "tasks_worsened_pct": round(
            100 * float(np.mean(differences < -1e-9)), 2
        ),
        "tasks_tied_pct": round(
            100 * float(np.mean(np.abs(differences) <= 1e-9)), 2
        )
    }

print(json.dumps(report, indent=2, ensure_ascii=False))

{
  "tasks": 332,
  "bootstrap_repetitions": 10000,
  "results": {
    "Success@1": {
      "mean_difference": 0.0461,
      "ci_95": [
        0.0086,
        0.083
      ],
      "tasks_improved_pct": 37.05,
      "tasks_worsened_pct": 26.81,
      "tasks_tied_pct": 36.14
    },
    "Success@5": {
      "mean_difference": 0.0707,
      "ci_95": [
        0.0364,
        0.1064
      ],
      "tasks_improved_pct": 34.34,
      "tasks_worsened_pct": 21.69,
      "tasks_tied_pct": 43.98
    },
    "Success@10": {
      "mean_difference": 0.0776,
      "ci_95": [
        0.0473,
        0.1076
      ],
      "tasks_improved_pct": 34.34,
      "tasks_worsened_pct": 18.37,
      "tasks_tied_pct": 47.29
    },
    "MRR": {
      "mean_difference": 0.0577,
      "ci_95": [
        0.0263,
        0.089
      ],
      "tasks_improved_pct": 55.12,
      "tasks_worsened_pct": 37.05,
      "tasks_tied_pct": 7.83
    }
  }
}


In [ ]:
import random
import json
import numpy as np
from collections import defaultdict

# Используем индекс и запросы из предыдущих экспериментов.
required = ["X", "Q", "candidate_tasks", "queries"]
missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        f"Не найдены переменные {missing}. "
        "Напиши мне, и я помогу восстановить эксперимент."
    )

java_tasks = sorted(set(candidate_tasks))
task_position = {t: i for i, t in enumerate(java_tasks)}
indices_by_task = {
    t: np.flatnonzero(candidate_tasks == t).tolist()
    for t in java_tasks
}

query_tasks = [t for t, _ in queries]
metrics = ["Success@1", "Success@5", "Success@10", "MRR"]
k_values = [1, 5, 10, "all"]
seeds = range(5)

# Создаём вложенные подвыборки: k=1 ⊂ k=5 ⊂ k=10 ⊂ all.
selections = {}
for seed in seeds:
    rng = random.Random(seed)
    for task, indices in indices_by_task.items():
        order = indices.copy()
        rng.shuffle(order)
        for k in k_values:
            selections[seed, task, k] = (
                order if k == "all"
                else order[:min(k, len(order))]
            )

def measure(scores, gold_task):
    gold = scores[task_position[gold_task]]
    higher = int(np.sum(scores > gold + 1e-12))
    tied = max(1, int(np.sum(np.abs(scores - gold) <= 1e-12)))

    result = {}
    for k in (1, 5, 10):
        result[f"Success@{k}"] = (
            min(max(k - higher, 0), tied) / tied
        )
    result["MRR"] = np.mean([
        1 / (higher + j) for j in range(1, tied + 1)
    ])
    return result

# Results сохраняет исходы по запросам и задачам для каждой конфигурации.
results = defaultdict(lambda: defaultdict(list))
per_task = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

print("Запускаю вложенное сравнение...")

for start in range(0, len(queries), 32):
    batch = (Q[start:start + 32] @ X.T).toarray()

    for offset, row in enumerate(batch):
        gold_task = query_tasks[start + offset]

        for seed in seeds:
            for k in k_values:
                chosen = selections
                for method in ("max", "mean"):
                    task_scores = np.empty(len(java_tasks))

                    for j, task in enumerate(java_tasks):
                        vals = row[chosen[seed, task, k]]
                        task_scores[j] = (
                            np.max(vals) if method == "max"
                            else np.mean(vals)
                        )

                    key = (seed, k, method)
                    outcome = measure(task_scores, gold_task)

                    for metric, value in outcome.items():
                        results[key][metric].append(value)
                        per_task[key][gold_task][metric].append(value)

report = {"queries": len(queries), "tasks": len(java_tasks), "results": {}}

for k in k_values:
    report["results"][str(k)] = {}

    for method in ("max", "mean"):
        seeds_summary = []

        for seed in seeds:
            key = (seed, k, method)

            query_avg = {
                m: float(np.mean(results[key][m]))
                for m in metrics
            }

            task_avg = {
                m: float(np.mean([
                    np.mean(per_task[key][t][m])
                    for t in per_task[key]
                ]))
                for m in metrics
            }

            seeds_summary.append({
                "query": query_avg,
                "task": task_avg
            })

        report["results"][str(k)][method] = {
            "query_weighted_mean": {
                m: round(float(np.mean([s["query"][m] for s in seeds_summary])), 4)
                for m in metrics
            },
            "task_weighted_mean": {
                m: round(float(np.mean([s["task"][m] for s in seeds_summary])), 4)
                for m in metrics
            },
            "task_weighted_sd_across_seeds": {
                m: round(float(np.std([s["task"][m] for s in seeds_summary])), 4)
                for m in metrics
            }
        }

    report["results"][str(k)]["mean_minus_max_task_weighted"] = {
        m: round(float(np.mean([
            per_task[(seed, k, "mean")][t][m].__class__ is list
            and np.mean(per_task[(seed, k, "mean")][t][m])
            - np.mean(per_task[(seed, k, "max")][t][m])
            or 0
            for seed in seeds for t in []
        ])), 4)
        for m in []
    }

print("\n=== NESTED SUBSAMPLING RESULTS ===")
print(json.dumps(report, indent=2, ensure_ascii=False))

Запускаю вложенное сравнение...

=== NESTED SUBSAMPLING RESULTS ===
{
  "queries": 1603,
  "tasks": 360,
  "results": {
    "1": {
      "max": {
        "query_weighted_mean": {
          "Success@1": 0.1945,
          "Success@5": 0.344,
          "Success@10": 0.4228,
          "MRR": 0.273
        },
        "task_weighted_mean": {
          "Success@1": 0.1906,
          "Success@5": 0.3378,
          "Success@10": 0.4177,
          "MRR": 0.2686
        },
        "task_weighted_sd_across_seeds": {
          "Success@1": 0.0045,
          "Success@5": 0.0093,
          "Success@10": 0.0119,
          "MRR": 0.0053
        }
      },
      "mean": {
        "query_weighted_mean": {
          "Success@1": 0.1945,
          "Success@5": 0.344,
          "Success@10": 0.4228,
          "MRR": 0.273
        },
        "task_weighted_mean": {
          "Success@1": 0.1906,
          "Success@5": 0.3378,
          "Success@10": 0.4177,
          "MRR": 0.2686
        },
        "task_we

In [ ]:
import random
import json
import numpy as np
from collections import defaultdict

required = ["X", "Q", "candidate_tasks", "queries"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Не найдены переменные: {missing}")

candidate_task_array = np.asarray(candidate_tasks)
all_task_indices = {
    t: np.flatnonzero(candidate_task_array == t)
    for t in sorted(set(candidate_task_array))
}

# Фиксированная когорта: у каждого кандидата-задачи не менее 10 решений.
eligible_tasks = sorted(
    t for t, ix in all_task_indices.items() if len(ix) >= 10
)
eligible_set = set(eligible_tasks)

query_indices = [
    i for i, (task, _) in enumerate(queries)
    if task in eligible_set
]
query_tasks = [queries[i][0] for i in query_indices]
eval_tasks = sorted(set(query_tasks))

if not query_indices or not eval_tasks:
    raise RuntimeError("Не осталось запросов после фильтрации.")

# Оставляем только кандидатов из фиксированной когорты.
candidate_indices = np.flatnonzero(
    np.isin(candidate_task_array, eligible_tasks)
)
X_fixed = X[candidate_indices, :]
candidate_tasks_fixed = candidate_task_array[candidate_indices]
Q_fixed = Q[query_indices]

local_indices = {
    t: np.flatnonzero(candidate_tasks_fixed == t)
    for t in eligible_tasks
}
task_pos = {t: i for i, t in enumerate(eligible_tasks)}

# Во всех режимах используем одну и ту же когорту задач.
K_VALUES = [1, 5, 10, "all"]
SEEDS = range(5)
selections = {}

for seed in SEEDS:
    rng = random.Random(seed)
    for task in eligible_tasks:
        order = local_indices[task].tolist()
        rng.shuffle(order)

        for k in K_VALUES:
            selections[seed, task, k] = (
                order if k == "all" else order[:k]
            )

metrics = ["Success@1", "Success@5", "Success@10", "MRR"]
per_task = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

def evaluate(scores, gold_task):
    correct = scores[task_pos[gold_task]]
    higher = int(np.sum(scores > correct + 1e-12))
    tied = max(1, int(np.sum(np.abs(scores - correct) <= 1e-12)))

    result = {}
    for k in (1, 5, 10):
        result[f"Success@{k}"] = (
            min(max(k - higher, 0), tied) / tied
        )
    result["MRR"] = np.mean([
        1 / (higher + j) for j in range(1, tied + 1)
    ])
    return result

print("Задач-кандидатов:", len(eligible_tasks))
print("Задач с Python-запросами:", len(eval_tasks))
print("Запросов:", len(query_indices))
print("Java-кандидатов в фиксированной когорте:", X_fixed.shape[0])
print("Запускаю фиксированное сравнение...")

for start in range(0, len(query_indices), 32):
    sims = (Q_fixed[start:start + 32] @ X_fixed.T).toarray()

    for offset, row in enumerate(sims):
        qi = start + offset
        gold_task = query_tasks[qi]

        for seed in SEEDS:
            for k in K_VALUES:
                for method in ("max", "mean"):
                    task_scores = np.empty(len(eligible_tasks))

                    for j, task in enumerate(eligible_tasks):
                        vals = row[selections[seed, task, k]]
                        task_scores[j] = (
                            np.max(vals) if method == "max"
                            else np.mean(vals)
                        )

                    outcome = evaluate(task_scores, gold_task)
                    key = (seed, k, method)

                    for metric, value in outcome.items():
                        per_task[key][gold_task][metric].append(value)

# Сводка по метрикам и парная разница mean - max на уровне задач.
report = {
    "eligible_candidate_tasks": len(eligible_tasks),
    "query_tasks": len(eval_tasks),
    "queries": len(query_indices),
    "results": {}
}
rng = np.random.default_rng(20261028)

for k in K_VALUES:
    report["results"][str(k)] = {}

    for method in ("max", "mean"):
        per_seed = []

        for seed in SEEDS:
            key = (seed, k, method)
            per_seed.append({
                metric: np.mean([
                    np.mean(per_task[key][task][metric])
                    for task in eval_tasks
                ])
                for metric in metrics
            })

        report["results"][str(k)][method] = {
            metric: {
                "mean": round(float(np.mean([s[metric] for s in per_seed])), 4),
                "sd_across_seeds": round(float(np.std([s[metric] for s in per_seed])), 4)
            }
            for metric in metrics
        }

    report["results"][str(k)]["mean_minus_max"] = {}

    for metric in metrics:
        diffs = []
        for task in eval_tasks:
            diffs.append(np.mean([
                np.mean(per_task[(seed, k, "mean")][task][metric])
                - np.mean(per_task[(seed, k, "max")][task][metric])
                for seed in SEEDS
            ]))

        diffs = np.asarray(diffs)
        boot = np.array([
            np.mean(rng.choice(diffs, size=len(diffs), replace=True))
            for _ in range(5000)
        ])

        report["results"][str(k)]["mean_minus_max"][metric] = {
            "difference": round(float(diffs.mean()), 4),
            "ci95": [
                round(float(np.quantile(boot, 0.025)), 4),
                round(float(np.quantile(boot, 0.975)), 4)
            ]
        }

print("\n=== FIXED COHORT RESULTS ===")
print(json.dumps(report, indent=2, ensure_ascii=False))

Задач-кандидатов: 294
Задач с Python-запросами: 290
Запросов: 1438
Java-кандидатов в фиксированной когорте: 18116
Запускаю фиксированное сравнение...

=== FIXED COHORT RESULTS ===
{
  "eligible_candidate_tasks": 294,
  "query_tasks": 290,
  "queries": 1438,
  "results": {
    "1": {
      "max": {
        "Success@1": {
          "mean": 0.2039,
          "sd_across_seeds": 0.01
        },
        "Success@5": {
          "mean": 0.3539,
          "sd_across_seeds": 0.013
        },
        "Success@10": {
          "mean": 0.434,
          "sd_across_seeds": 0.014
        },
        "MRR": {
          "mean": 0.2821,
          "sd_across_seeds": 0.0105
        }
      },
      "mean": {
        "Success@1": {
          "mean": 0.2039,
          "sd_across_seeds": 0.01
        },
        "Success@5": {
          "mean": 0.3539,
          "sd_across_seeds": 0.013
        },
        "Success@10": {
          "mean": 0.434,
          "sd_across_seeds": 0.014
        },
        "MRR": {
  

In [ ]:
import numpy as np
import json

# Используем результаты предыдущего фиксированного эксперимента.
if "per_task" not in globals() or "eval_tasks" not in globals():
    raise RuntimeError(
        "Результаты предыдущего эксперимента отсутствуют. "
        "Напиши мне, и мы восстановим нужные переменные."
    )

SEEDS = range(5)
K_VALUES = [1, 5, 10, "all"]
METHODS = ["max", "mean"]
METRICS = ["Success@1", "Success@5", "Success@10", "MRR"]
COMPARISONS = [(1, 5), (1, 10), (5, 10), (10, "all")]

rng = np.random.default_rng(20261028)
tasks = sorted(eval_tasks)

def average_task_score(task, k, method, metric):
    # Сначала усредняем запросы внутри задачи,
    # затем усредняем результаты по пяти случайным seeds.
    return float(np.mean([
        np.mean(per_task[(seed, k, method)][task][metric])
        for seed in SEEDS
    ]))

report = {
    "tasks": len(tasks),
    "bootstrap_repetitions": 10000,
    "comparisons": {}
}

for method in METHODS:
    report["comparisons"][method] = {}

    for k1, k2 in COMPARISONS:
        key = f"{k1} -> {k2}"
        report["comparisons"][method][key] = {}

        for metric in METRICS:
            differences = np.array([
                average_task_score(task, k2, method, metric)
                - average_task_score(task, k1, method, metric)
                for task in tasks
            ])

            boot = np.empty(10000)

            for i in range(len(boot)):
                sample = rng.choice(
                    differences,
                    size=len(differences),
                    replace=True
                )
                boot[i] = sample.mean()

            report["comparisons"][method][key][metric] = {
                "mean_difference": round(float(differences.mean()), 4),
                "ci95": [
                    round(float(np.quantile(boot, 0.025)), 4),
                    round(float(np.quantile(boot, 0.975)), 4)
                ],
                "tasks_improved_pct": round(
                    100 * float(np.mean(differences > 1e-9)), 2
                ),
                "tasks_worsened_pct": round(
                    100 * float(np.mean(differences < -1e-9)), 2
                )
            }

print("=== REFERENCE COUNT BOOTSTRAP ===")
print(json.dumps(report, indent=2, ensure_ascii=False))

=== REFERENCE COUNT BOOTSTRAP ===
{
  "tasks": 290,
  "bootstrap_repetitions": 10000,
  "comparisons": {
    "max": {
      "1 -> 5": {
        "Success@1": {
          "mean_difference": 0.137,
          "ci95": [
            0.1177,
            0.1568
          ],
          "tasks_improved_pct": 72.41,
          "tasks_worsened_pct": 15.52
        },
        "Success@5": {
          "mean_difference": 0.193,
          "ci95": [
            0.1732,
            0.2134
          ],
          "tasks_improved_pct": 85.17,
          "tasks_worsened_pct": 9.31
        },
        "Success@10": {
          "mean_difference": 0.2057,
          "ci95": [
            0.1848,
            0.2267
          ],
          "tasks_improved_pct": 85.52,
          "tasks_worsened_pct": 7.59
        },
        "MRR": {
          "mean_difference": 0.1608,
          "ci95": [
            0.1436,
            0.1788
          ],
          "tasks_improved_pct": 87.93,
          "tasks_worsened_pct": 11.72
    

In [ ]:
import numpy as np
import json

# Используем результаты предыдущего фиксированного эксперимента.
if "per_task" not in globals() or "eval_tasks" not in globals():
    raise RuntimeError(
        "Результаты предыдущего эксперимента отсутствуют. "
        "Напиши мне, и мы восстановим нужные переменные."
    )

SEEDS = range(5)
K_VALUES = [1, 5, 10, "all"]
METHODS = ["max", "mean"]
METRICS = ["Success@1", "Success@5", "Success@10", "MRR"]
COMPARISONS = [(1, 5), (1, 10), (5, 10), (10, "all")]

rng = np.random.default_rng(20261028)
tasks = sorted(eval_tasks)

def average_task_score(task, k, method, metric):
    # Сначала усредняем запросы внутри задачи,
    # затем усредняем результаты по пяти случайным seeds.
    return float(np.mean([
        np.mean(per_task[(seed, k, method)][task][metric])
        for seed in SEEDS
    ]))

report = {
    "tasks": len(tasks),
    "bootstrap_repetitions": 10000,
    "comparisons": {}
}

for method in METHODS:
    report["comparisons"][method] = {}

    for k1, k2 in COMPARISONS:
        key = f"{k1} -> {k2}"
        report["comparisons"][method][key] = {}

        for metric in METRICS:
            differences = np.array([
                average_task_score(task, k2, method, metric)
                - average_task_score(task, k1, method, metric)
                for task in tasks
            ])

            boot = np.empty(10000)

            for i in range(len(boot)):
                sample = rng.choice(
                    differences,
                    size=len(differences),
                    replace=True
                )
                boot[i] = sample.mean()

            report["comparisons"][method][key][metric] = {
                "mean_difference": round(float(differences.mean()), 4),
                "ci95": [
                    round(float(np.quantile(boot, 0.025)), 4),
                    round(float(np.quantile(boot, 0.975)), 4)
                ],
                "tasks_improved_pct": round(
                    100 * float(np.mean(differences > 1e-9)), 2
                ),
                "tasks_worsened_pct": round(
                    100 * float(np.mean(differences < -1e-9)), 2
                )
            }

print("=== REFERENCE COUNT BOOTSTRAP ===")
print(json.dumps(report, indent=2, ensure_ascii=False))

=== REFERENCE COUNT BOOTSTRAP ===
{
  "tasks": 290,
  "bootstrap_repetitions": 10000,
  "comparisons": {
    "max": {
      "1 -> 5": {
        "Success@1": {
          "mean_difference": 0.137,
          "ci95": [
            0.1177,
            0.1568
          ],
          "tasks_improved_pct": 72.41,
          "tasks_worsened_pct": 15.52
        },
        "Success@5": {
          "mean_difference": 0.193,
          "ci95": [
            0.1732,
            0.2134
          ],
          "tasks_improved_pct": 85.17,
          "tasks_worsened_pct": 9.31
        },
        "Success@10": {
          "mean_difference": 0.2057,
          "ci95": [
            0.1848,
            0.2267
          ],
          "tasks_improved_pct": 85.52,
          "tasks_worsened_pct": 7.59
        },
        "MRR": {
          "mean_difference": 0.1608,
          "ci95": [
            0.1436,
            0.1788
          ],
          "tasks_improved_pct": 87.93,
          "tasks_worsened_pct": 11.72
    

In [ ]:
%pip -q install datasets==3.1.0 huggingface_hub==0.26.2 pyarrow==21.0.0

import json
import ast
from collections import Counter, defaultdict
from datasets import load_dataset

REPO = "hq-bench/coreb"
SPLIT = "release_v2603"

# Hardcoded revision for exact reproducibility
revision = "e3f40f2609b4c52cc6d750594a87b29b370c3e06"

print("CoREB revision:", revision)
print("Загружаю таблицы...")

corpus = list(load_dataset(
    REPO, "code_corpus", split=SPLIT, revision=revision
))
queries = list(load_dataset(
    REPO, "code2code_queries", split=SPLIT, revision=revision
))
qrels = list(load_dataset(
    REPO, "code2code_qrels", split=SPLIT, revision=revision
))

def get_meta(row):
    m = row.get("meta") or {}
    if isinstance(m, dict):
        return m
    if isinstance(m, str):
        for parser in (json.loads, ast.literal_eval):
            try:
                result = parser(m)
                if isinstance(result, dict):
                    return result
            except Exception:
                pass
    return {}

code_by_id = {
    row.get("code_id"): row
    for row in corpus
    if row.get("code_id")
}

query_ids = {q.get("query_id") for q in queries}
cross_queries = [
    q for q in queries
    if q.get("subtask") == "c2c_cross_lang"
]
cross_ids = {q["query_id"] for q in cross_queries}

cross_qrels = [
    r for r in qrels
    if r.get("query_id") in cross_ids
]

qrels_by_query = defaultdict(list)
for r in cross_qrels:
    qrels_by_query[r["query_id"]].append(r)

corpus_languages = Counter(
    str(r.get("language", "missing")) for r in corpus
)
corpus_models = Counter(
    str(r.get("model", "missing")) for r in corpus
)
query_subtasks = Counter(
    str(q.get("subtask", "missing")) for q in queries
)
relevance_counts = Counter(
    int(r.get("relevance", -1)) for r in cross_qrels
)

positive_counts = []
hard_negative_counts = []
missing_qrel_docs = 0
anchors_found = 0
anchors_in_qrels = 0
target_language_counts = Counter()
problem_ids = Counter()
problem_language_groups = Counter()
meta_coverage = 0

for q in cross_queries:
    qid = q["query_id"]
    meta = get_meta(q)

    anchor_id = meta.get("anchor_code_id")
    if anchor_id in code_by_id:
        anchors_found += 1

    rows = qrels_by_query[qid]
    positive = [
        r for r in rows if int(r.get("relevance", -1)) == 2
    ]
    hard = [
        r for r in rows if int(r.get("relevance", -1)) == 1
    ]

    positive_counts.append(len(positive))
    hard_negative_counts.append(len(hard))

    if any(r.get("doc_id") == anchor_id for r in rows):
        anchors_in_qrels += 1

    targets = meta.get("target_languages") or []
    if isinstance(targets, list):
        target_language_counts.update(map(str, targets))
    elif targets:
        target_language_counts[str(targets)] += 1

    pid = meta.get("source_problem_id")
    if pid is not None:
        problem_ids[str(pid)] += 1

    for r in rows:
        if r.get("doc_id") not in code_by_id:
            missing_qrel_docs += 1

for row in corpus:
    meta = get_meta(row)
    pid = (
        meta.get("source_problem_id")
        or row.get("source_problem_id")
        or row.get("problem_id")
    )
    if pid is not None:
        meta_coverage += 1
        problem_language_groups[
            (str(pid), str(row.get("language", "missing")))
        ] += 1

group_sizes = Counter(problem_language_groups.values())

def distribution(values):
    if not values:
        return {"count": 0, "min": None, "median": None, "mean": None, "max": None}
    vals = sorted(values)
    return {
        "count": len(vals),
        "min": min(vals),
        "median": vals[len(vals) // 2],
        "mean": round(sum(vals) / len(vals), 3),
        "max": max(vals)
    }

report = {
    "revision": revision,
    "rows": {
        "code_corpus": len(corpus),
        "code2code_queries": len(queries),
        "cross_language_queries": len(cross_queries),
        "code2code_qrels": len(qrels),
        "cross_language_qrels": len(cross_qrels)
    },
    "query_subtasks": dict(query_subtasks),
    "corpus_languages": dict(corpus_languages),
    "corpus_models": dict(corpus_models),
    "cross_language_relevance_counts": dict(relevance_counts),
    "positive_qrels_per_query": distribution(positive_counts),
    "hard_negatives_per_query": distribution(hard_negative_counts),
    "anchors_found_in_corpus": anchors_found,
    "anchors_appearing_in_qrels": anchors_in_qrels,
    "qrel_documents_missing_from_corpus": missing_qrel_docs,
    "target_language_mentions": dict(target_language_counts),
    "problem_id_metadata_coverage": {
        "documents_with_problem_id": meta_coverage,
        "total_documents": len(corpus)
    },
    "documents_per_problem_language_group": dict(group_sizes),
    "sample_query_fields": list(queries[0].keys()) if queries else [],
    "sample_code_fields": list(corpus[0].keys()) if corpus else [],
    "sample_query_metadata": get_meta(queries[0]) if queries else {}
}

print("\n=== COREB ROW-LEVEL AUDIT ===")
print(json.dumps(report, ensure_ascii=False, indent=2))

In [ ]:
# ============================================================
# CoREB ROW-LEVEL AUDIT
# Dataset: hq-bench/coreb
# Split: release_v2603
# ============================================================

%pip -q install datasets==3.1.0 huggingface_hub==0.26.2 pyarrow==21.0.0

from datasets import load_dataset
from collections import Counter, defaultdict
from statistics import median
import json

REPO = "hq-bench/coreb"
SPLIT = "release_v2603"
REVISION = "e3f40f2609b4c52cc6d750594a87b29b370c3e06"

print("=" * 70)
print("COREB ROW-LEVEL AUDIT — START")
print("=" * 70)

print("Dataset:", REPO)
print("Split:", SPLIT)
print("Revision:", REVISION)

# ------------------------------------------------------------
# 1. Load datasets
# ------------------------------------------------------------

print("\n[1] Loading datasets...")

corpus_ds = load_dataset(
    REPO, "code_corpus", split=SPLIT, revision=REVISION
)

queries_ds = load_dataset(
    REPO, "code2code_queries", split=SPLIT, revision=REVISION
)

qrels_ds = load_dataset(
    REPO, "code2code_qrels", split=SPLIT, revision=REVISION
)

corpus = [dict(row) for row in corpus_ds]
queries = [dict(row) for row in queries_ds]
qrels = [dict(row) for row in qrels_ds]

print("\n[2] Dataset sizes")
print("-" * 50)
print("Code corpus rows:", len(corpus))
print("Code-to-code queries:", len(queries))
print("Code-to-code relevance judgments:", len(qrels))

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def shorten(value, limit=180):
    """Keep output readable by shortening long values."""
    if value is None:
        return "None"
    if isinstance(value, str):
        value = value.replace("\n", "\\n")
    else:
        value = json.dumps(value, ensure_ascii=False, default=str)
    if len(value) > limit:
        return value[:limit] + "..."
    return value

def show_schema(name, records):
    print(f"\n{name}")
    print("-" * 50)
    if not records:
        print("Dataset is empty.")
        return
    print("Columns:", list(records[0].keys()))
    print("First row:")
    for key, value in records[0].items():
        print(f"  {key}: {shorten(value)}")

def duplicate_count(records, key):
    values = [
        row.get(key)
        for row in records
        if row.get(key) is not None
    ]
    return len(values) - len(set(values))

def to_int(value):
    try:
        return int(value)
    except (ValueError, TypeError):
        return None

def normalize_language(value):
    if value is None:
        return None
    return str(value).strip().lower()

# ------------------------------------------------------------
# 3. Inspect schemas
# ------------------------------------------------------------

print("\n[3] Inspecting dataset schemas")
show_schema("CODE CORPUS", corpus)
show_schema("CODE-TO-CODE QUERIES", queries)
show_schema("CODE-TO-CODE QRELS", qrels)

# ------------------------------------------------------------
# 4. Check ID integrity and duplicates
# ------------------------------------------------------------

print("\n[4] Identifier integrity")
print("-" * 50)

corpus_ids = {
    row.get("code_id")
    for row in corpus
    if row.get("code_id") is not None
}

query_ids = {
    row.get("query_id")
    for row in queries
    if row.get("query_id") is not None
}

print("Unique code IDs:", len(corpus_ids))
print("Duplicate code IDs:", duplicate_count(corpus, "code_id"))
print("Unique query IDs:", len(query_ids))
print("Duplicate query IDs:", duplicate_count(queries, "query_id"))

qrel_pairs = [
    (row.get("query_id"), row.get("doc_id"))
    for row in qrels
    if row.get("query_id") is not None and row.get("doc_id") is not None
]
print("Duplicate query-document pairs:", len(qrel_pairs) - len(set(qrel_pairs)))

missing_query_refs = [
    row for row in qrels
    if row.get("query_id") not in query_ids
]
missing_doc_refs = [
    row for row in qrels
    if row.get("doc_id") not in corpus_ids
]
print("Qrels referencing missing queries:", len(missing_query_refs))
print("Qrels referencing missing documents:", len(missing_doc_refs))

# ------------------------------------------------------------
# 5. Inspect the code corpus
# ------------------------------------------------------------

print("\n[5] Code corpus composition")
print("-" * 50)

language_counts = Counter(
    str(row.get("language", "<missing>"))
    for row in corpus
)
print("Languages:")
for language, count in language_counts.most_common():
    print(f"  {language}: {count}")

for field, label in [
    ("model", "Generator models"),
    ("meta_passed", "Pass-status values"),
    ("meta_source_problem_id", "Source problem IDs"),
]:
    if any(field in row for row in corpus):
        values = Counter(
            str(row.get(field, "<missing>"))
            for row in corpus
        )
        print(f"\n{label}:")
        if field == "meta_source_problem_id":
            unique_values = {
                row.get(field)
                for row in corpus
                if row.get(field) is not None
            }
            print("  Distinct values:", len(unique_values))
        else:
            for value, count in values.most_common():
                print(f"  {value}: {count}")

# ------------------------------------------------------------
# 6. Identify cross-language queries
# ------------------------------------------------------------

print("\n[6] Cross-language query audit")
print("-" * 50)

cross_queries = []
for row in queries:
    query_id = str(row.get("query_id", "")).lower()
    subtask = str(row.get("subtask", "")).lower()
    if "cross_lang" in query_id or "cross_lang" in subtask:
        cross_queries.append(row)

cross_query_ids = {
    row.get("query_id")
    for row in cross_queries
    if row.get("query_id") is not None
}

print("All code-to-code queries:", len(queries))
print("Cross-language queries:", len(cross_queries))
print("Other code-to-code queries:", len(queries) - len(cross_queries))

print("\nCross-language query examples:")
for row in cross_queries[:3]:
    print({
        key: shorten(value, 120)
        for key, value in row.items()
    })

# ------------------------------------------------------------
# 7. Analyze relevance labels
# ------------------------------------------------------------

print("\n[7] Relevance label audit")
print("-" * 50)

def label_counts(records):
    return Counter(
        str(to_int(row.get("relevance")))
        if to_int(row.get("relevance")) is not None
        else "<missing>"
        for row in records
    )

print("All code-to-code qrels:")
for label, count in sorted(label_counts(qrels).items()):
    print(f"  relevance={label}: {count}")

cross_qrels = [
    row for row in qrels
    if row.get("query_id") in cross_query_ids
]
print("\nCross-language qrels:", len(cross_qrels))
print("Cross-language relevance labels:")
cross_label_counts = label_counts(cross_qrels)
for label, count in sorted(cross_label_counts.items()):
    print(f"  relevance={label}: {count}")

cross_qrel_query_ids = {
    row.get("query_id")
    for row in cross_qrels
    if row.get("query_id") is not None
}
print("Cross-language queries with judgments:", len(cross_qrel_query_ids))
print("Cross-language queries without judgments:", len(cross_query_ids - cross_qrel_query_ids))

# ------------------------------------------------------------
# 8. Positives and hard negatives per query
# ------------------------------------------------------------

print("\n[8] Positives and hard negatives per query")
print("-" * 50)

relevance_by_query = defaultdict(Counter)
for row in cross_qrels:
    query_id = row.get("query_id")
    relevance = to_int(row.get("relevance"))
    if query_id is not None and relevance is not None:
        relevance_by_query[query_id][relevance] += 1

positive_counts = [
    relevance_by_query[qid].get(2, 0)
    for qid in cross_query_ids
]
hard_negative_counts = [
    relevance_by_query[qid].get(1, 0)
    for qid in cross_query_ids
]

if positive_counts:
    print("Positive documents (relevance=2):")
    print("  Total:", sum(positive_counts))
    print("  Queries with zero positives:", sum(x == 0 for x in positive_counts))
    print("  Minimum per query:", min(positive_counts))
    print("  Median per query:", median(positive_counts))
    print("  Maximum per query:", max(positive_counts))

if hard_negative_counts:
    print("\nHard negatives (relevance=1):")
    print("  Total:", sum(hard_negative_counts))
    print("  Queries with zero hard negatives:", sum(x == 0 for x in hard_negative_counts))
    print("  Minimum per query:", min(hard_negative_counts))
    print("  Median per query:", median(hard_negative_counts))
    print("  Maximum per query:", max(hard_negative_counts))

# ------------------------------------------------------------
# 9. Cross-language consistency check
# ------------------------------------------------------------

print("\n[9] Cross-language consistency check")
print("-" * 50)

corpus_by_id = {
    row.get("code_id"): row
    for row in corpus
    if row.get("code_id") is not None
}

qrels_by_query = defaultdict(list)
for row in cross_qrels:
    qrels_by_query[row.get("query_id")].append(row)

# Correct metadata field names
anchor_field = "meta_anchor_code_id"
has_anchor_field = any(
    anchor_field in row
    for row in cross_queries
)

print("meta_anchor_code_id field present:", has_anchor_field)

positive_pairs = 0
different_language_pairs = 0
same_language_pairs = 0
unknown_language_pairs = 0
missing_anchor_codes = 0
anchor_self_matches = 0

if has_anchor_field:
    for query in cross_queries:
        query_id = query.get("query_id")
        anchor_id = query.get(anchor_field)

        if anchor_id is None or anchor_id not in corpus_by_id:
            missing_anchor_codes += 1
            continue

        source_language = normalize_language(
            query.get("anchor_language") or corpus_by_id[anchor_id].get("language")
        )

        for judgement in qrels_by_query.get(query_id, []):
            if to_int(judgement.get("relevance")) != 2:
                continue

            doc_id = judgement.get("doc_id")
            target = corpus_by_id.get(doc_id)
            if target is None:
                continue

            positive_pairs += 1
            target_language = normalize_language(
                target.get("language")
            )

            if doc_id == anchor_id:
                anchor_self_matches += 1

            if source_language is None or target_language is None:
                unknown_language_pairs += 1
            elif source_language == target_language:
                same_language_pairs += 1
            else:
                different_language_pairs += 1

    print("Missing or unmatched anchor codes:", missing_anchor_codes)
    print("Positive pairs checked:", positive_pairs)
    print("Positive pairs with different languages:", different_language_pairs)
    print("Positive pairs with the same language:", same_language_pairs)
    print("Pairs with unknown language:", unknown_language_pairs)
    print("Positive judgments pointing to the query itself:", anchor_self_matches)
else:
    print("The meta_anchor_code_id field was not found; consistency could not be checked.")

# ------------------------------------------------------------
# 10. Show example judgments
# ------------------------------------------------------------

print("\n[10] Example cross-language judgments")
print("-" * 50)

shown = 0
for query in cross_queries:
    query_id = query.get("query_id")
    anchor_id = query.get("meta_anchor_code_id")
    anchor = corpus_by_id.get(anchor_id, {})
    source_language = query.get("anchor_language") or anchor.get("language", "<unknown>")

    for judgement in qrels_by_query.get(query_id, []):
        doc_id = judgement.get("doc_id")
        target = corpus_by_id.get(doc_id, {})

        print({
            "query_id": query_id,
            "anchor_code_id": anchor_id,
            "source_language": source_language,
            "doc_id": doc_id,
            "target_language": target.get("language", "<missing>"),
            "relevance": judgement.get("relevance"),
            "target_problem_id": target.get("meta_source_problem_id"),
            "target_passed": target.get("meta_passed"),
        })
        shown += 1
        if shown >= 10:
            break
    if shown >= 10:
        break

if shown == 0:
    print("No example judgments found.")

# ------------------------------------------------------------
# 11. Final summary
# ------------------------------------------------------------

summary = {
    "repository": REPO,
    "split": SPLIT,
    "revision": REVISION,
    "corpus_rows": len(corpus),
    "unique_corpus_ids": len(corpus_ids),
    "duplicate_corpus_ids": duplicate_count(corpus, "code_id"),
    "query_rows": len(queries),
    "unique_query_ids": len(query_ids),
    "qrels_rows": len(qrels),
    "cross_language_queries": len(cross_queries),
    "cross_language_qrels": len(cross_qrels),
    "cross_language_positive_qrels": cross_label_counts.get("2", 0),
    "cross_language_hard_negative_qrels": cross_label_counts.get("1", 0),
    "qrels_missing_query_refs": len(missing_query_refs),
    "qrels_missing_document_refs": len(missing_doc_refs),
    "positive_pairs_different_languages": different_language_pairs,
    "positive_pairs_same_language": same_language_pairs,
}

print("\n" + "=" * 70)
print("COREB ROW-LEVEL AUDIT — SUMMARY")
print("=" * 70)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nCOREB ROW-LEVEL AUDIT — END")

In [ ]:
# ============================================================
# CoREB ROW-LEVEL AUDIT
# Dataset: hq-bench/coreb
# Split: release_v2603
# ============================================================

%pip -q install datasets==3.1.0 huggingface_hub==0.26.2 pyarrow==21.0.0

from datasets import load_dataset
from collections import Counter, defaultdict
from statistics import median
import json

REPO = "hq-bench/coreb"
SPLIT = "release_v2603"

print("=" * 70)
print("COREB ROW-LEVEL AUDIT — START")
print("=" * 70)

# ------------------------------------------------------------
# 1. Use a fixed dataset revision
# ------------------------------------------------------------

revision = "e3f40f2609b4c52cc6d750594a87b29b370c3e06"

print("\n[1] Using pinned dataset revision:", revision)

print("Dataset:", REPO)
print("Split:", SPLIT)
print("Revision:", revision)

# ------------------------------------------------------------
# 2. Load the three datasets needed for our audit
# ------------------------------------------------------------

print("\n[2] Loading datasets...")

corpus_ds = load_dataset(
    REPO,
    "code_corpus",
    split=SPLIT,
    revision=revision
)

queries_ds = load_dataset(
    REPO,
    "code2code_queries",
    split=SPLIT,
    revision=revision
)

qrels_ds = load_dataset(
    REPO,
    "code2code_qrels",
    split=SPLIT,
    revision=revision
)

corpus = [dict(row) for row in corpus_ds]
queries = [dict(row) for row in queries_ds]
qrels = [dict(row) for row in qrels_ds]

print("\n[3] Dataset sizes")
print("-" * 50)
print("Code corpus rows:", len(corpus))
print("Code-to-code queries:", len(queries))
print("Code-to-code relevance judgments:", len(qrels))


# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def shorten(value, limit=180):
    """Keep output readable by shortening long values."""
    if value is None:
        return "None"

    if isinstance(value, str):
        value = value.replace("\n", "\\n")
    else:
        value = json.dumps(
            value,
            ensure_ascii=False,
            default=str
        )

    if len(value) > limit:
        return value[:limit] + "..."

    return value


def show_schema(name, records):
    print(f"\n{name}")
    print("-" * 50)

    if not records:
        print("Dataset is empty.")
        return

    print("Columns:", list(records[0].keys()))
    print("First row:")

    for key, value in records[0].items():
        print(f"  {key}: {shorten(value)}")


def duplicate_count(records, key):
    values = [
        row.get(key)
        for row in records
        if row.get(key) is not None
    ]
    return len(values) - len(set(values))


def to_int(value):
    try:
        return int(value)
    except (ValueError, TypeError):
        return None


def normalize_language(value):
    if value is None:
        return None
    return str(value).strip().lower()


# ------------------------------------------------------------
# 4. Inspect schemas
# ------------------------------------------------------------

print("\n[4] Inspecting dataset schemas")

show_schema("CODE CORPUS", corpus)
show_schema("CODE-TO-CODE QUERIES", queries)
show_schema("CODE-TO-CODE QRELS", qrels)


# ------------------------------------------------------------
# 5. Check ID integrity and duplicates
# ------------------------------------------------------------

print("\n[5] Identifier integrity")
print("-" * 50)

corpus_ids = {
    row.get("code_id")
    for row in corpus
    if row.get("code_id") is not None
}

query_ids = {
    row.get("query_id")
    for row in queries
    if row.get("query_id") is not None
}

print("Unique code IDs:", len(corpus_ids))
print("Duplicate code IDs:", duplicate_count(corpus, "code_id"))

print("Unique query IDs:", len(query_ids))
print("Duplicate query IDs:", duplicate_count(queries, "query_id"))

qrel_pairs = [
    (row.get("query_id"), row.get("doc_id"))
    for row in qrels
    if row.get("query_id") is not None
    and row.get("doc_id") is not None
]

print(
    "Duplicate query-document pairs:",
    len(qrel_pairs) - len(set(qrel_pairs))
)

missing_query_refs = [
    row for row in qrels
    if row.get("query_id") not in query_ids
]

missing_doc_refs = [
    row for row in qrels
    if row.get("doc_id") not in corpus_ids
]

print("Qrels referencing missing queries:", len(missing_query_refs))
print("Qrels referencing missing documents:", len(missing_doc_refs))


# ------------------------------------------------------------
# 6. Inspect the code corpus
# ------------------------------------------------------------

print("\n[6] Code corpus composition")
print("-" * 50)

language_counts = Counter(
    str(row.get("language", "<missing>"))
    for row in corpus
)

print("Languages:")
for language, count in language_counts.most_common():
    print(f"  {language}: {count}")

for field, label in [
    ("model", "Generator models"),
    ("meta_passed", "Pass-status values"),
    ("meta_source_problem_id", "Source problem IDs"),
]:
    if any(field in row for row in corpus):
        values = Counter(
            str(row.get(field, "<missing>"))
            for row in corpus
        )

        print(f"\n{label}:")

        if field == "meta_source_problem_id":
            unique_values = {
                row.get(field)
                for row in corpus
                if row.get(field) is not None
            }
            print("  Distinct values:", len(unique_values))
        else:
            for value, count in values.most_common():
                print(f"  {value}: {count}")


# ------------------------------------------------------------
# 7. Identify cross-language queries
# ------------------------------------------------------------

print("\n[7] Cross-language query audit")
print("-" * 50)

cross_queries = []

for row in queries:
    query_id = str(row.get("query_id", "")).lower()
    subtask = str(row.get("subtask", "")).lower()

    if "cross_lang" in query_id or "cross_lang" in subtask:
        cross_queries.append(row)

cross_query_ids = {
    row.get("query_id")
    for row in cross_queries
    if row.get("query_id") is not None
}

print("All code-to-code queries:", len(queries))
print("Cross-language queries:", len(cross_queries))
print("Other code-to-code queries:", len(queries) - len(cross_queries))

print("\nCross-language query examples:")

for row in cross_queries[:3]:
    print({
        key: shorten(value, 120)
        for key, value in row.items()
    })


# ------------------------------------------------------------
# 8. Analyze relevance labels
#
# CoREB:
#   relevance=2 -> positive result
#   relevance=1 -> hard negative
#   absent from qrels -> unjudged corpus candidate
# ------------------------------------------------------------

print("\n[8] Relevance label audit")
print("-" * 50)

def label_counts(records):
    return Counter(
        str(to_int(row.get("relevance")))
        if to_int(row.get("relevance")) is not None
        else "<missing>"
        for row in records
    )


print("All code-to-code qrels:")

for label, count in sorted(label_counts(qrels).items()):
    print(f"  relevance={label}: {count}")

cross_qrels = [
    row for row in qrels
    if row.get("query_id") in cross_query_ids
]

print("\nCross-language qrels:", len(cross_qrels))
print("Cross-language relevance labels:")

cross_label_counts = label_counts(cross_qrels)

for label, count in sorted(cross_label_counts.items()):
    print(f"  relevance={label}: {count}")

cross_qrel_query_ids = {
    row.get("query_id")
    for row in cross_qrels
    if row.get("query_id") is not None
}

print(
    "Cross-language queries with judgments:",
    len(cross_qrel_query_ids)
)

print(
    "Cross-language queries without judgments:",
    len(cross_query_ids - cross_qrel_query_ids)
)


# ------------------------------------------------------------
# 9. Count positives and hard negatives per query
# ------------------------------------------------------------

print("\n[9] Positives and hard negatives per query")
print("-" * 50)

relevance_by_query = defaultdict(Counter)

for row in cross_qrels:
    query_id = row.get("query_id")
    relevance = to_int(row.get("relevance"))

    if query_id is not None and relevance is not None:
        relevance_by_query[query_id][relevance] += 1

positive_counts = [
    relevance_by_query[qid].get(2, 0)
    for qid in cross_query_ids
]

hard_negative_counts = [
    relevance_by_query[qid].get(1, 0)
    for qid in cross_query_ids
]

if positive_counts:
    print("Positive documents (relevance=2):")
    print("  Total:", sum(positive_counts))
    print("  Queries with zero positives:", sum(x == 0 for x in positive_counts))
    print("  Minimum per query:", min(positive_counts))
    print("  Median per query:", median(positive_counts))
    print("  Maximum per query:", max(positive_counts))

if hard_negative_counts:
    print("\nHard negatives (relevance=1):")
    print("  Total:", sum(hard_negative_counts))
    print(
        "  Queries with zero hard negatives:",
        sum(x == 0 for x in hard_negative_counts)
    )
    print("  Minimum per query:", min(hard_negative_counts))
    print("  Median per query:", median(hard_negative_counts))
    print("  Maximum per query:", max(hard_negative_counts))


# ------------------------------------------------------------
# 10. Check whether positive documents use another language
# ------------------------------------------------------------

print("\n[10] Cross-language consistency check")
print("-" * 50)

corpus_by_id = {
    row.get("code_id"): row
    for row in corpus
    if row.get("code_id") is not None
}

qrels_by_query = defaultdict(list)

for row in cross_qrels:
    qrels_by_query[row.get("query_id")].append(row)

anchor_field = "anchor_code_id"

has_anchor_field = any(
    anchor_field in row
    for row in cross_queries
)

print("anchor_code_id field present:", has_anchor_field)

positive_pairs = 0
different_language_pairs = 0
same_language_pairs = 0
unknown_language_pairs = 0
missing_anchor_codes = 0
anchor_self_matches = 0

if has_anchor_field:
    for query in cross_queries:
        query_id = query.get("query_id")
        anchor_id = query.get(anchor_field)

        if anchor_id is None or anchor_id not in corpus_by_id:
            missing_anchor_codes += 1
            continue

        source_language = normalize_language(
            corpus_by_id[anchor_id].get("language")
        )

        for judgement in qrels_by_query.get(query_id, []):
            if to_int(judgement.get("relevance")) != 2:
                continue

            doc_id = judgement.get("doc_id")
            target = corpus_by_id.get(doc_id)

            if target is None:
                continue

            positive_pairs += 1
            target_language = normalize_language(
                target.get("language")
            )

            if doc_id == anchor_id:
                anchor_self_matches += 1

            if source_language is None or target_language is None:
                unknown_language_pairs += 1
            elif source_language == target_language:
                same_language_pairs += 1
            else:
                different_language_pairs += 1

    print("Missing or unmatched anchor codes:", missing_anchor_codes)
    print("Positive pairs checked:", positive_pairs)
    print("Positive pairs with different languages:", different_language_pairs)
    print("Positive pairs with the same language:", same_language_pairs)
    print("Pairs with unknown language:", unknown_language_pairs)
    print("Positive judgments pointing to the query itself:", anchor_self_matches)

else:
    print(
        "The anchor field was not found; "
        "language consistency could not be checked."
    )


# ------------------------------------------------------------
# 11. Show example judgments
# ------------------------------------------------------------

print("\n[11] Example cross-language judgments")
print("-" * 50)

shown = 0

for query in cross_queries:
    query_id = query.get("query_id")
    anchor_id = query.get("anchor_code_id")

    anchor = corpus_by_id.get(anchor_id, {})
    source_language = anchor.get("language", "<unknown>")

    for judgement in qrels_by_query.get(query_id, []):
        doc_id = judgement.get("doc_id")
        target = corpus_by_id.get(doc_id, {})

        print({
            "query_id": query_id,
            "anchor_code_id": anchor_id,
            "source_language": source_language,
            "doc_id": doc_id,
            "target_language": target.get("language", "<missing>"),
            "relevance": judgement.get("relevance"),
            "target_problem_id": target.get("meta_source_problem_id"),
            "target_passed": target.get("meta_passed"),
        })

        shown += 1

        if shown >= 10:
            break

    if shown >= 10:
        break

if shown == 0:
    print("No example judgments found.")


# ------------------------------------------------------------
# 12. Final summary
# ------------------------------------------------------------

summary = {
    "repository": REPO,
    "split": SPLIT,
    "revision": revision,
    "corpus_rows": len(corpus),
    "unique_corpus_ids": len(corpus_ids),
    "duplicate_corpus_ids": duplicate_count(corpus, "code_id"),
    "query_rows": len(queries),
    "unique_query_ids": len(query_ids),
    "qrels_rows": len(qrels),
    "cross_language_queries": len(cross_queries),
    "cross_language_qrels": len(cross_qrels),
    "cross_language_positive_qrels": cross_label_counts.get("2", 0),
    "cross_language_hard_negative_qrels": cross_label_counts.get("1", 0),
    "qrels_missing_query_refs": len(missing_query_refs),
    "qrels_missing_document_refs": len(missing_doc_refs),
    "positive_pairs_different_languages": different_language_pairs,
    "positive_pairs_same_language": same_language_pairs,
}

print("\n" + "=" * 70)
print("COREB ROW-LEVEL AUDIT — SUMMARY")
print("=" * 70)

print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\nCOREB ROW-LEVEL AUDIT — END")

In [ ]:
from collections import Counter
import json

# Build lookup by code ID.
corpus_by_id = {
    row["code_id"]: row
    for row in corpus
}

# Select cross-language queries using the actual dataset schema.
cross_queries = [
    q for q in queries
    if q.get("subtask") == "c2c_cross_lang"
]

cross_query_ids = {q["query_id"] for q in cross_queries}

cross_qrels = [
    r for r in qrels
    if r.get("query_id") in cross_query_ids
]

queries_by_id = {q["query_id"]: q for q in cross_queries}

# Count checks and mismatches.
stats = Counter()
problems = []

for q in cross_queries:
    qid = q["query_id"]

    # Correct metadata field:
    anchor_id = q.get("meta_anchor_code_id")
    anchor = corpus_by_id.get(anchor_id)

    if anchor is None:
        stats["missing_anchor"] += 1
        problems.append((qid, "Anchor code not found", anchor_id))
        continue

    stats["anchors_found"] += 1

    source_language = str(
        q.get("anchor_language") or anchor.get("language") or ""
    ).strip().lower()

    anchor_language = str(anchor.get("language") or "").strip().lower()

    if source_language != anchor_language:
        stats["anchor_language_mismatch"] += 1
        problems.append((
            qid, "Query/anchor language mismatch",
            source_language, anchor_language
        ))

    source_problem = q.get("meta_source_problem_id")

    if source_problem != anchor.get("meta_source_problem_id"):
        stats["anchor_problem_mismatch"] += 1
        problems.append((qid, "Query/anchor problem mismatch"))

    # Target languages are stored as a JSON string in this dataset.
    raw_targets = q.get("meta_target_languages")

    try:
        if isinstance(raw_targets, str):
            target_languages = json.loads(raw_targets)
        elif isinstance(raw_targets, list):
            target_languages = raw_targets
        else:
            target_languages = []
    except (json.JSONDecodeError, TypeError):
        target_languages = []

    target_languages = {
        str(lang).strip().lower()
        for lang in target_languages
    }

    stats["queries_checked"] += 1

    # Inspect all judged results for this query.
    for r in cross_qrels:
        if r.get("query_id") != qid:
            continue

        doc_id = r.get("doc_id")
        doc = corpus_by_id.get(doc_id)

        if doc is None:
            continue

        relevance = int(r["relevance"])
        target_language = str(
            doc.get("language") or ""
        ).strip().lower()

        if doc_id == anchor_id:
            stats["anchor_appears_in_qrels"] += 1

        if doc.get("meta_source_problem_id") != source_problem:
            stats["judged_doc_problem_mismatch"] += 1

        if relevance == 2:
            stats["positive_pairs"] += 1

            if target_language == source_language:
                stats["positive_same_language"] += 1

            if (
                target_languages
                and target_language not in target_languages
            ):
                stats["positive_not_in_target_languages"] += 1

            if doc.get("meta_passed") is True:
                stats["positive_passed"] += 1
            elif doc.get("meta_passed") is False:
                stats["positive_failed"] += 1
            else:
                stats["positive_pass_status_unknown"] += 1

        elif relevance == 1:
            stats["hard_negative_pairs"] += 1

            if doc.get("meta_passed") is True:
                stats["hard_negatives_that_passed"] += 1

print("=" * 60)
print("COREB CROSS-LANGUAGE VALIDATION")
print("=" * 60)

print("Cross-language queries:", len(cross_queries))
print("Cross-language qrels:", len(cross_qrels))

for key, value in sorted(stats.items()):
    print(f"{key}: {value}")

print("\nFirst validation issues, if any:")
if problems:
    for problem in problems[:10]:
        print(problem)
else:
    print("No anchor or source-problem mismatches detected.")

print("\nVALIDATION COMPLETE")

COREB CROSS-LANGUAGE VALIDATION
Cross-language queries: 190
Cross-language qrels: 999
anchors_found: 190
hard_negative_pairs: 570
hard_negatives_that_passed: 391
positive_pairs: 429
positive_passed: 429
queries_checked: 190

First validation issues, if any:
No anchor or source-problem mismatches detected.

VALIDATION COMPLETE


In [ ]:
# ============================================================
# CoREB: TF-IDF CROSS-LANGUAGE CODE RETRIEVAL BASELINE
# Evaluation: relevance=2 counts as relevant
# ============================================================

import numpy as np
import math
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer

print("=" * 65)
print("COREB TF-IDF BASELINE — START")
print("=" * 65)

# 1. Select cross-language queries and prepare corpus
cross_queries = [
    q for q in queries
    if q.get("subtask") == "c2c_cross_lang"
]

code_ids = [row["code_id"] for row in corpus]
code_id_to_index = {
    code_id: i for i, code_id in enumerate(code_ids)
}

documents = [
    str(row.get("code") or "")
    for row in corpus
]

query_texts = [
    str(q.get("query") or "")
    for q in cross_queries
]

print("Corpus documents:", len(documents))
print("Cross-language queries:", len(cross_queries))

# 2. Build a lexical code representation.
# Tokens include identifiers, numbers, operators, and punctuation.
vectorizer = TfidfVectorizer(
    token_pattern=(
        r"[A-Za-z_]\w*|\d+(?:\.\d+)?|==|!=|<=|>=|&&|\|\|"
        r"|\+\+|--|->|[^\s]"
    ),
    ngram_range=(1, 2),
    lowercase=False,
    sublinear_tf=True,
    dtype=np.float32
)

X_docs = vectorizer.fit_transform(documents)
X_queries = vectorizer.transform(query_texts)

# L2-normalized TF-IDF vectors mean dot product = cosine similarity.
similarities = (X_queries @ X_docs.T).toarray()

print("TF-IDF vocabulary size:", len(vectorizer.vocabulary_))

# 3. Build relevance judgments for cross-language queries.
# Only relevance=2 is treated as relevant.
qrels_by_query = defaultdict(dict)

for row in qrels:
    qid = row.get("query_id")
    if qid is not None:
        qrels_by_query[qid][row["doc_id"]] = int(row["relevance"])

# 4. Evaluate the ranked candidates.
# We exclude each query's anchor implementation from its candidate ranking.
# The full remaining code corpus is used as the candidate pool.

K_VALUES = (1, 5, 10)
per_query_results = []
missing_anchor = 0
queries_without_positive = 0

for i, q in enumerate(cross_queries):
    qid = q["query_id"]
    anchor_id = q.get("meta_anchor_code_id")

    if anchor_id in code_id_to_index:
        similarities[i, code_id_to_index[anchor_id]] = -np.inf
    else:
        missing_anchor += 1

    ranked_indices = np.argsort(-similarities[i], kind="stable")
    ranked_ids = [code_ids[j] for j in ranked_indices]

    judgments = qrels_by_query.get(qid, {})

    relevant_ids = {
        doc_id
        for doc_id, relevance in judgments.items()
        if relevance >= 2
    }

    if not relevant_ids:
        queries_without_positive += 1
        continue

    # Rank position of every judged positive, if retrieved.
    positive_ranks = [
        rank
        for rank, doc_id in enumerate(ranked_ids, start=1)
        if doc_id in relevant_ids
    ]

    first_rank = min(positive_ranks) if positive_ranks else None
    row = {
        "query_id": qid,
        "source_language": q.get("anchor_language", "<unknown>"),
        "num_positives": len(relevant_ids),
        "MRR": 1.0 / first_rank if first_rank is not None else 0.0,
    }

    for k in K_VALUES:
        found = sum(rank <= k for rank in positive_ranks)

        row[f"Hit@{k}"] = float(found > 0)
        row[f"Recall@{k}"] = found / len(relevant_ids)

    # Binary nDCG@10: only relevance=2 receives gain.
    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank in positive_ranks
        if rank <= 10
    )

    ideal_count = min(len(relevant_ids), 10)
    idcg = sum(
        1.0 / math.log2(rank + 1)
        for rank in range(1, ideal_count + 1)
    )

    row["nDCG@10"] = dcg / idcg if idcg else 0.0

    per_query_results.append(row)

# 5. Summarize metrics.
metric_names = [
    "Hit@1", "Hit@5", "Hit@10",
    "Recall@1", "Recall@5", "Recall@10",
    "MRR", "nDCG@10"
]

print("\n" + "=" * 65)
print("OVERALL RESULTS")
print("=" * 65)

print("Queries evaluated:", len(per_query_results))
print("Queries without positive qrels:", queries_without_positive)
print("Queries without a matching anchor ID:", missing_anchor)

for metric in metric_names:
    values = [row[metric] for row in per_query_results]
    mean_value = float(np.mean(values)) if values else float("nan")
    print(f"{metric:12s}: {mean_value:.4f}")

# 6. Results by source language.
print("\n" + "=" * 65)
print("RESULTS BY SOURCE LANGUAGE")
print("=" * 65)

languages = sorted({
    row["source_language"]
    for row in per_query_results
})

for language in languages:
    subset = [
        row for row in per_query_results
        if row["source_language"] == language
    ]

    print(f"\nLanguage: {language} | Queries: {len(subset)}")

    for metric in metric_names:
        value = float(np.mean([row[metric] for row in subset]))
        print(f"  {metric:12s}: {value:.4f}")

# 7. Save per-query results for reproducibility.
import pandas as pd

results_df = pd.DataFrame(per_query_results)
results_df.to_csv(
    "/content/coreb_tfidf_per_query_results.csv",
    index=False
)

print("\nSaved per-query results:")
print("/content/coreb_tfidf_per_query_results.csv")

print("\nCOREB TF-IDF BASELINE — END")

COREB TF-IDF BASELINE — START
Corpus documents: 1744
Cross-language queries: 190
TF-IDF vocabulary size: 62313

OVERALL RESULTS
Queries evaluated: 190
Queries without positive qrels: 0
Queries without a matching anchor ID: 0
Hit@1       : 0.1842
Hit@5       : 0.6105
Hit@10      : 0.6684
Recall@1    : 0.0855
Recall@5    : 0.4013
Recall@10   : 0.5013
MRR         : 0.3574
nDCG@10     : 0.3444

RESULTS BY SOURCE LANGUAGE

Language: cpp | Queries: 50
  Hit@1       : 0.3600
  Hit@5       : 0.9400
  Hit@10      : 0.9400
  Recall@1    : 0.0983
  Recall@5    : 0.4550
  Recall@10   : 0.5850
  MRR         : 0.5983
  nDCG@10     : 0.4813

Language: go | Queries: 53
  Hit@1       : 0.1698
  Hit@5       : 0.5472
  Hit@10      : 0.6226
  Recall@1    : 0.0723
  Recall@5    : 0.2925
  Recall@10   : 0.4057
  MRR         : 0.3126
  nDCG@10     : 0.2852

Language: java | Queries: 41
  Hit@1       : 0.0244
  Hit@5       : 0.1220
  Hit@10      : 0.1951
  Recall@1    : 0.0122
  Recall@5    : 0.0732
  Recall@

In [ ]:
# ============================================================
# CoREB: MAX vs MEAN AGGREGATION
# Group candidates by (problem, language)
# Compare k=1, k=2, and all available implementations
# ============================================================

import numpy as np
import pandas as pd
import math
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import TfidfVectorizer

print("=" * 65)
print("COREB GROUPED AGGREGATION EXPERIMENT — START")
print("=" * 65)

# ------------------------------------------------------------
# 1. Prepare the cross-language queries and code corpus
# ------------------------------------------------------------

cross_queries = [
    q for q in queries
    if q.get("subtask") == "c2c_cross_lang"
]

code_ids = [row["code_id"] for row in corpus]
documents = [str(row.get("code") or "") for row in corpus]
query_texts = [str(q.get("query") or "") for q in cross_queries]

id_to_doc_index = {
    code_id: i for i, code_id in enumerate(code_ids)
}

corpus_by_id = {
    row["code_id"]: row for row in corpus
}

# Recompute similarities independently of the earlier baseline cell.
vectorizer = TfidfVectorizer(
    token_pattern=(
        r"[A-Za-z_]\w*|\d+(?:\.\d+)?|==|!=|<=|>=|&&|\|\|"
        r"|\+\+|--|->|[^\s]"
    ),
    ngram_range=(1, 2),
    lowercase=False,
    sublinear_tf=True,
    dtype=np.float32
)

X_docs = vectorizer.fit_transform(documents)
X_queries = vectorizer.transform(query_texts)

# Query-by-document cosine similarities.
sim = (X_queries @ X_docs.T).toarray().astype(np.float32)

print("Cross-language queries:", len(cross_queries))
print("Corpus documents:", len(corpus))


# ------------------------------------------------------------
# 2. Group implementations by source problem and language
# ------------------------------------------------------------

group_to_doc_indices = defaultdict(list)

for i, row in enumerate(corpus):
    problem_id = row.get("meta_source_problem_id")
    language = row.get("language")

    if problem_id is None or language is None:
        continue

    group_key = (problem_id, language)
    group_to_doc_indices[group_key].append(i)

group_keys = list(group_to_doc_indices.keys())
group_members = [
    group_to_doc_indices[key] for key in group_keys
]

group_key_to_index = {
    key: i for i, key in enumerate(group_keys)
}

doc_id_to_group_index = {}

for group_index, indices in enumerate(group_members):
    for doc_index in indices:
        doc_id_to_group_index[code_ids[doc_index]] = group_index

group_size_counts = Counter(
    len(indices) for indices in group_members
)

print("\nCandidate groups:", len(group_keys))
print("Implementations per problem-language group:")

for size, count in sorted(group_size_counts.items()):
    print(f"  {size} implementation(s): {count} groups")


# ------------------------------------------------------------
# 3. Convert document-level judgments to group-level judgments
#
# Group relevance = maximum relevance label among judged members.
# Therefore, a group is positive if ANY member has relevance=2.
# ------------------------------------------------------------

cross_query_ids = {q["query_id"] for q in cross_queries}

group_qrels = defaultdict(dict)

for judgment in qrels:
    qid = judgment.get("query_id")
    doc_id = judgment.get("doc_id")

    if qid not in cross_query_ids:
        continue

    group_index = doc_id_to_group_index.get(doc_id)

    if group_index is None:
        continue

    relevance = int(judgment["relevance"])
    previous = group_qrels[qid].get(group_index, 0)

    group_qrels[qid][group_index] = max(previous, relevance)

queries_without_group_positive = 0

for q in cross_queries:
    labels = group_qrels.get(q["query_id"], {})
    if not any(rel >= 2 for rel in labels.values()):
        queries_without_group_positive += 1

print("\nQueries without a positive group:", queries_without_group_positive)


# ------------------------------------------------------------
# 4. Calculate grouped scores for each query
# ------------------------------------------------------------

def score_groups(k, seed):
    """
    k=1: randomly sample one implementation per group.
    k=2: randomly sample up to two implementations per group.
    k=None: use all implementations in each group.

    The query's anchor code is excluded from its candidate pool.
    The same sampled implementations are used for max and mean.
    """

    max_scores = np.full(
        (len(cross_queries), len(group_keys)),
        -np.inf,
        dtype=np.float32
    )

    mean_scores = np.full_like(max_scores, -np.inf)

    for qi, query in enumerate(cross_queries):
        anchor_id = query.get("meta_anchor_code_id")

        rng = np.random.default_rng(seed + (qi + 1) * 10007)

        for gi, members in enumerate(group_members):
            available = [
                doc_index
                for doc_index in members
                if code_ids[doc_index] != anchor_id
            ]

            if not available:
                continue

            if k is not None and len(available) > k:
                selected = rng.choice(
                    available,
                    size=k,
                    replace=False
                )
            else:
                selected = available

            selected_scores = sim[qi, selected]

            max_scores[qi, gi] = float(np.max(selected_scores))
            mean_scores[qi, gi] = float(np.mean(selected_scores))

    return max_scores, mean_scores


# ------------------------------------------------------------
# 5. Evaluate grouped retrieval
# ------------------------------------------------------------

def evaluate_grouped(scores):
    metrics = defaultdict(list)

    for qi, query in enumerate(cross_queries):
        qid = query["query_id"]
        labels = group_qrels.get(qid, {})

        relevant = {
            gi for gi, rel in labels.items()
            if rel >= 2 and np.isfinite(scores[qi, gi])
        }

        if not relevant:
            continue

        ranked = np.argsort(-scores[qi], kind="stable")
        ranked = [
            gi for gi in ranked
            if np.isfinite(scores[qi, gi])
        ]

        positive_ranks = [
            rank
            for rank, gi in enumerate(ranked, start=1)
            if gi in relevant
        ]

        first_rank = min(positive_ranks) if positive_ranks else None

        for k_value in (1, 5, 10):
            found = sum(rank <= k_value for rank in positive_ranks)

            metrics[f"Hit@{k_value}"].append(
                float(found > 0)
            )

            metrics[f"Recall@{k_value}"].append(
                found / len(relevant)
            )

        metrics["MRR"].append(
            1.0 / first_rank if first_rank is not None else 0.0
        )

        dcg = sum(
            1.0 / math.log2(rank + 1)
            for rank in positive_ranks
            if rank <= 10
        )

        ideal_count = min(len(relevant), 10)

        idcg = sum(
            1.0 / math.log2(rank + 1)
            for rank in range(1, ideal_count + 1)
        )

        metrics["nDCG@10"].append(
            dcg / idcg if idcg else 0.0
        )

    return {
        metric: float(np.mean(values))
        for metric, values in metrics.items()
    }


# ------------------------------------------------------------
# 6. Run repeated trials for k=1 and k=2.
# "all" uses every available implementation and is deterministic.
# ------------------------------------------------------------

experiment_specs = [
    ("k=1", 1, list(range(10))),
    ("k=2", 2, list(range(10))),
    ("all", None, [0]),
]

experiment_results = []

for setting, k, seeds in experiment_specs:
    print(f"\nRunning setting: {setting}")

    for seed in seeds:
        max_scores, mean_scores = score_groups(k=k, seed=seed)

        for method, score_matrix in [
            ("max", max_scores),
            ("mean", mean_scores)
        ]:
            metrics = evaluate_grouped(score_matrix)

            row = {
                "setting": setting,
                "method": method,
                "seed": seed,
                **metrics
            }

            experiment_results.append(row)

results_df = pd.DataFrame(experiment_results)


# ------------------------------------------------------------
# 7. Summarize results
# ------------------------------------------------------------

metric_names = [
    "Hit@1",
    "Hit@5",
    "Hit@10",
    "Recall@1",
    "Recall@5",
    "Recall@10",
    "MRR",
    "nDCG@10"
]

print("\n" + "=" * 65)
print("GROUPED RETRIEVAL RESULTS")
print("=" * 65)

for setting, _, _ in experiment_specs:
    print(f"\n--- {setting} ---")

    subset = results_df[results_df["setting"] == setting]

    for method in ("max", "mean"):
        method_rows = subset[subset["method"] == method]

        print(f"\nMethod: {method}")

        for metric in metric_names:
            values = method_rows[metric].dropna().to_numpy()

            if len(values) == 0:
                continue

            average = float(np.mean(values))

            if len(values) > 1:
                sd = float(np.std(values, ddof=1))
                print(f"  {metric:12s}: {average:.4f} +/- {sd:.4f}")
            else:
                print(f"  {metric:12s}: {average:.4f}")

# Save detailed trial results.
output_path = "/content/coreb_grouped_aggregation_results.csv"

results_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)
print("\nCOREB GROUPED AGGREGATION EXPERIMENT — END")

COREB GROUPED AGGREGATION EXPERIMENT — START
Cross-language queries: 190
Corpus documents: 1744

Candidate groups: 875
Implementations per problem-language group:
  1 implementation(s): 6 groups
  2 implementation(s): 869 groups

Queries without a positive group: 0

Running setting: k=1

Running setting: k=2

Running setting: all

GROUPED RETRIEVAL RESULTS

--- k=1 ---

Method: max
  Hit@1       : 0.1221 +/- 0.0201
  Hit@5       : 0.5247 +/- 0.0225
  Hit@10      : 0.6100 +/- 0.0207
  Recall@1    : 0.0575 +/- 0.0109
  Recall@5    : 0.3409 +/- 0.0158
  Recall@10   : 0.4309 +/- 0.0143
  MRR         : 0.3041 +/- 0.0163
  nDCG@10     : 0.2890 +/- 0.0106

Method: mean
  Hit@1       : 0.1221 +/- 0.0201
  Hit@5       : 0.5247 +/- 0.0225
  Hit@10      : 0.6100 +/- 0.0207
  Recall@1    : 0.0575 +/- 0.0109
  Recall@5    : 0.3409 +/- 0.0158
  Recall@10   : 0.4309 +/- 0.0143
  MRR         : 0.3041 +/- 0.0163
  nDCG@10     : 0.2890 +/- 0.0106

--- k=2 ---

Method: max
  Hit@1       : 0.1842 +/- 0.00

In [ ]:
import pandas as pd
import ast
import re

# Приводим загруженные данные к DataFrame
corpus_df = pd.DataFrame(corpus)
queries_df = pd.DataFrame(queries)
qrels_df = pd.DataFrame(qrels)

# Оставляем только cross-language code-to-code запросы
cross_q = queries_df[
    queries_df["subtask"].eq("c2c_cross_lang")
].copy()

cross_qrels = qrels_df[
    qrels_df["query_id"].isin(cross_q["query_id"])
].copy()

# Добавляем язык и ID задачи каждому документу из qrels
doc_meta = corpus_df[
    ["code_id", "language", "meta_source_problem_id"]
].copy()

judged = cross_qrels.merge(
    doc_meta,
    left_on="doc_id",
    right_on="code_id",
    how="left",
    validate="many_to_one"
)

print("=" * 65)
print("1. POSITIVE DOCUMENTS VS EXPECTED TARGET LANGUAGES")
print("=" * 65)

def normalize_languages(value):
    if isinstance(value, (list, tuple, set)):
        values = value
    else:
        s = str(value).strip()
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, set)):
                values = parsed
            else:
                values = re.split(r"[,;]", s)
        except (ValueError, SyntaxError):
            values = re.split(r"[,;]", s)

    return {
        str(v).strip().lower()
        for v in values
        if str(v).strip()
    }

query_meta = cross_q[
    ["query_id", "anchor_language", "meta_target_languages"]
].copy()

positives = judged[judged["relevance"].eq(2)].merge(
    query_meta,
    on="query_id",
    how="left"
)

positives["target_language_set"] = (
    positives["meta_target_languages"].apply(normalize_languages)
)

positives["language_matches_target"] = positives.apply(
    lambda row: str(row["language"]).lower()
    in row["target_language_set"],
    axis=1
)

print("Positive document judgments:", len(positives))
print(
    "Positives matching target language:",
    int(positives["language_matches_target"].sum())
)
print(
    "Positives outside target language:",
    int((~positives["language_matches_target"]).sum())
)

print("\nExample target-language metadata:")
print(
    cross_q[
        ["anchor_language", "meta_target_languages"]
    ].head(8).to_string(index=False)
)

print("\n" + "=" * 65)
print("2. GROUPS CONTAINING BOTH POSITIVES AND HARD NEGATIVES")
print("=" * 65)

group_labels = (
    judged.groupby(
        ["query_id", "meta_source_problem_id", "language"]
    )["relevance"]
    .agg(
        min_rel="min",
        max_rel="max",
        distinct_labels="nunique",
        judged_docs="size"
    )
    .reset_index()
)

mixed_groups = group_labels[
    group_labels["distinct_labels"] > 1
]

print("Judged query/problem/language groups:", len(group_labels))
print("Groups containing multiple relevance labels:", len(mixed_groups))

if len(mixed_groups):
    print("\nExamples:")
    print(mixed_groups.head(15).to_string(index=False))

print("\n" + "=" * 65)
print("3. SUMMARY")
print("=" * 65)
print("Cross-language queries:", len(cross_q))
print("Cross-language qrels:", len(judged))
print(
    "Queries with at least one positive judgment:",
    judged.loc[judged["relevance"].eq(2), "query_id"].nunique()
)

1. POSITIVE DOCUMENTS VS EXPECTED TARGET LANGUAGES
Positive document judgments: 429
Positives matching target language: 429
Positives outside target language: 0

Example target-language metadata:
anchor_language meta_target_languages
         python                [ruby]
            cpp      [go, java, ruby]
             go          [java, ruby]
           java                [ruby]
         python                [ruby]
         python                [ruby]
            cpp            [go, java]
             go                [java]

2. GROUPS CONTAINING BOTH POSITIVES AND HARD NEGATIVES
Judged query/problem/language groups: 735
Groups containing multiple relevance labels: 220

Examples:
             query_id meta_source_problem_id language  min_rel  max_rel  distinct_labels  judged_docs
q_c2c_cross_lang_0047               lcb_3704     ruby        1        2                2            2
q_c2c_cross_lang_0048               lcb_3709     java        1        2                2            

In [ ]:
import ast
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

# ============================================================
# 1. PREPARE DATA
# ============================================================

corpus_df = pd.DataFrame(corpus).copy().reset_index(drop=True)
queries_df = pd.DataFrame(queries).copy()
qrels_df = pd.DataFrame(qrels).copy()

required_corpus = {
    "code_id", "code", "language", "meta_source_problem_id"
}
required_queries = {
    "query_id", "meta_anchor_code_id",
    "anchor_language", "meta_target_languages"
}
required_qrels = {"query_id", "doc_id", "relevance"}

assert required_corpus.issubset(corpus_df.columns), (
    f"Missing corpus columns: {required_corpus - set(corpus_df.columns)}"
)
assert required_queries.issubset(queries_df.columns), (
    f"Missing query columns: {required_queries - set(queries_df.columns)}"
)
assert required_qrels.issubset(qrels_df.columns), (
    f"Missing qrels columns: {required_qrels - set(qrels_df.columns)}"
)

# Standardize IDs to avoid int/string mismatches.
for df, columns in [
    (corpus_df, ["code_id", "meta_source_problem_id", "language"]),
    (queries_df, ["query_id", "meta_anchor_code_id"]),
    (qrels_df, ["query_id", "doc_id"]),
]:
    for col in columns:
        df[col] = df[col].astype("string")

corpus_df["language"] = corpus_df["language"].str.lower()
corpus_df["code"] = corpus_df["code"].fillna("").astype(str)

if corpus_df["code_id"].duplicated().any():
    raise ValueError("Duplicate code_id values found in corpus.")

queries_df = queries_df[
    queries_df["subtask"].eq("c2c_cross_lang")
].copy()

queries_df["query_id"] = queries_df["query_id"].astype(str)
queries_df["meta_anchor_code_id"] = (
    queries_df["meta_anchor_code_id"].astype(str)
)

qrels_df["query_id"] = qrels_df["query_id"].astype(str)
qrels_df["doc_id"] = qrels_df["doc_id"].astype(str)
qrels_df["relevance"] = pd.to_numeric(
    qrels_df["relevance"], errors="coerce"
).fillna(0)

cross_query_ids = set(queries_df["query_id"])
qrels_df = qrels_df[
    qrels_df["query_id"].isin(cross_query_ids)
].copy()

print("=" * 65)
print("CORRECTED CoREB EXPERIMENT")
print("=" * 65)
print("Cross-language queries:", len(queries_df))
print("Corpus documents:", len(corpus_df))
print("Judgments:", len(qrels_df))

# ============================================================
# 2. PARSE TARGET LANGUAGES
# ============================================================

def parse_languages(value):
    if isinstance(value, (list, tuple, set, np.ndarray)):
        values = list(value)
    else:
        s = str(value).strip()

        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, set)):
                values = list(parsed)
            else:
                values = [parsed]
        except (ValueError, SyntaxError):
            s = s.strip("[](){}")
            values = re.split(r"[,;]", s)

    return {
        str(v).strip().lower().strip("'\"")
        for v in values
        if str(v).strip()
    }

queries_df["target_languages"] = (
    queries_df["meta_target_languages"].apply(parse_languages)
)

# ============================================================
# 3. FIT TF-IDF
# ============================================================

vectorizer = TfidfVectorizer(
    token_pattern=(
        r"[A-Za-z_]\w*|\d+(?:\.\d+)?|==|!=|<=|>=|&&|\|\||"
        r"\+\+|--|->|[^\s]"
    ),
    ngram_range=(1, 2),
    lowercase=False,
    sublinear_tf=True,
)

X = vectorizer.fit_transform(corpus_df["code"])

doc_ids = corpus_df["code_id"].astype(str).tolist()
doc_id_to_idx = {doc_id: i for i, doc_id in enumerate(doc_ids)}

doc_languages = corpus_df["language"].astype(str).tolist()
doc_problems = corpus_df["meta_source_problem_id"].astype(str).tolist()

# ============================================================
# 4. GROUP CANDIDATES BY PROBLEM AND LANGUAGE
# ============================================================

groups = {}

for i, row in corpus_df.iterrows():
    problem = row["meta_source_problem_id"]
    language = row["language"]

    if pd.isna(problem) or pd.isna(language):
        continue

    key = (str(problem), str(language).lower())
    groups.setdefault(key, []).append(i)

# Keep a query-specific relevance map.
qrel_maps = {}

for qid, sub in qrels_df.groupby("query_id"):
    qrel_maps[str(qid)] = dict(
        zip(
            sub["doc_id"].astype(str),
            sub["relevance"].astype(float)
        )
    )

print("Problem-language groups in corpus:", len(groups))

# ============================================================
# 5. RETRIEVAL METRICS
# ============================================================

def calculate_metrics(ranked_relevances):
    rel = np.asarray(ranked_relevances, dtype=float)
    total_relevant = int(rel.sum())

    result = {}

    for k in (1, 5, 10):
        top = rel[:k]
        result[f"Hit@{k}"] = float(top.sum() > 0)
        result[f"Recall@{k}"] = (
            float(top.sum() / total_relevant)
            if total_relevant > 0 else 0.0
        )

    positions = np.flatnonzero(rel > 0)
    result["MRR"] = (
        float(1.0 / (positions[0] + 1))
        if len(positions) else 0.0
    )

    top10 = rel[:10]
    discounts = 1.0 / np.log2(np.arange(2, len(top10) + 2))
    dcg = float(np.sum(top10 * discounts))

    ideal_length = min(total_relevant, 10)
    if ideal_length:
        ideal_discounts = 1.0 / np.log2(
            np.arange(2, ideal_length + 2)
        )
        idcg = float(np.sum(ideal_discounts))
        result["nDCG@10"] = dcg / idcg
    else:
        result["nDCG@10"] = 0.0

    return result

# ============================================================
# 6. RUN THE EXPERIMENT
# ============================================================

N_SEEDS = 10
SEEDS = range(N_SEEDS)

per_query_rows = []
skipped = []

for seed in SEEDS:
    rng = np.random.default_rng(seed)

    for _, query in queries_df.iterrows():
        qid = str(query["query_id"])
        anchor_id = str(query["meta_anchor_code_id"])
        targets = query["target_languages"]

        if anchor_id not in doc_id_to_idx:
            skipped.append((qid, "anchor_missing"))
            continue

        if not targets:
            skipped.append((qid, "target_language_missing"))
            continue

        # Filter candidates to the requested target language(s).
        candidate_groups = []

        for key, member_indices in groups.items():
            problem, language = key

            if language not in targets:
                continue

            # Exclude the query anchor from every candidate group.
            members = [
                i for i in member_indices
                if doc_ids[i] != anchor_id
            ]

            if members:
                candidate_groups.append((key, members))

        candidate_groups.sort(key=lambda item: item[0])

        if not candidate_groups:
            skipped.append((qid, "no_candidate_groups"))
            continue

        # Similarities between the query anchor and all corpus documents.
        q_idx = doc_id_to_idx[anchor_id]
        query_similarities = (
            X @ X.getrow(q_idx).T
        ).toarray().ravel()

        # Randomize each group's member order once.
        # Thus k=1 is nested inside k=2, and the same sample is used
        # for max and mean within a given setting.
        sampled_groups = {}

        for key, members in candidate_groups:
            shuffled = rng.permutation(members).tolist()
            sampled_groups[key] = {
                "k=1": shuffled[:1],
                "k=2": shuffled[:2],
                "all": members[:],
            }

        settings_to_run = ["k=1", "k=2"]

        # With 1–2 implementations per group, "all" equals "k=2"
        # here, but we record it explicitly for clarity.
        if seed == 0:
            settings_to_run.append("all")

        query_qrels = qrel_maps.get(qid, {})

        for setting in settings_to_run:
            for method in ("max", "mean"):
                scored_groups = []

                for key, members in candidate_groups:
                    chosen = sampled_groups[key][setting]

                    member_scores = query_similarities[chosen]

                    if method == "max":
                        group_score = float(np.max(member_scores))
                    else:
                        group_score = float(np.mean(member_scores))

                    # A selected reference set is relevant if at least
                    # one selected document is judged relevant (rel >= 2).
                    # Unjudged documents are treated as non-relevant.
                    group_relevant = any(
                        query_qrels.get(doc_ids[i], 0) >= 2
                        for i in chosen
                    )

                    scored_groups.append(
                        (key, group_score, int(group_relevant))
                    )

                # Deterministic tie-break by group key.
                ranked = sorted(
                    scored_groups,
                    key=lambda item: (-item[1], item[0])
                )

                ranked_relevances = [item[2] for item in ranked]
                metrics = calculate_metrics(ranked_relevances)

                per_query_rows.append({
                    "query_id": qid,
                    "seed": seed,
                    "setting": setting,
                    "method": method,
                    "n_candidate_groups": len(ranked),
                    "n_relevant_groups": int(sum(ranked_relevances)),
                    **metrics,
                })

    print(f"Completed seed {seed + 1}/{N_SEEDS}")

# ============================================================
# 7. SAVE PER-QUERY RESULTS
# ============================================================

per_query_df = pd.DataFrame(per_query_rows)

if per_query_df.empty:
    raise RuntimeError("No results were produced. Check dataset columns.")

per_query_path = "/content/coreb_grouped_corrected_per_query.csv"
per_query_df.to_csv(per_query_path, index=False)

metric_cols = [
    "Hit@1", "Hit@5", "Hit@10",
    "Recall@1", "Recall@5", "Recall@10",
    "MRR", "nDCG@10"
]

# First average over queries for each seed, then summarize the seeds.
run_means = (
    per_query_df.groupby(["setting", "method", "seed"])[metric_cols]
    .mean()
    .reset_index()
)

summary = (
    run_means.groupby(["setting", "method"])[metric_cols]
    .agg(["mean", "std"])
)

print("\n" + "=" * 65)
print("CORRECTED RETRIEVAL RESULTS — MEAN +/- SD ACROSS SEEDS")
print("=" * 65)

for setting in ("k=1", "k=2", "all"):
    if setting not in set(run_means["setting"]):
        continue

    print(f"\n--- {setting} ---")

    for method in ("max", "mean"):
        if (setting, method) not in summary.index:
            continue

        print(f"\nMethod: {method}")

        for metric in metric_cols:
            mean_value = summary.loc[(setting, method), (metric, "mean")]
            sd_value = summary.loc[(setting, method), (metric, "std")]

            if pd.isna(sd_value):
                sd_value = 0.0

            print(f"  {metric:12s}: {mean_value:.4f} +/- {sd_value:.4f}")

# ============================================================
# 8. PAIRED BOOTSTRAP: MAX MINUS MEAN
# ============================================================

# Average each query's metric over its available seeds.
# Then resample queries, keeping max and mean paired.
query_means = (
    per_query_df.groupby(["setting", "query_id", "method"])[metric_cols]
    .mean()
    .reset_index()
)

N_BOOTSTRAP = 5000
bootstrap_rng = np.random.default_rng(20261029)

print("\n" + "=" * 65)
print("PAIRED BOOTSTRAP: MAX - MEAN (95% CI)")
print("=" * 65)

for setting in ("k=1", "k=2", "all"):
    subset = query_means[query_means["setting"] == setting]

    if subset.empty:
        continue

    for metric in metric_cols:
        pivot = subset.pivot(
            index="query_id",
            columns="method",
            values=metric
        ).dropna(subset=["max", "mean"])

        if len(pivot) < 2:
            continue

        differences = (
            pivot["max"].to_numpy() - pivot["mean"].to_numpy()
        )
        n = len(differences)

        boot_means = np.empty(N_BOOTSTRAP)

        for b in range(N_BOOTSTRAP):
            indices = bootstrap_rng.integers(0, n, size=n)
            boot_means[b] = differences[indices].mean()

        point_estimate = differences.mean()
        lower, upper = np.quantile(boot_means, [0.025, 0.975])

        print(
            f"{setting:5s} | {metric:10s} | "
            f"max-mean = {point_estimate:+.4f} "
            f"[{lower:+.4f}, {upper:+.4f}]"
        )

print("\nSkipped queries:", len(skipped))
if skipped:
    print("First skipped queries:", skipped[:10])

print("\nSaved per-query results:", per_query_path)
print("\nCORRECTED EXPERIMENT — END")

CORRECTED CoREB EXPERIMENT
Cross-language queries: 190
Corpus documents: 1744
Judgments: 999
Problem-language groups in corpus: 875
Completed seed 1/10
Completed seed 2/10
Completed seed 3/10
Completed seed 4/10
Completed seed 5/10
Completed seed 6/10
Completed seed 7/10
Completed seed 8/10
Completed seed 9/10
Completed seed 10/10

CORRECTED RETRIEVAL RESULTS — MEAN +/- SD ACROSS SEEDS

--- k=1 ---

Method: max
  Hit@1       : 0.5837 +/- 0.0397
  Hit@5       : 0.7032 +/- 0.0476
  Hit@10      : 0.7121 +/- 0.0460
  Recall@1    : 0.4319 +/- 0.0351
  Recall@5    : 0.6679 +/- 0.0446
  Recall@10   : 0.6860 +/- 0.0441
  MRR         : 0.6378 +/- 0.0433
  nDCG@10     : 0.6321 +/- 0.0427

Method: mean
  Hit@1       : 0.5837 +/- 0.0397
  Hit@5       : 0.7032 +/- 0.0476
  Hit@10      : 0.7121 +/- 0.0460
  Recall@1    : 0.4319 +/- 0.0351
  Recall@5    : 0.6679 +/- 0.0446
  Recall@10   : 0.6860 +/- 0.0441
  MRR         : 0.6378 +/- 0.0433
  nDCG@10     : 0.6321 +/- 0.0427

--- k=2 ---

Method: max
 

In [ ]:
import ast
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

# ============================================================
# 1. PREPARE DATA
# ============================================================

corpus_df = pd.DataFrame(corpus).copy().reset_index(drop=True)
queries_df = pd.DataFrame(queries).copy()
qrels_df = pd.DataFrame(qrels).copy()

corpus_df["code_id"] = corpus_df["code_id"].astype(str)
corpus_df["code"] = corpus_df["code"].fillna("").astype(str)
corpus_df["language"] = corpus_df["language"].astype(str).str.lower()
corpus_df["meta_source_problem_id"] = (
    corpus_df["meta_source_problem_id"].astype(str)
)

queries_df["query_id"] = queries_df["query_id"].astype(str)
queries_df["meta_anchor_code_id"] = (
    queries_df["meta_anchor_code_id"].astype(str)
)

qrels_df["query_id"] = qrels_df["query_id"].astype(str)
qrels_df["doc_id"] = qrels_df["doc_id"].astype(str)
qrels_df["relevance"] = pd.to_numeric(
    qrels_df["relevance"], errors="coerce"
).fillna(0).astype(int)

queries_df = queries_df[
    queries_df["subtask"].eq("c2c_cross_lang")
].copy()

query_ids = set(queries_df["query_id"])
qrels_df = qrels_df[qrels_df["query_id"].isin(query_ids)].copy()

assert corpus_df["code_id"].is_unique, "Duplicate code_id values found."

# ============================================================
# 2. PARSE TARGET LANGUAGES
# ============================================================

def parse_languages(value):
    if isinstance(value, (list, tuple, set, np.ndarray)):
        values = list(value)
    else:
        s = str(value).strip()
        try:
            parsed = ast.literal_eval(s)
            values = (
                list(parsed)
                if isinstance(parsed, (list, tuple, set))
                else [parsed]
            )
        except (ValueError, SyntaxError):
            values = re.split(r"[,;]", s.strip("[](){}"))

    return {
        str(v).strip().lower().strip("'\"")
        for v in values
        if str(v).strip()
    }

queries_df["target_languages"] = (
    queries_df["meta_target_languages"].apply(parse_languages)
)

# ============================================================
# 3. TF-IDF REPRESENTATIONS
# ============================================================

vectorizer = TfidfVectorizer(
    token_pattern=(
        r"[A-Za-z_]\w*|\d+(?:\.\d+)?|==|!=|<=|>=|&&|\|\||"
        r"\+\+|--|->|[^\s]"
    ),
    ngram_range=(1, 2),
    lowercase=False,
    sublinear_tf=True,
)

X = vectorizer.fit_transform(corpus_df["code"])

doc_ids = corpus_df["code_id"].tolist()
doc_id_to_idx = {doc_id: i for i, doc_id in enumerate(doc_ids)}

# Group candidate implementations by source problem and language.
groups = {}

for i, row in corpus_df.iterrows():
    key = (
        str(row["meta_source_problem_id"]),
        str(row["language"]).lower()
    )
    groups.setdefault(key, []).append(i)

# Query-specific qrels: doc_id -> relevance grade.
qrel_maps = {}

for qid, sub in qrels_df.groupby("query_id"):
    qrel_maps[str(qid)] = dict(
        zip(sub["doc_id"], sub["relevance"])
    )

# ============================================================
# 4. METRICS
# ============================================================

def calculate_metrics(ranked_grades):
    grades = np.asarray(ranked_grades, dtype=int)
    relevant = grades >= 2
    total_relevant = int(relevant.sum())

    result = {}

    for k in (1, 5, 10):
        top_relevant = relevant[:k]
        result[f"Hit@{k}"] = float(top_relevant.any())
        result[f"Recall@{k}"] = (
            float(top_relevant.sum() / total_relevant)
            if total_relevant else 0.0
        )

    positions = np.flatnonzero(relevant)
    result["MRR"] = (
        float(1.0 / (positions[0] + 1))
        if len(positions) else 0.0
    )

    # Graded nDCG: rel=2 is positive, rel=1 is a hard negative.
    top_grades = grades[:10]
    discounts = 1.0 / np.log2(np.arange(2, len(top_grades) + 2))
    gains = np.power(2.0, top_grades) - 1.0
    dcg = float(np.sum(gains * discounts))

    ideal_grades = np.sort(grades)[::-1][:10]
    ideal_gains = np.power(2.0, ideal_grades) - 1.0
    ideal_discounts = 1.0 / np.log2(
        np.arange(2, len(ideal_grades) + 2)
    )
    idcg = float(np.sum(ideal_gains * ideal_discounts))

    result["nDCG@10"] = dcg / idcg if idcg > 0 else 0.0

    return result


metric_cols = [
    "Hit@1", "Hit@5", "Hit@10",
    "Recall@1", "Recall@5", "Recall@10",
    "MRR", "nDCG@10"
]

# ============================================================
# 5. RUN WITH FIXED GROUP RELEVANCE
# ============================================================

N_SEEDS = 10
per_query_rows = []
skipped = []

for seed in range(N_SEEDS):
    rng = np.random.default_rng(seed)

    for _, query in queries_df.iterrows():
        qid = str(query["query_id"])
        anchor_id = str(query["meta_anchor_code_id"])
        targets = query["target_languages"]

        if anchor_id not in doc_id_to_idx:
            skipped.append((qid, "anchor_missing"))
            continue

        if not targets:
            skipped.append((qid, "target_language_missing"))
            continue

        query_qrels = qrel_maps.get(qid, {})

        # Keep only groups whose language is allowed by this query.
        candidate_groups = []

        for key, original_members in groups.items():
            problem, language = key

            if language not in targets:
                continue

            members = [
                i for i in original_members
                if doc_ids[i] != anchor_id
            ]

            if not members:
                continue

            # CRITICAL FIX:
            # Group relevance is computed from the FULL group.
            # It does not change when k=1 or k=2 selects a subset.
            group_grade = max(
                (query_qrels.get(doc_ids[i], 0) for i in members),
                default=0
            )

            candidate_groups.append((key, members, int(group_grade)))

        candidate_groups.sort(key=lambda item: item[0])

        if not candidate_groups:
            skipped.append((qid, "no_candidate_groups"))
            continue

        q_idx = doc_id_to_idx[anchor_id]
        similarities = (X @ X.getrow(q_idx).T).toarray().ravel()

        # Sampling is performed once per group per seed.
        # Max and mean use the exact same selected documents.
        sampled = {}

        for key, members, grade in candidate_groups:
            shuffled = rng.permutation(members).tolist()
            sampled[key] = {
                "k=1": shuffled[:1],
                "k=2": shuffled[:2],
                "all": members[:],
            }

        for setting in ("k=1", "k=2", "all"):
            for method in ("max", "mean"):
                ranked_groups = []

                for key, members, fixed_grade in candidate_groups:
                    chosen = sampled[key][setting]
                    scores = similarities[chosen]

                    if method == "max":
                        score = float(np.max(scores))
                    else:
                        score = float(np.mean(scores))

                    # Same fixed grade for every setting and method.
                    ranked_groups.append(
                        (key, score, fixed_grade)
                    )

                ranked_groups.sort(
                    key=lambda item: (-item[1], item[0])
                )

                ranked_grades = [
                    item[2] for item in ranked_groups
                ]

                metrics = calculate_metrics(ranked_grades)

                per_query_rows.append({
                    "query_id": qid,
                    "seed": seed,
                    "setting": setting,
                    "method": method,
                    **metrics,
                })

    print(f"Completed seed {seed + 1}/{N_SEEDS}")

results_df = pd.DataFrame(per_query_rows)

if results_df.empty:
    raise RuntimeError("No results produced.")

results_path = "/content/coreb_grouped_fixed_labels_results.csv"
results_df.to_csv(results_path, index=False)

# ============================================================
# 6. SUMMARY: MEAN +/- SD ACROSS SEEDS
# ============================================================

run_means = (
    results_df.groupby(["setting", "method", "seed"])[metric_cols]
    .mean()
    .reset_index()
)

summary = (
    run_means.groupby(["setting", "method"])[metric_cols]
    .agg(["mean", "std"])
)

print("\n" + "=" * 65)
print("FIXED-LABEL RESULTS — MEAN +/- SD ACROSS SEEDS")
print("=" * 65)

for setting in ("k=1", "k=2", "all"):
    print(f"\n--- {setting} ---")

    for method in ("max", "mean"):
        if (setting, method) not in summary.index:
            continue

        print(f"\nMethod: {method}")

        for metric in metric_cols:
            mean_val = summary.loc[(setting, method), (metric, "mean")]
            sd_val = summary.loc[(setting, method), (metric, "std")]

            if pd.isna(sd_val):
                sd_val = 0.0

            print(
                f"  {metric:12s}: {mean_val:.4f} +/- {sd_val:.4f}"
            )

# ============================================================
# 7. PAIRED BOOTSTRAP
# ============================================================

# Average repeated random selections within each query first.
query_means = (
    results_df.groupby(["setting", "query_id", "method"])[metric_cols]
    .mean()
    .reset_index()
)

N_BOOTSTRAP = 5000
boot_rng = np.random.default_rng(20261029)

def paired_bootstrap_ci(differences, n_bootstrap=5000):
    differences = np.asarray(differences, dtype=float)
    n = len(differences)

    boot = np.empty(n_bootstrap)
    for b in range(n_bootstrap):
        idx = boot_rng.integers(0, n, size=n)
        boot[b] = differences[idx].mean()

    return (
        float(differences.mean()),
        float(np.quantile(boot, 0.025)),
        float(np.quantile(boot, 0.975))
    )

bootstrap_metrics = ["Hit@1", "Hit@5", "Recall@10", "MRR", "nDCG@10"]

print("\n" + "=" * 65)
print("PAIRED BOOTSTRAP: MAX - MEAN (95% CI)")
print("=" * 65)

for setting in ("k=1", "k=2", "all"):
    subset = query_means[query_means["setting"] == setting]

    for metric in bootstrap_metrics:
        pivot = subset.pivot(
            index="query_id",
            columns="method",
            values=metric
        ).dropna(subset=["max", "mean"])

        if len(pivot) < 2:
            continue

        delta, lower, upper = paired_bootstrap_ci(
            pivot["max"].to_numpy() - pivot["mean"].to_numpy(),
            N_BOOTSTRAP
        )

        print(
            f"{setting:5s} | {metric:10s} | "
            f"{delta:+.4f} [{lower:+.4f}, {upper:+.4f}]"
        )

print("\n" + "=" * 65)
print("PAIRED BOOTSTRAP: k=2 - k=1 (95% CI)")
print("=" * 65)

for method in ("max", "mean"):
    subset = query_means[query_means["method"] == method]

    for metric in bootstrap_metrics:
        pivot = subset.pivot(
            index="query_id",
            columns="setting",
            values=metric
        ).dropna(subset=["k=1", "k=2"])

        if len(pivot) < 2:
            continue

        delta, lower, upper = paired_bootstrap_ci(
            pivot["k=2"].to_numpy() - pivot["k=1"].to_numpy(),
            N_BOOTSTRAP
        )

        print(
            f"{method:4s} | {metric:10s} | "
            f"{delta:+.4f} [{lower:+.4f}, {upper:+.4f}]"
        )

print("\nSkipped queries:", len(skipped))
if skipped:
    print("Examples:", skipped[:10])

print("Saved:", results_path)
print("\nFIXED-LABEL EXPERIMENT — END")

Completed seed 1/10
Completed seed 2/10
Completed seed 3/10
Completed seed 4/10
Completed seed 5/10
Completed seed 6/10
Completed seed 7/10
Completed seed 8/10
Completed seed 9/10
Completed seed 10/10

FIXED-LABEL RESULTS — MEAN +/- SD ACROSS SEEDS

--- k=1 ---

Method: max
  Hit@1       : 0.8389 +/- 0.0211
  Hit@5       : 0.9153 +/- 0.0156
  Hit@10      : 0.9442 +/- 0.0093
  Recall@1    : 0.4665 +/- 0.0148
  Recall@5    : 0.7721 +/- 0.0173
  Recall@10   : 0.8315 +/- 0.0130
  MRR         : 0.8741 +/- 0.0169
  nDCG@10     : 0.7969 +/- 0.0156

Method: mean
  Hit@1       : 0.8389 +/- 0.0211
  Hit@5       : 0.9153 +/- 0.0156
  Hit@10      : 0.9442 +/- 0.0093
  Recall@1    : 0.4665 +/- 0.0148
  Recall@5    : 0.7721 +/- 0.0173
  Recall@10   : 0.8315 +/- 0.0130
  MRR         : 0.8741 +/- 0.0169
  nDCG@10     : 0.7969 +/- 0.0156

--- k=2 ---

Method: max
  Hit@1       : 0.9211 +/- 0.0000
  Hit@5       : 0.9842 +/- 0.0000
  Hit@10      : 1.0000 +/- 0.0000
  Recall@1    : 0.5276 +/- 0.0000
  Rec

In [ ]:

import ast
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

# ============================================================
# 1. PREPARE DATA
# ============================================================

c = pd.DataFrame(corpus).copy().reset_index(drop=True)
q = pd.DataFrame(queries).copy()
qr = pd.DataFrame(qrels).copy()

c["code_id"] = c["code_id"].astype(str)
c["code"] = c["code"].fillna("").astype(str)
c["language"] = c["language"].astype(str).str.lower()
c["meta_source_problem_id"] = (
    c["meta_source_problem_id"].astype(str)
)

q["query_id"] = q["query_id"].astype(str)
q["meta_anchor_code_id"] = q["meta_anchor_code_id"].astype(str)

qr["query_id"] = qr["query_id"].astype(str)
qr["doc_id"] = qr["doc_id"].astype(str)
qr["relevance"] = pd.to_numeric(
    qr["relevance"], errors="coerce"
).fillna(0).astype(int)

assert c["code_id"].is_unique, "Duplicate code_id values found."
assert "meta_passed" in c.columns, "meta_passed is missing."

q = q[q["subtask"].eq("c2c_cross_lang")].copy()
query_ids = set(q["query_id"])
qr = qr[qr["query_id"].isin(query_ids)].copy()

def to_bool(value):
    if pd.isna(value):
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {
        "true", "1", "yes"
    }

c["passed"] = c["meta_passed"].apply(to_bool)

def parse_languages(value):
    if isinstance(value, (list, tuple, set, np.ndarray)):
        values = list(value)
    else:
        s = str(value).strip()
        try:
            parsed = ast.literal_eval(s)
            values = (
                list(parsed)
                if isinstance(parsed, (list, tuple, set))
                else [parsed]
            )
        except (ValueError, SyntaxError):
            values = re.split(r"[,;]", s.strip("[](){}"))

    return {
        str(v).strip().lower().strip("'\"")
        for v in values
        if str(v).strip()
    }

q["target_languages"] = q["meta_target_languages"].apply(
    parse_languages
)

print("=" * 65)
print("REFERENCE QUALITY EXPERIMENT")
print("=" * 65)
print("Cross-language queries:", len(q))
print("Corpus documents:", len(c))
print("Documents passed:", int(c["passed"].sum()))
print("Documents failed:", int((~c["passed"]).sum()))

# Check whether positive judgments agree with pass status.
judged = qr.merge(
    c[["code_id", "passed"]],
    left_on="doc_id",
    right_on="code_id",
    how="left",
    validate="many_to_one"
)

positive_docs = judged[judged["relevance"].ge(2)]
print(
    "Positive judged documents not marked passed:",
    int((~positive_docs["passed"].fillna(False)).sum())
)

# ============================================================
# 2. TF-IDF
# ============================================================

vectorizer = TfidfVectorizer(
    token_pattern=(
        r"[A-Za-z_]\w*|\d+(?:\.\d+)?|==|!=|<=|>=|&&|\|\||"
        r"\+\+|--|->|[^\s]"
    ),
    ngram_range=(1, 2),
    lowercase=False,
    sublinear_tf=True,
)

X = vectorizer.fit_transform(c["code"])

doc_ids = c["code_id"].tolist()
doc_id_to_idx = {doc_id: i for i, doc_id in enumerate(doc_ids)}
passed_flags = c["passed"].tolist()

# Group documents by source problem and language.
groups = {}

for i, row in c.iterrows():
    key = (
        str(row["meta_source_problem_id"]),
        str(row["language"]).lower()
    )
    groups.setdefault(key, []).append(i)

qrel_maps = {}

for qid, sub in qr.groupby("query_id"):
    qrel_maps[str(qid)] = dict(
        zip(sub["doc_id"], sub["relevance"])
    )

# ============================================================
# 3. METRICS
# CoREB relevance threshold = 2.
# rel=1 hard negatives get zero gain in these metrics.
# ============================================================

metric_cols = [
    "Hit@1", "Hit@5", "Hit@10",
    "Recall@1", "Recall@5", "Recall@10",
    "MRR", "nDCG@10"
]

def calculate_metrics(group_grades):
    grades = np.asarray(group_grades, dtype=int)
    rel = (grades >= 2).astype(int)
    total_relevant = int(rel.sum())

    result = {}

    for k in (1, 5, 10):
        top = rel[:k]
        result[f"Hit@{k}"] = float(top.sum() > 0)
        result[f"Recall@{k}"] = (
            float(top.sum() / total_relevant)
            if total_relevant else 0.0
        )

    positions = np.flatnonzero(rel)
    result["MRR"] = (
        float(1.0 / (positions[0] + 1))
        if len(positions) else 0.0
    )

    top10 = rel[:10]
    discounts = 1.0 / np.log2(np.arange(2, len(top10) + 2))
    dcg = float(np.sum(top10 * discounts))

    ideal = np.sort(rel)[::-1][:10]
    ideal_discounts = 1.0 / np.log2(
        np.arange(2, len(ideal) + 2)
    )
    idcg = float(np.sum(ideal * ideal_discounts))

    result["nDCG@10"] = dcg / idcg if idcg > 0 else 0.0
    return result

# ============================================================
# 4. RUN EXPERIMENT
# Group relevance stays fixed across quality conditions.
# Candidate scores change according to selected implementations.
# ============================================================

N_SEEDS = 10
rows = []
skipped = []
relevant_group_checks = []

for seed in range(N_SEEDS):
    for query_number, (_, query) in enumerate(q.iterrows()):
        qid = str(query["query_id"])
        anchor_id = str(query["meta_anchor_code_id"])
        targets = query["target_languages"]

        if anchor_id not in doc_id_to_idx:
            skipped.append((qid, "anchor_missing"))
            continue

        if not targets:
            skipped.append((qid, "target_language_missing"))
            continue

        query_qrels = qrel_maps.get(qid, {})
        candidate_groups = []

        # Prepare each target group once, with a fixed relevance label.
        for key, original_members in groups.items():
            problem, language = key

            if language not in targets:
                continue

            members = [
                i for i in original_members
                if doc_ids[i] != anchor_id
            ]

            if not members:
                continue

            passed_members = [
                i for i in members if passed_flags[i]
            ]

            # Relevance is based on the full group, not a random subset.
            fixed_grade = max(
                (query_qrels.get(doc_ids[i], 0) for i in members),
                default=0
            )

            candidate_groups.append({
                "key": key,
                "all": members,
                "passed": passed_members,
                "grade": int(fixed_grade),
            })

        candidate_groups.sort(key=lambda x: x["key"])

        if not candidate_groups:
            skipped.append((qid, "no_candidate_groups"))
            continue

        # Positive groups should remain available after passed filtering,
        # because positive CoREB documents are expected to have passed.
        n_relevant_any = sum(
            g["grade"] >= 2 for g in candidate_groups
        )
        n_relevant_passed = sum(
            g["grade"] >= 2 and len(g["passed"]) > 0
            for g in candidate_groups
        )

        relevant_group_checks.append({
            "query_id": qid,
            "all_relevant_groups": n_relevant_any,
            "passed_relevant_groups": n_relevant_passed,
        })

        if n_relevant_any != n_relevant_passed:
            raise ValueError(
                f"Relevant groups lost after pass filtering: {qid}"
            )

        query_idx = doc_id_to_idx[anchor_id]
        similarities = (X @ X.getrow(query_idx).T).toarray().ravel()

        rng_any = np.random.default_rng(
            (seed + 1) * 1_000_003 + query_number * 101 + 1
        )
        rng_passed = np.random.default_rng(
            (seed + 1) * 1_000_003 + query_number * 101 + 2
        )

        conditions = [
            ("one_any", "all", "one"),
            ("one_passed", "passed", "one"),
            ("all_any", "all", "all"),
            ("all_passed", "passed", "all"),
        ]

        for condition, member_mode, selection_mode in conditions:
            method_list = (
                ["single"]
                if selection_mode == "one"
                else ["max", "mean"]
            )

            # Remove groups that have no available documents
            # under the chosen quality filter.
            available = [
                g for g in candidate_groups
                if len(g[member_mode]) > 0
            ]

            for method in method_list:
                scored = []

                for g in available:
                    members = g[member_mode]

                    if selection_mode == "one":
                        rng = (
                            rng_any
                            if member_mode == "all"
                            else rng_passed
                        )
                        chosen = [int(rng.choice(members))]
                    else:
                        chosen = members

                    member_scores = similarities[chosen]

                    if method == "max":
                        score = float(np.max(member_scores))
                    else:
                        score = float(np.mean(member_scores))

                    scored.append((
                        g["key"],
                        score,
                        g["grade"]
                    ))

                scored.sort(key=lambda x: (-x[1], x[0]))
                grades = [x[2] for x in scored]
                metrics = calculate_metrics(grades)

                rows.append({
                    "query_id": qid,
                    "seed": seed,
                    "condition": condition,
                    "method": method,
                    "candidate_groups": len(scored),
                    "relevant_groups": int(sum(g >= 2 for g in grades)),
                    **metrics,
                })

    print(f"Completed seed {seed + 1}/{N_SEEDS}")

results = pd.DataFrame(rows)

if results.empty:
    raise RuntimeError("No results produced.")

# ============================================================
# 5. SUMMARY TABLE
# ============================================================

run_means = (
    results.groupby(["condition", "method", "seed"])[
        metric_cols + ["candidate_groups"]
    ]
    .mean()
    .reset_index()
)

summary = (
    run_means.groupby(["condition", "method"])[
        metric_cols + ["candidate_groups"]
    ]
    .agg(["mean", "std"])
)

print("\n" + "=" * 65)
print("QUALITY EXPERIMENT RESULTS — MEAN +/- SD")
print("=" * 65)

for condition in (
    "one_any", "one_passed", "all_any", "all_passed"
):
    print(f"\n--- {condition} ---")

    for method in (
        ["single"] if condition.startswith("one_")
        else ["max", "mean"]
    ):
        if (condition, method) not in summary.index:
            continue

        print(f"\nMethod: {method}")

        for metric in metric_cols + ["candidate_groups"]:
            mean_value = summary.loc[
                (condition, method), (metric, "mean")
            ]
            sd_value = summary.loc[
                (condition, method), (metric, "std")
            ]

            if pd.isna(sd_value):
                sd_value = 0.0

            print(
                f"  {metric:16s}: "
                f"{mean_value:.4f} +/- {sd_value:.4f}"
            )

# ============================================================
# 6. PAIRED BOOTSTRAP: PASSED FILTER MINUS ALL
# Positive delta means the passed-only condition performed better.
# ============================================================

query_means = (
    results.groupby(["condition", "query_id", "method"])[metric_cols]
    .mean()
    .reset_index()
)

BOOTSTRAPS = 5000
boot_rng = np.random.default_rng(20261029)

def bootstrap_ci(difference):
    difference = np.asarray(difference, dtype=float)
    n = len(difference)

    sampled_means = np.empty(BOOTSTRAPS)

    for b in range(BOOTSTRAPS):
        idx = boot_rng.integers(0, n, size=n)
        sampled_means[b] = difference[idx].mean()

    return (
        float(difference.mean()),
        float(np.quantile(sampled_means, 0.025)),
        float(np.quantile(sampled_means, 0.975))
    )

print("\n" + "=" * 65)
print("PAIRED BOOTSTRAP: PASSED-ONLY MINUS ALL")
print("=" * 65)

comparisons = [
    ("one_passed", "one_any", "single"),
    ("all_passed", "all_any", "max"),
    ("all_passed", "all_any", "mean"),
]

for better_condition, baseline_condition, method in comparisons:
    a = query_means[
        (query_means["condition"] == better_condition)
        & (query_means["method"] == method)
    ].set_index("query_id")

    b = query_means[
        (query_means["condition"] == baseline_condition)
        & (query_means["method"] == method)
    ].set_index("query_id")

    joined = a[metric_cols].join(
        b[metric_cols],
        how="inner",
        lsuffix="_passed",
        rsuffix="_all"
    )

    print(
        f"\nComparison: {better_condition} - {baseline_condition}"
        f" ({method})"
    )

    for metric in [
        "Hit@1", "Hit@5", "Recall@10", "MRR", "nDCG@10"
    ]:
        delta, lo, hi = bootstrap_ci(
            joined[f"{metric}_passed"].to_numpy()
            - joined[f"{metric}_all"].to_numpy()
        )

        print(
            f"  {metric:10s}: {delta:+.4f} "
            f"[{lo:+.4f}, {hi:+.4f}]"
        )

print("\nQueries skipped:", len(skipped))
print("Relevant-group checks:",
      len(relevant_group_checks))

check_df = pd.DataFrame(relevant_group_checks)
if not check_df.empty:
    print(
        "Queries where relevant group counts differ:",
        int((
            check_df["all_relevant_groups"]
            != check_df["passed_relevant_groups"]
        ).sum())
    )

results_path = "/content/coreb_reference_quality_results.csv"
results.to_csv(results_path, index=False)

print("\nSaved:", results_path)
print("\nREFERENCE QUALITY EXPERIMENT — END")


REFERENCE QUALITY EXPERIMENT
Cross-language queries: 190
Corpus documents: 1744
Documents passed: 537
Documents failed: 1207
Positive judged documents not marked passed: 0
Completed seed 1/10
Completed seed 2/10
Completed seed 3/10
Completed seed 4/10
Completed seed 5/10
Completed seed 6/10
Completed seed 7/10
Completed seed 8/10
Completed seed 9/10
Completed seed 10/10

QUALITY EXPERIMENT RESULTS — MEAN +/- SD

--- one_any ---

Method: single
  Hit@1           : 0.8284 +/- 0.0109
  Hit@5           : 0.9184 +/- 0.0134
  Hit@10          : 0.9432 +/- 0.0133
  Recall@1        : 0.4614 +/- 0.0065
  Recall@5        : 0.7697 +/- 0.0129
  Recall@10       : 0.8272 +/- 0.0126
  MRR             : 0.8681 +/- 0.0090
  nDCG@10         : 0.7906 +/- 0.0075
  candidate_groups: 395.1316 +/- 0.0000

--- one_passed ---

Method: single
  Hit@1           : 0.9047 +/- 0.0139
  Hit@5           : 0.9663 +/- 0.0112
  Hit@10          : 0.9805 +/- 0.0082
  Recall@1        : 0.5138 +/- 0.0105
  Recall@5        : 

In [ ]:

# FINAL CONTROL: SAME CANDIDATE GROUPS, DIFFERENT REFERENCES
# Requires variables from the previous quality experiment cell.

import numpy as np
import pandas as pd

metrics_to_check = [
    "Hit@1", "Hit@5", "Hit@10",
    "Recall@1", "Recall@5", "Recall@10",
    "MRR", "nDCG@10"
]

control_rows = []
candidate_counts = []
lost_relevant_groups = []

for _, query in q.iterrows():
    qid = str(query["query_id"])
    anchor_id = str(query["meta_anchor_code_id"])
    targets = query["target_languages"]
    query_qrels = qrel_maps.get(qid, {})

    if anchor_id not in doc_id_to_idx:
        continue

    query_idx = doc_id_to_idx[anchor_id]
    similarities = (X @ X.getrow(query_idx).T).toarray().ravel()

    fixed_candidates = []

    for key, original_members in groups.items():
        problem, language = key

        if language not in targets:
            continue

        members = [
            i for i in original_members
            if doc_ids[i] != anchor_id
        ]

        if not members:
            continue

        passed_members = [
            i for i in members
            if passed_flags[i]
        ]

        # Keep exactly the groups that have at least one passed solution.
        if not passed_members:
            continue

        # Keep relevance labels fixed across all compared methods.
        grade = max(
            (query_qrels.get(doc_ids[i], 0) for i in members),
            default=0
        )

        fixed_candidates.append({
            "key": key,
            "all_members": members,
            "passed_members": passed_members,
            "grade": int(grade),
        })

    fixed_candidates.sort(key=lambda g: g["key"])
    candidate_counts.append(len(fixed_candidates))

    # Safety check: every positive group in the original candidate pool
    # must survive in the shared candidate set.
    original_relevant = set()

    for key, original_members in groups.items():
        if key[1] not in targets:
            continue

        members = [
            i for i in original_members
            if doc_ids[i] != anchor_id
        ]

        if any(query_qrels.get(doc_ids[i], 0) >= 2 for i in members):
            original_relevant.add(key)

    retained_relevant = {
        g["key"] for g in fixed_candidates
        if g["grade"] >= 2
    }

    lost = original_relevant - retained_relevant
    lost_relevant_groups.append(len(lost))

    if lost:
        raise RuntimeError(
            f"Relevant groups lost for query {qid}: {len(lost)}"
        )

    # Candidate group set is identical for all four methods.
    for aggregation in ("max", "mean"):
        for ref_set in ("all", "passed"):
            scored = []

            for group in fixed_candidates:
                members = (
                    group["all_members"]
                    if ref_set == "all"
                    else group["passed_members"]
                )

                member_scores = similarities[members]

                if aggregation == "max":
                    score = float(np.max(member_scores))
                else:
                    score = float(np.mean(member_scores))

                scored.append((
                    group["key"],
                    score,
                    group["grade"]
                ))

            scored.sort(key=lambda x: (-x[1], x[0]))
            ranked_grades = [item[2] for item in scored]

            control_rows.append({
                "query_id": qid,
                "condition": f"{ref_set}_{aggregation}",
                "candidate_groups": len(scored),
                **calculate_metrics(ranked_grades)
            })

control = pd.DataFrame(control_rows)

print("=" * 65)
print("MATCHED-CANDIDATE CONTROL")
print("=" * 65)
print("Queries evaluated:", control["query_id"].nunique())
print("Mean candidate groups:", round(np.mean(candidate_counts), 2))
print("Maximum relevant groups lost:", max(lost_relevant_groups, default=0))

# Mean per-query metrics for each deterministic condition.
print("\nMETRICS")
summary_control = (
    control.groupby("condition")[metrics_to_check]
    .mean()
    .round(4)
)
print(summary_control.to_string())

# Paired bootstrap by query: passed references minus all references.
BOOTSTRAPS = 5000
rng = np.random.default_rng(20261030)

print("\nPAIRED BOOTSTRAP: PASSED - ALL")

for aggregation in ("max", "mean"):
    a = (
        control[control["condition"] == f"passed_{aggregation}"]
        .set_index("query_id")[metrics_to_check]
    )
    b = (
        control[control["condition"] == f"all_{aggregation}"]
        .set_index("query_id")[metrics_to_check]
    )

    paired = a.join(b, how="inner", lsuffix="_passed", rsuffix="_all")

    print(f"\nAggregation: {aggregation}")

    for metric in ("Hit@1", "Hit@5", "Recall@10", "MRR", "nDCG@10"):
        delta = (
            paired[f"{metric}_passed"].to_numpy()
            - paired[f"{metric}_all"].to_numpy()
        )

        n = len(delta)
        samples = np.empty(BOOTSTRAPS)

        for j in range(BOOTSTRAPS):
            idx = rng.integers(0, n, size=n)
            samples[j] = delta[idx].mean()

        lo, hi = np.quantile(samples, [0.025, 0.975])

        print(
            f"{metric:10s}: {delta.mean():+.4f} "
            f"[{lo:+.4f}, {hi:+.4f}]"
        )

control.to_csv(
    "/content/coreb_matched_reference_quality_control.csv",
    index=False
)
print("\nSaved: /content/coreb_matched_reference_quality_control.csv")
print("MATCHED-CANDIDATE CONTROL — END")


MATCHED-CANDIDATE CONTROL
Queries evaluated: 190
Mean candidate groups: 151.45
Maximum relevant groups lost: 0

METRICS
              Hit@1   Hit@5  Hit@10  Recall@1  Recall@5  Recall@10     MRR  nDCG@10
condition                                                                          
all_max      0.9421  0.9947     1.0    0.5421    0.9289     0.9548  0.9627   0.9320
all_mean     0.9421  0.9842     1.0    0.5434    0.9140     0.9421  0.9611   0.9231
passed_max   0.9421  0.9947     1.0    0.5421    0.9254     0.9557  0.9631   0.9331
passed_mean  0.9526  0.9842     1.0    0.5500    0.9066     0.9425  0.9662   0.9236

PAIRED BOOTSTRAP: PASSED - ALL

Aggregation: max
Hit@1     : +0.0000 [+0.0000, +0.0000]
Hit@5     : +0.0000 [+0.0000, +0.0000]
Recall@10 : +0.0009 [-0.0039, +0.0061]
MRR       : +0.0004 [+0.0000, +0.0013]
nDCG@10   : +0.0011 [-0.0018, +0.0038]

Aggregation: mean
Hit@1     : +0.0105 [-0.0105, +0.0316]
Hit@5     : +0.0000 [+0.0000, +0.0000]
Recall@10 : +0.0004 [-0.0079, +0.0

In [ ]:

# COREB: CLUSTER BOOTSTRAP BY SOURCE PROBLEM
# Uses existing variables from the previous experiment cells:
# q, results, control
#
# The point estimates are query-weighted.
# Bootstrap resampling is performed at source-problem level.

import numpy as np
import pandas as pd

METRICS = [
    "Hit@1", "Hit@5", "Recall@10", "MRR", "nDCG@10"
]
N_BOOT = 5000
RANDOM_SEED = 20261030

# ------------------------------------------------------------
# 1. Map each query to its source problem.
# ------------------------------------------------------------

if "meta_source_problem_id" not in q.columns:
    raise ValueError(
        "meta_source_problem_id is missing from q. "
        "Check the available query metadata before continuing."
    )

problem_map = q[
    ["query_id", "meta_source_problem_id"]
].copy()

problem_map["query_id"] = problem_map["query_id"].astype(str)
problem_map["meta_source_problem_id"] = (
    problem_map["meta_source_problem_id"].astype(str)
)

problem_map = problem_map.drop_duplicates()

if problem_map["query_id"].duplicated().any():
    raise ValueError(
        "A query maps to multiple source problems. "
        "Resolve this mapping before bootstrapping."
    )

# ------------------------------------------------------------
# 2. Paired cluster-bootstrap function.
#
# The observed difference is the mean difference across queries.
# Each bootstrap replicate samples source problems with
# replacement and includes all paired queries for each sampled
# problem. Repeatedly sampled problems are included repeatedly.
# ------------------------------------------------------------

def cluster_bootstrap_difference(
    df,
    condition_col,
    method_col,
    condition_a,
    condition_b,
    method,
    comparison_name,
    seed
):
    required = {
        "query_id", condition_col, method_col, *METRICS
    }
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{comparison_name}: missing columns {sorted(missing)}"
        )

    a = df[
        (df[condition_col] == condition_a)
        & (df[method_col] == method)
    ].copy()

    b = df[
        (df[condition_col] == condition_b)
        & (df[method_col] == method)
    ].copy()

    # Average random reference-selection seeds within each query.
    a = a.groupby("query_id", as_index=False)[METRICS].mean()
    b = b.groupby("query_id", as_index=False)[METRICS].mean()

    a["query_id"] = a["query_id"].astype(str)
    b["query_id"] = b["query_id"].astype(str)

    a = a.rename(
        columns={m: f"{m}_a" for m in METRICS}
    )
    b = b.rename(
        columns={m: f"{m}_b" for m in METRICS}
    )

    paired = a.merge(b, on="query_id", how="inner")
    paired = paired.merge(
        problem_map,
        on="query_id",
        how="left",
        validate="one_to_one"
    )

    if paired["meta_source_problem_id"].isna().any():
        raise ValueError(
            f"{comparison_name}: some queries lack problem IDs."
        )

    if len(paired) == 0:
        raise ValueError(
            f"{comparison_name}: no paired queries found."
        )

    clusters = paired["meta_source_problem_id"].unique()

    if len(clusters) < 2:
        raise ValueError(
            f"{comparison_name}: not enough source problems."
        )

    delta_columns = {}

    for metric in METRICS:
        delta_col = f"delta_{metric}"
        paired[delta_col] = (
            paired[f"{metric}_a"] - paired[f"{metric}_b"]
        )
        delta_columns[metric] = delta_col

    # Cache query-level values for each source problem.
    cluster_values = {
        cluster: sub[
            [delta_columns[m] for m in METRICS]
        ].to_numpy(dtype=float)
        for cluster, sub in paired.groupby(
            "meta_source_problem_id", sort=False
        )
    }

    rng = np.random.default_rng(seed)
    bootstrap = np.empty((N_BOOT, len(METRICS)), dtype=float)

    cluster_ids = list(cluster_values.keys())

    for b_idx in range(N_BOOT):
        sampled_clusters = rng.choice(
            cluster_ids,
            size=len(cluster_ids),
            replace=True
        )

        sampled_rows = np.concatenate(
            [cluster_values[c] for c in sampled_clusters],
            axis=0
        )

        bootstrap[b_idx, :] = sampled_rows.mean(axis=0)

    observed = paired[
        [delta_columns[m] for m in METRICS]
    ].mean(axis=0).to_numpy(dtype=float)

    low = np.quantile(bootstrap, 0.025, axis=0)
    high = np.quantile(bootstrap, 0.975, axis=0)

    print("\n" + "=" * 72)
    print(comparison_name)
    print(f"Queries paired: {len(paired)}")
    print(f"Source-problem clusters: {len(clusters)}")
    print("Difference = condition A minus condition B")
    print("=" * 72)

    records = []

    for i, metric in enumerate(METRICS):
        print(
            f"{metric:12s}: {observed[i]:+.4f} "
            f"[{low[i]:+.4f}, {high[i]:+.4f}]"
        )

        records.append({
            "comparison": comparison_name,
            "method": method,
            "metric": metric,
            "difference_query_weighted": observed[i],
            "ci95_low_cluster_bootstrap": low[i],
            "ci95_high_cluster_bootstrap": high[i],
            "queries": len(paired),
            "source_problems": len(clusters),
        })

    return records

# ------------------------------------------------------------
# 3. Reanalyse the unrestricted filtering comparisons.
# `results` is the long-form output from the reference-quality
# experiment, with separate condition and method columns.
# ------------------------------------------------------------

all_records = []
seed_offset = 0

unrestricted_comparisons = [
    (
        "one_passed", "one_any", "single",
        "Unrestricted: one passed reference minus one random reference"
    ),
    (
        "all_passed", "all_any", "max",
        "Unrestricted: all-passed versus all references, Max"
    ),
    (
        "all_passed", "all_any", "mean",
        "Unrestricted: all-passed versus all references, Mean"
    ),
]

for condition_a, condition_b, method, label in unrestricted_comparisons:
    all_records.extend(
        cluster_bootstrap_difference(
            df=results,
            condition_col="condition",
            method_col="method",
            condition_a=condition_a,
            condition_b=condition_b,
            method=method,
            comparison_name=label,
            seed=RANDOM_SEED + seed_offset
        )
    )
    seed_offset += 1

# ------------------------------------------------------------
# 4. Reanalyse the matched-candidate control.
# `control` stores conditions named all_max, passed_max,
# all_mean, and passed_mean.
# ------------------------------------------------------------

control_long = control.copy()

control_long["method"] = (
    control_long["condition"].str.rsplit("_", n=1).str[-1]
)

control_long["reference_filter"] = (
    control_long["condition"].str.rsplit("_", n=1).str[0]
)

matched_comparisons = [
    (
        "passed", "all", "max",
        "Matched candidate set: passed versus all references, Max"
    ),
    (
        "passed", "all", "mean",
        "Matched candidate set: passed versus all references, Mean"
    ),
]

for condition_a, condition_b, method, label in matched_comparisons:
    all_records.extend(
        cluster_bootstrap_difference(
            df=control_long,
            condition_col="reference_filter",
            method_col="method",
            condition_a=condition_a,
            condition_b=condition_b,
            method=method,
            comparison_name=label,
            seed=RANDOM_SEED + seed_offset
        )
    )
    seed_offset += 1

# ------------------------------------------------------------
# 5. Save all updated intervals.
# ------------------------------------------------------------

cluster_ci = pd.DataFrame(all_records)

output_path = (
    "/content/coreb_cluster_bootstrap_audit.csv"
)
cluster_ci.to_csv(output_path, index=False)

print("\nSaved:", output_path)
print("COREB CLUSTER BOOTSTRAP AUDIT — END")



Unrestricted: one passed reference minus one random reference
Queries paired: 190
Source-problem clusters: 61
Difference = condition A minus condition B
Hit@1       : +0.0763 [+0.0479, +0.1083]
Hit@5       : +0.0479 [+0.0304, +0.0674]
Recall@10   : +0.0757 [+0.0481, +0.1068]
MRR         : +0.0639 [+0.0409, +0.0899]
nDCG@10     : +0.0837 [+0.0566, +0.1147]

Unrestricted: all-passed versus all references, Max
Queries paired: 190
Source-problem clusters: 61
Difference = condition A minus condition B
Hit@1       : +0.0211 [+0.0000, +0.0514]
Hit@5       : +0.0105 [+0.0000, +0.0267]
Recall@10   : +0.0149 [+0.0053, +0.0257]
MRR         : +0.0148 [+0.0030, +0.0312]
nDCG@10     : +0.0213 [+0.0120, +0.0327]

Unrestricted: all-passed versus all references, Mean
Queries paired: 190
Source-problem clusters: 61
Difference = condition A minus condition B
Hit@1       : +0.0158 [-0.0057, +0.0462]
Hit@5       : +0.0105 [+0.0000, +0.0267]
Recall@10   : +0.0189 [+0.0087, +0.0299]
MRR         : +0.0096 [-

In [14]:
# ============================================================
# COREB: CLUSTER BOOTSTRAP FOR REFERENCE COUNT (k=2 vs k=1)
# Grouped by meta_source_problem_id
# ============================================================

import numpy as np
import pandas as pd
import ast
import re
from sklearn.feature_extraction.text import TfidfVectorizer

METRICS = ["Hit@1", "Hit@5", "Recall@10", "MRR", "nDCG@10"]
N_BOOT = 5000
RANDOM_SEED = 20261031

# ============================================================
# 1. Reconstruct experiment results from in-memory datasets
# ============================================================
print("Reconstructing TF-IDF grouped retrieval experiment in memory...")

c_df = pd.DataFrame(corpus).copy().reset_index(drop=True)
q_df = pd.DataFrame(queries).copy()
qr_df = pd.DataFrame(qrels).copy()

c_df["code_id"] = c_df["code_id"].astype(str)
c_df["code"] = c_df["code"].fillna("").astype(str)
c_df["language"] = c_df["language"].astype(str).str.lower()
c_df["meta_source_problem_id"] = c_df["meta_source_problem_id"].astype(str)

q_df["query_id"] = q_df["query_id"].astype(str)
q_df["meta_anchor_code_id"] = q_df["meta_anchor_code_id"].astype(str)

qr_df["query_id"] = qr_df["query_id"].astype(str)
qr_df["doc_id"] = qr_df["doc_id"].astype(str)
qr_df["relevance"] = pd.to_numeric(qr_df["relevance"], errors="coerce").fillna(0).astype(int)

q_df = q_df[q_df["subtask"].eq("c2c_cross_lang")].copy()
cross_ids = set(q_df["query_id"])
qr_df = qr_df[qr_df["query_id"].isin(cross_ids)].copy()

def parse_langs(val):
    if isinstance(val, (list, tuple, set, np.ndarray)):
        vals = list(val)
    else:
        s = str(val).strip()
        try:
            parsed = ast.literal_eval(s)
            vals = list(parsed) if isinstance(parsed, (list, tuple, set)) else [parsed]
        except (ValueError, SyntaxError):
            vals = re.split(r"[,;]", s.strip("[](){}"))
    return {str(v).strip().lower().strip("'\"") for v in vals if str(v).strip()}

q_df["target_languages"] = q_df["meta_target_languages"].apply(parse_langs)

vectorizer = TfidfVectorizer(
    token_pattern=r"[A-Za-z_]\w*|\d+(?:\.\d+)?|==|!=|<=|>=|&&|\|\||\+\+|--|->|[^\s]",
    ngram_range=(1, 2),
    lowercase=False,
    sublinear_tf=True,
)
X_tfidf = vectorizer.fit_transform(c_df["code"])
doc_ids = c_df["code_id"].tolist()
doc_id_to_idx = {d: idx for idx, d in enumerate(doc_ids)}

groups = {}
for i, row in c_df.iterrows():
    key = (str(row["meta_source_problem_id"]), str(row["language"]).lower())
    groups.setdefault(key, []).append(i)

qrel_maps = {}
for qid, sub in qr_df.groupby("query_id"):
    qrel_maps[str(qid)] = dict(zip(sub["doc_id"], sub["relevance"]))

def eval_metrics(grades):
    g = np.asarray(grades, dtype=int)
    rel = g >= 2
    tot = int(rel.sum())
    res = {}
    for k in (1, 5, 10):
        top_r = rel[:k]
        res[f"Hit@{k}"] = float(top_r.any())
        res[f"Recall@{k}"] = float(top_r.sum() / tot) if tot else 0.0
    pos = np.flatnonzero(rel)
    res["MRR"] = float(1.0 / (pos[0] + 1)) if len(pos) else 0.0

    top_g = g[:10]
    discounts = 1.0 / np.log2(np.arange(2, len(top_g) + 2))
    gains = np.power(2.0, top_g) - 1.0
    dcg = float(np.sum(gains * discounts))

    ideal_g = np.sort(g)[::-1][:10]
    ideal_gains = np.power(2.0, ideal_g) - 1.0
    ideal_disc = 1.0 / np.log2(np.arange(2, len(ideal_g) + 2))
    idcg = float(np.sum(ideal_gains * ideal_disc))
    res["nDCG@10"] = dcg / idcg if idcg > 0 else 0.0
    return res

N_SEEDS = 10
rows = []
for seed in range(N_SEEDS):
    rng = np.random.default_rng(seed)
    for _, query in q_df.iterrows():
        qid = str(query["query_id"])
        anchor_id = str(query["meta_anchor_code_id"])
        targets = query["target_languages"]
        if anchor_id not in doc_id_to_idx or not targets:
            continue
        query_qrels = qrel_maps.get(qid, {})

        candidate_groups = []
        for key, original_members in groups.items():
            problem, language = key
            if language not in targets:
                continue
            members = [i for i in original_members if doc_ids[i] != anchor_id]
            if not members:
                continue
            group_grade = max((query_qrels.get(doc_ids[i], 0) for i in members), default=0)
            candidate_groups.append((key, members, int(group_grade)))

        candidate_groups.sort(key=lambda item: item[0])
        if not candidate_groups:
            continue

        q_idx = doc_id_to_idx[anchor_id]
        similarities = (X_tfidf @ X_tfidf.getrow(q_idx).T).toarray().ravel()

        sampled = {}
        for key, members, grade in candidate_groups:
            shuffled = rng.permutation(members).tolist()
            sampled[key] = {
                "k=1": shuffled[:1],
                "k=2": shuffled[:2]
            }

        for setting in ("k=1", "k=2"):
            for method in ("max", "mean"):
                ranked_groups = []
                for key, members, fixed_grade in candidate_groups:
                    chosen = sampled[key][setting]
                    scores = similarities[chosen]
                    score = float(np.max(scores)) if method == "max" else float(np.mean(scores))
                    ranked_groups.append((key, score, fixed_grade))

                ranked_groups.sort(key=lambda item: (-item[1], item[0]))
                metrics_dict = eval_metrics([item[2] for item in ranked_groups])
                rows.append({
                    "query_id": qid,
                    "seed": seed,
                    "setting": setting,
                    "method": method,
                    **metrics_dict
                })

results_df = pd.DataFrame(rows)
results_df.to_csv("/content/coreb_grouped_fixed_labels_results.csv", index=False)
print("Reconstructed results cached to file.")

# ============================================================
# 2. Map queries to their source problem & Aggregate seeds
# ============================================================
problem_map = q_df[["query_id", "meta_source_problem_id"]].copy().drop_duplicates()
problem_map["query_id"] = problem_map["query_id"].astype(str)
problem_map["meta_source_problem_id"] = problem_map["meta_source_problem_id"].astype(str)

query_estimates = results_df.groupby(["setting", "method", "query_id"])[METRICS].mean().reset_index()
query_estimates["query_id"] = query_estimates["query_id"].astype(str)

# ============================================================
# 3. Cluster Bootstrap Comparison
# ============================================================
def evaluate_k_difference(method_name):
    k1 = query_estimates[(query_estimates["setting"] == "k=1") & (query_estimates["method"] == method_name)].copy()
    k2 = query_estimates[(query_estimates["setting"] == "k=2") & (query_estimates["method"] == method_name)].copy()

    k1 = k1.rename(columns={m: f"{m}_k1" for m in METRICS})
    k2 = k2.rename(columns={m: f"{m}_k2" for m in METRICS})

    paired = k2.merge(k1, on="query_id", how="inner")
    paired = paired.merge(problem_map, on="query_id", how="left")

    clusters = paired["meta_source_problem_id"].unique()

    delta_cols = {}
    for metric in METRICS:
        delta_col = f"delta_{metric}"
        paired[delta_col] = paired[f"{metric}_k2"] - paired[f"{metric}_k1"]
        delta_cols[metric] = delta_col

    cluster_values = {
        c: sub[[delta_cols[m] for m in METRICS]].to_numpy(dtype=float)
        for c, sub in paired.groupby("meta_source_problem_id", sort=False)
    }

    rng = np.random.default_rng(RANDOM_SEED)
    bootstrap_replicates = np.empty((N_BOOT, len(METRICS)), dtype=float)
    cluster_ids = list(cluster_values.keys())

    for b_idx in range(N_BOOT):
        sampled = rng.choice(cluster_ids, size=len(cluster_ids), replace=True)
        sampled_rows = np.concatenate([cluster_values[c] for c in sampled], axis=0)
        bootstrap_replicates[b_idx, :] = sampled_rows.mean(axis=0)

    observed_diffs = paired[[delta_cols[m] for m in METRICS]].mean(axis=0).to_numpy(dtype=float)
    low_bounds = np.quantile(bootstrap_replicates, 0.025, axis=0)
    high_bounds = np.quantile(bootstrap_replicates, 0.975, axis=0)

    print(f"\n{'='*72}")
    print(f"CLUSTER BOOTSTRAP: k=2 minus k=1 (Method: {method_name.upper()})")
    print(f"Queries: {len(paired)} | Clusters (Problems): {len(clusters)}")
    print(f"{'='*72}")

    results_list = []
    for i, metric in enumerate(METRICS):
        print(f"{metric:12s}: {observed_diffs[i]:+.4f} [{low_bounds[i]:+.4f}, {high_bounds[i]:+.4f}]")
        results_list.append({
            "method": method_name,
            "metric": metric,
            "difference": observed_diffs[i],
            "ci_lower": low_bounds[i],
            "ci_upper": high_bounds[i]
        })
    return results_list

bootstrap_results = []
bootstrap_results.extend(evaluate_k_difference("max"))
bootstrap_results.extend(evaluate_k_difference("mean"))

pd.DataFrame(bootstrap_results).to_csv("/content/coreb_k_comparison_bootstrap.csv", index=False)
print("\nSaved cluster bootstrap comparison to: /content/coreb_k_comparison_bootstrap.csv")

Reconstructing TF-IDF grouped retrieval experiment in memory...
Reconstructed results cached to file.

CLUSTER BOOTSTRAP: k=2 minus k=1 (Method: MAX)
Queries: 190 | Clusters (Problems): 61
Hit@1       : +0.0821 [+0.0511, +0.1169]
Hit@5       : +0.0689 [+0.0409, +0.1021]
Recall@10   : +0.1093 [+0.0748, +0.1464]
MRR         : +0.0742 [+0.0487, +0.1036]
nDCG@10     : +0.1149 [+0.0820, +0.1505]

CLUSTER BOOTSTRAP: k=2 minus k=1 (Method: MEAN)
Queries: 190 | Clusters (Problems): 61
Hit@1       : +0.0979 [+0.0662, +0.1330]
Hit@5       : +0.0584 [+0.0294, +0.0919]
Recall@10   : +0.0922 [+0.0597, +0.1287]
MRR         : +0.0826 [+0.0563, +0.1129]
nDCG@10     : +0.1114 [+0.0815, +0.1448]

Saved cluster bootstrap comparison to: /content/coreb_k_comparison_bootstrap.csv


In [2]:
import os
import glob
import pandas as pd

print("=== CSV FILES IN /content ===")
for path in sorted(glob.glob("/content/*.csv")):
    print(f"{os.path.basename(path):55s} "
          f"{os.path.getsize(path) / 1024:.1f} KB")

print("\n=== DATAFRAMES IN MEMORY ===")
for name, obj in list(globals().items()):
    if not isinstance(obj, pd.DataFrame):
        continue

    cols = list(obj.columns)
    relevant = [
        c for c in cols
        if any(term in str(c).lower() for term in [
            "query", "task", "problem", "success",
            "mrr", "recall", "seed", "method",
            "aggregate", "reference", "condition",
            "rank", "hit", "ndcg"
        ])
    ]

    if relevant:
        print(f"\n{name}: shape={obj.shape}")
        print("Columns:", cols[:35])

=== CSV FILES IN /content ===

=== DATAFRAMES IN MEMORY ===


In [15]:
import os
import shutil

src = "/content/coreb_k_comparison_bootstrap.csv"
dst = "/content/drive/MyDrive/coreb_k_comparison_bootstrap.csv"

if os.path.exists(src):
    shutil.copy2(src, dst)
    print("Результаты сохранены на Google Drive:")
    print(dst)
else:
    print("CSV не найден. Не запускай эксперимент повторно — сообщи мне.")

Результаты сохранены на Google Drive:
/content/drive/MyDrive/coreb_k_comparison_bootstrap.csv


In [16]:
import os
import pandas as pd
import glob

# 1. Определим параметры из кода последней ячейки (3ffdf9c7)
# METRICS = ['Hit@1', 'Hit@5', 'Recall@10', 'MRR', 'nDCG@10']
# N_BOOT = 5000
# RANDOM_SEED = 20261031
# Метод интервалов: np.quantile(..., 0.025) и np.quantile(..., 0.975) (процентильный бутстрап)
# Способ формирования кластеров: meta_source_problem_id (оригинальные задачи)

print("=== ПАРАМЕТРЫ BOOTSTRAP (k=2 vs k=1) ===")
print("Число репликаций:", 5000)
print("Случайный seed:", 20261031)
print("Метод 95% CI: Процентильный (percentile bootstrap) по квантилям 0.025 и 0.975")
print("Ключ кластеризации: meta_source_problem_id")

# 2. Создаем директорию на Google Drive
drive_dest = "/content/drive/MyDrive/Research_Code_Retrieval"
os.makedirs(drive_dest, exist_ok=True)
print(f"\nСоздана директория на Google Drive: {drive_dest}")


=== ПАРАМЕТРЫ BOOTSTRAP (k=2 vs k=1) ===
Число репликаций: 5000
Случайный seed: 20261031
Метод 95% CI: Процентильный (percentile bootstrap) по квантилям 0.025 и 0.975
Ключ кластеризации: meta_source_problem_id

Создана директория на Google Drive: /content/drive/MyDrive/Research_Code_Retrieval


In [17]:
import shutil

# 3. Ищем все CSV в /content и копируем их на Google Drive
csv_files = glob.glob("/content/*.csv")
copied_files = []

for src_file in csv_files:
    filename = os.path.basename(src_file)
    dst_file = os.path.join(drive_dest, filename)
    shutil.copy2(src_file, dst_file)
    copied_files.append(dst_file)

# 4. Проверяем наличие архивов AtCoder
atcoder_archive = "/content/AtCoder.mongo.zip"
atcoder_exists = os.path.exists(atcoder_archive)

print("=== РЕЗУЛЬТАТЫ ЭКСПОРТА ===")
if copied_files:
    print("Успешно сохранены файлы результатов:")
    for f in copied_files:
        print(f"  - {f}")
else:
    print("CSV-файлы не найдены в /content.")

print(f"\nСтатус архива AtCoder: {'НАЙДЕН' if atcoder_exists else 'ОТСУТСТВУЕТ (повторная загрузка заблокирована согласно инструкциям)'}")


=== РЕЗУЛЬТАТЫ ЭКСПОРТА ===
Успешно сохранены файлы результатов:
  - /content/drive/MyDrive/Research_Code_Retrieval/coreb_grouped_fixed_labels_results.csv
  - /content/drive/MyDrive/Research_Code_Retrieval/coreb_k_comparison_bootstrap.csv

Статус архива AtCoder: ОТСУТСТВУЕТ (повторная загрузка заблокирована согласно инструкциям)


In [20]:

import os
import time
import hashlib
import zipfile
import shutil
from pathlib import Path

import requests
from tqdm.auto import tqdm

# ---------- Configuration ----------
FILENAME = "AtCoder.mongo.zip"
EXPECTED_MD5 = "11e95fae502b9fffcd9771aa1d9cb01a"
API_URL = "https://zenodo.org/api/records/4002869"

from google.colab import drive

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    drive.mount("/content/drive")

TARGET = DRIVE_ROOT / "Research_Code_Retrieval" / "data" / FILENAME
LOCAL = Path("/content") / FILENAME
PART = Path("/content") / (FILENAME + ".part")

def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def matches_md5(path):
    if not path.is_file():
        return False
    digest = file_md5(path)
    print(f"Проверка {path}")
    print(f"  Размер: {path.stat().st_size:,} байт")
    print(f"  MD5:    {digest}")
    print(f"  Совпадает: {digest == EXPECTED_MD5}")
    return digest == EXPECTED_MD5

# ---------- 1. Look for existing copies ----------
known_paths = [
    LOCAL,
    TARGET,
    DRIVE_ROOT / FILENAME,
    DRIVE_ROOT / "Research_Code_Retrieval" / FILENAME,
    DRIVE_ROOT / "data" / FILENAME,
]

found_paths = []
checked_paths = set()
valid_source = None

def check_candidate(path):
    global valid_source

    if not path.is_file():
        return False

    resolved = str(path.resolve())
    if resolved in checked_paths:
        return False
    checked_paths.add(resolved)
    found_paths.append(path)

    if matches_md5(path):
        valid_source = path
        return True
    return False

for path in known_paths:
    if check_candidate(path):
        break

# Search other Drive folders if no valid known copy was found.
if valid_source is None:
    for root, dirs, files in os.walk(DRIVE_ROOT):
        dirs[:] = [
            d for d in dirs
            if d not in {".Trash", ".ipynb_checkpoints", "node_modules"}
        ]
        if FILENAME in files:
            path = Path(root) / FILENAME
            if check_candidate(path):
                break

print("\nАрхив найден до загрузки:", "ДА" if found_paths else "НЕТ")

# ---------- 2. Obtain the download URL from the official API ----------
if valid_source is None:
    response = requests.get(API_URL, timeout=60)
    response.raise_for_status()
    record = response.json()

    file_meta = next(
        (
            item for item in record.get("files", [])
            if item.get("key", item.get("filename")) == FILENAME
        ),
        None
    )

    if file_meta is None:
        raise RuntimeError(
            f"{FILENAME} отсутствует в метаданных официальной записи Zenodo."
        )

    official_md5 = file_meta.get("checksum", "").split(":")[-1].lower()
    expected_size = int(file_meta["size"])
    download_url = file_meta.get("links", {}).get("content")

    print("\nОфициальные метаданные Zenodo:")
    print("  Название:", FILENAME)
    print(f"  Размер:  {expected_size:,} байт")
    print("  MD5:     ", official_md5)
    print("  URL получен из API:", bool(download_url))

    if official_md5 != EXPECTED_MD5:
        raise RuntimeError(
            "Контрольная сумма в Zenodo не совпадает с заданной. "
            "Загрузка остановлена."
        )
    if not download_url:
        raise RuntimeError(
            "API не вернул ссылку content. Загрузка не начиналась."
        )

    # ---------- 3. Stream the download with limited retries ----------
    downloaded_ok = False

    for attempt in range(1, 4):
        try:
            offset = PART.stat().st_size if PART.exists() else 0

            if offset > expected_size:
                PART.unlink()
                offset = 0

            if offset < expected_size:
                headers = {"Range": f"bytes={offset}-"} if offset else {}

                with requests.get(
                    download_url,
                    headers=headers,
                    stream=True,
                    timeout=(30, 120)
                ) as r:
                    if offset and r.status_code == 206:
                        mode = "ab"
                        progress_start = offset
                    else:
                        r.raise_for_status()
                        mode = "wb"
                        progress_start = 0

                    with open(PART, mode) as f, tqdm(
                        total=expected_size,
                        initial=progress_start,
                        unit="B",
                        unit_scale=True,
                        desc=f"Загрузка, попытка {attempt}/3"
                    ) as bar:
                        for chunk in r.iter_content(
                            chunk_size=4 * 1024 * 1024
                        ):
                            if chunk:
                                f.write(chunk)
                                bar.update(len(chunk))

            actual_size = PART.stat().st_size
            if actual_size != expected_size:
                raise IOError(
                    f"Неполная загрузка: {actual_size} байт "
                    f"вместо {expected_size}."
                )

            digest = file_md5(PART)
            print(f"MD5 загруженного архива: {digest}")

            if digest != EXPECTED_MD5:
                PART.unlink(missing_ok=True)
                raise ValueError("MD5 не совпадает; файл будет загружен заново.")

            with zipfile.ZipFile(PART, "r") as zf:
                bad_member = zf.testzip()

            if bad_member is not None:
                PART.unlink(missing_ok=True)
                raise ValueError(f"Повреждённый элемент ZIP: {bad_member}")

            downloaded_ok = True
            valid_source = PART
            print("Проверка загруженного ZIP пройдена.")
            break

        except Exception as exc:
            print(f"Попытка {attempt}/3 не удалась: {exc}")
            if attempt == 3:
                raise RuntimeError(
                    "Не удалось восстановить архив за три попытки."
                ) from exc
            time.sleep(2 * attempt)

    if not downloaded_ok:
        raise RuntimeError("Архив не прошёл проверку.")

# ---------- 4. Copy only a verified archive to Google Drive ----------
TARGET.parent.mkdir(parents=True, exist_ok=True)

same_file = (
    TARGET.exists()
    and os.path.samefile(valid_source, TARGET)
)

if not same_file:
    drive_temp = TARGET.with_name(TARGET.name + ".part")
    drive_temp.unlink(missing_ok=True)

    print(f"\nКопирование проверенного файла на Drive: {TARGET}")
    shutil.copyfile(valid_source, drive_temp)

    if file_md5(drive_temp) != EXPECTED_MD5:
        drive_temp.unlink(missing_ok=True)
        raise RuntimeError("MD5 копии на Drive не совпала. Файл не опубликован.")

    # Temporary file is in the same Drive directory as the final file.
    os.replace(drive_temp, TARGET)

# ---------- 5. Verify the exact final path ----------
if not TARGET.is_file():
    raise FileNotFoundError(f"Итоговый файл не найден: {TARGET}")

final_size = TARGET.stat().st_size
final_md5 = file_md5(TARGET)

with zipfile.ZipFile(TARGET, "r") as zf:
    bad_member = zf.testzip()

zip_ok = bad_member is None
md5_ok = final_md5 == EXPECTED_MD5

if not md5_ok or not zip_ok:
    raise RuntimeError(
        f"Финальная проверка не пройдена: MD5={md5_ok}, ZIP={zip_ok}"
    )

# Remove the local temporary download only after final verification.
if PART.exists() and PART.resolve() != TARGET.resolve():
    PART.unlink()

print("\n" + "=" * 55)
print("ИТОГОВЫЙ ОТЧЁТ")
print("=" * 55)
print("Архив существовал до загрузки:", "ДА" if found_paths else "НЕТ")
print("Источник восстановления:", "существующая копия" if found_paths and valid_source != PART else "официальный Zenodo API" if valid_source == PART else "проверенная копия")
print(f"Фактический размер: {final_size:,} байт")
print(f"Ожидаемый MD5:      {EXPECTED_MD5}")
print(f"Фактический MD5:    {final_md5}")
print("MD5 совпадает:      ", md5_ok)
print("ZIP целостен:       ", zip_ok)
print("Сохранённый путь:   ", str(TARGET))
print("СТАТУС:             УСПЕХ")


Проверка /content/AtCoder.mongo.zip
  Размер: 350,063,499 байт
  MD5:    11e95fae502b9fffcd9771aa1d9cb01a
  Совпадает: True

Архив найден до загрузки: ДА

ИТОГОВЫЙ ОТЧЁТ
Архив существовал до загрузки: ДА
Источник восстановления: существующая копия
Фактический размер: 350,063,499 байт
Ожидаемый MD5:      11e95fae502b9fffcd9771aa1d9cb01a
Фактический MD5:    11e95fae502b9fffcd9771aa1d9cb01a
MD5 совпадает:       True
ZIP целостен:        True
Сохранённый путь:    /content/drive/MyDrive/Research_Code_Retrieval/data/AtCoder.mongo.zip
СТАТУС:             УСПЕХ


In [3]:
import os
import glob

print("=== FILES IN /content ===")

for name in sorted(os.listdir("/content")):
    path = os.path.join("/content", name)

    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"FILE  {name:45s} {size_mb:10.2f} MB")
    else:
        print(f"DIR   {name}/")

print("\n=== POSSIBLE ATCODER DATA FILES ===")

patterns = [
    "/content/*AtCoder*",
    "/content/*atcoder*",
    "/content/**/*.bson",
    "/content/**/*.pickle",
    "/content/**/*.pkl",
    "/content/**/*.parquet",
]

found = set()

for pattern in patterns:
    for path in glob.glob(pattern, recursive=True):
        if os.path.isfile(path):
            found.add(path)

for path in sorted(found):
    print(path)

=== FILES IN /content ===
DIR   .config/
DIR   sample_data/

=== POSSIBLE ATCODER DATA FILES ===


In [1]:
import os
import glob

print("=== FILES IN /content ===")

for name in sorted(os.listdir("/content")):
    path = os.path.join("/content", name)

    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"FILE  {name:45s} {size_mb:10.2f} MB")
    else:
        print(f"DIR   {name}/")

print("\n=== POSSIBLE ATCODER DATA FILES ===")

patterns = [
    "/content/*AtCoder*",
    "/content/*atcoder*",
    "/content/**/*.bson",
    "/content/**/*.pickle",
    "/content/**/*.pkl",
    "/content/**/*.parquet",
]

found = set()

for pattern in patterns:
    for path in glob.glob(pattern, recursive=True):
        if os.path.isfile(path):
            found.add(path)

for path in sorted(found):
    print(path)

=== FILES IN /content ===
DIR   .config/
DIR   sample_data/

=== POSSIBLE ATCODER DATA FILES ===


In [2]:
import os

print("Google Drive подключён:", os.path.exists("/content/drive/MyDrive"))

for path in [
    "/content/AtCoder.mongo.zip",
    "/content/XLCoST_data.zip",
]:
    print(path, "—", os.path.exists(path))

Google Drive подключён: False
/content/AtCoder.mongo.zip — False
/content/XLCoST_data.zip — False


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

drive_path = "/content/drive/MyDrive"

print("Файлы в корне Google Drive:")
for name in os.listdir(drive_path):
    print("📁" if os.path.isdir(os.path.join(drive_path, name)) else "📄", name)

print("\nПоиск архивов, ноутбуков и результатов исследования:")

targets = [
    "AtCoder.mongo.zip",
    "XLCoST_data.zip",
]

found = []

for root, dirs, files in os.walk(drive_path):
    # Не заходим в системные и скрытые папки
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for name in files:
        lower = name.lower()
        if (
            name in targets
            or lower.endswith(".ipynb")
            or "coreb" in lower
            or "atcoder" in lower
            or "cross_language" in lower
        ):
            found.append(os.path.join(root, name))

if found:
    for path in found:
        print(path)
else:
    print("Подходящие файлы не найдены.")

Файлы в корне Google Drive:
📁 Classroom
📁 Noteshelf
📄 Новый документ.gdoc
📄 Новая таблица.gsheet
📄 Grow me!.gdoc
📁 Сохраненное из приложения Google
📄 а когда бесконечное количество решений.gsheet
📄 transfer-receipt-13_876015286158960835.pdf
📄 The Evolution of Data Science Job Requirements: A Five-Year Analysis (2021–2026).gdoc
📁 Colab Notebooks

Поиск архивов, ноутбуков и результатов исследования:
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
